<a href="https://colab.research.google.com/github/COL-O/POTOK/blob/arena%2F9f300297-potok/pol_o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, subprocess, socket
try:
    import psutil
except ImportError:
    psutil = None
print("=== ПАМЯТЬ ===")
if psutil:
    vm = psutil.virtual_memory()
    print(f"всего {vm.total/2**30:.1f} ГБ, свободно {vm.available/2**30:.1f} ГБ, занято {vm.percent}%")
    rows = []
    for p in psutil.process_iter(["pid","name","memory_info","cmdline"]):
        try:
            rss = p.info["memory_info"].rss if p.info["memory_info"] else 0
            rows.append((rss, p.info["pid"], (p.info["name"] or "")[:20], " ".join(p.info["cmdline"] or [])[:110]))
        except Exception:
            pass
    print("=== ТОП-12 по памяти ===")
    for rss, pid, name, cmd in sorted(rows, reverse=True)[:12]:
        print(f"{rss/2**20:8.0f} МБ  {pid:6d}  {name:20s} {cmd}")
print("=== ПОРТЫ ===")
for port in (5080, 8000, 8001, 8080, 11434, 1234, 7860, 3000, 5000, 9000):
    s = socket.socket(); s.settimeout(0.25)
    print(port, "открыт" if s.connect_ex(("127.0.0.1", port)) == 0 else "-")
    s.close()
print("=== ЧТО ЗАПУЩЕНО (serve/llama/vllm) ===")
out = subprocess.run("ps -eo pid,rss,args | grep -Ei 'vllm|llama|serve|ollama' | grep -v grep | head -8", shell=True, capture_output=True, text=True).stdout
print(out or "не найдено")
print("=== ДИСК ===")
print(subprocess.run("df -h /content / 2>/dev/null | tail -3", shell=True, capture_output=True, text=True).stdout)


=== ПАМЯТЬ ===
всего 12.7 ГБ, свободно 6.5 ГБ, занято 48.9%
=== ТОП-12 по памяти ===
    4942 МБ    1173  python3              /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-bcf8838d-3ab6-462f-bcad
    1341 МБ   46601  node                 node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:793def5c2d8c82883608237a2b2
     192 МБ    7701  openobserve          /content/AMBER_SOCIETY_CAPSULE/openobserve_node/openobserve
     136 МБ      99  jupyter-server       /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= -
      76 МБ    1312  drive                /opt/google/drive/drive --features=crash_throttle_percentage:100,fuse_max_background:1000,max_read_qps:1000,ma
      69 МБ       7  node                 /tools/node/bin/node /datalab/web/app.js
      60 МБ      69  colab-fileshim.      python3 /usr/local/bin/colab-fileshim.py
      37 МБ   29043  surreal-db    

In [ ]:
import torch, os, sys, glob, json
print("=== ИНТЕРПРЕТАТОР ===", sys.executable)
try:
    print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)
print("=== КРУПНЫЕ ОБЪЕКТЫ В ЯДРЕ ===")
try:
    from transformers import AutoModelForCausalLM
except Exception:
    pass
import sys as _s
objs = []
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    t = type(val).__name__
    mod = type(val).__module__
    if mod.startswith(("transformers", "llama_cpp", "vllm", "torch")):
        n = 0
        try:
            n = sum(p.numel() for p in val.parameters())
        except Exception:
            pass
        objs.append((name, t, mod, n))
for o in objs[:20]:
    print(o)
print("=== ДИСКИ И КАПСУЛА ===")
print(subprocess_out := __import__("subprocess").run("ls -1 /content | head -20", shell=True, capture_output=True, text=True).stdout)
print(__import__("subprocess").run("ss -ltnp 2>/dev/null | head -15", shell=True, capture_output=True, text=True).stdout)


=== ИНТЕРПРЕТАТОР === /usr/bin/python3
torch 2.14.1+cpu cuda False 
=== КРУПНЫЕ ОБЪЕКТЫ В ЯДРЕ ===
('director_llm', 'Llama', 'llama_cpp.llama', 0)
=== ДИСКИ И КАПСУЛА ===
AMBER_SOCIETY_CAPSULE
drive
sample_data

State  Recv-Q Send-Q Local Address:Port  Peer Address:PortProcess                                 
LISTEN 0      4096     172.28.0.12:6000       0.0.0.0:*    users:(("kernel_manager_",pid=12,fd=8))
LISTEN 0      2048       127.0.0.1:7864       0.0.0.0:*    users:(("python3",pid=1173,fd=114))    
LISTEN 0      2048       127.0.0.1:7861       0.0.0.0:*    users:(("python3",pid=1173,fd=86))     
LISTEN 0      2048       127.0.0.1:7860       0.0.0.0:*    users:(("python3",pid=1173,fd=76))     
LISTEN 0      2048       127.0.0.1:7863       0.0.0.0:*    users:(("python3",pid=1173,fd=104))    
LISTEN 0      2048       127.0.0.1:7862       0.0.0.0:*    users:(("python3",pid=1173,fd=96))     
LISTEN 0      2048       127.0.0.1:8088       0.0.0.0:*    users:(("python3",pid=1173,fd=69))  

In [ ]:
import json, time, os
print("=== ДИРЕКТОР: кто ты и что видишь ===")
try:
    t0 = time.time()
    ans = director_llm("Ты Директор штаба. Ответь коротко, 3 строки: кто ты, какая модель в памяти, что сейчас запущено вокруг тебя.", max_tokens=160)
    print(ans)
    print(f"--- ответ за {time.time()-t0:.1f} с")
except Exception as e:
    print("ошибка вызова:", type(e).__name__, e)
    try:
        print("подпись:", director_llm.__doc__ or "")
    except Exception:
        pass


=== ДИРЕКТОР: кто ты и что видишь ===
{'id': 'cmpl-6178dd62-3ba8-4c7c-81b5-2ecc197552d6', 'object': 'text_completion', 'created': 1795292725, 'model': '/content/AMBER_SOCIETY_CAPSULE/models/executive_7b/DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf', 'choices': [{'text': ' /\n\nTranslation: You are the Director of the办公厅. Answer concisely, in 3 lines: who are you, what is in your memory model, what is currently running around you.\n\n/ /\n\n<think>\nOkay, so I need to answer this query where someone wants me to be the Director of the办公厅. They provided some specific instructions: respond in three short lines, answering who I am, what\'s in my memory model, and what\'s currently running around me. \n\nFirst, who am I? Well, I should be clear but professional. Maybe something like, "I am the Director of the办公厅, responsible for managing the office staff and ensuring smooth operations." That covers my role and responsibilities.\n\nNext, what\'s in my memory model. This is a bit tricky. Memory mod

# Добро пожаловать в Colab!

основная поверка кто дома , что осталось от основного отдела после отключения


---



*   • Хотите просто понять, кто занял ОЗУ: оставляете пункт 1. Только осмотреть память — скрипт разложит по полочкам каждый процесс в мегабайтах и покажет, висит ли в фоне дубликат от прошлой сессии или память занимают соратники (SmolLM2, deputy_7b).
• Если соратники мешают Командиру: выбираете 2 или 3 — соратники и фоновые копии уходят на диск, а один Командир остаётся с максимальным запасом свободной памяти.
• Закончили работу: выбираете 4 — все данные OpenObserve улетают в сейф на Google Диск, а вся группа выгружается из памяти отдыхать.
*   Новый пункт




Новая версия  они и озу , кто , что, с кеи и сколько отправить на отдых

In [ ]:
#@title Диспетчер памяти и режима отдыха (Осмотр ОЗУ + Чистка Pyright/Дублей + Выбор кого усыпить)
MEMORY_ACTION = "1. Только осмотреть память (кто сколько ест)" #@param ["1. Только осмотреть память (кто сколько ест)", "2. Усыпить только соратников (оставить одного Командира)", "3. Убить лишние дубли и раздутый Pyright (освободить место Командиру)", "4. Сохранить всё на Диск и отправить ВСЕХ отдыхать (0 МБ моделей)"]
AUTO_SAVE_TO_DRIVE = True #@param {type:"boolean"}

import os, gc, time, ctypes, shutil, signal, psutil

MY_PID = os.getpid()
SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_O2_DATA = "/content/AMBER_SOCIETY_CAPSULE/openobserve_node/data"
SAFE_O2_SNAP = os.path.join(SAFE_DIR, "o2_snapshot_latest.tar.gz")

def trim_linux_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def save_before_rest():
    """Тихое сохранение всех данных OpenObserve в сейф на Google Диск"""
    if os.path.exists(WORK_O2_DATA) and os.listdir(WORK_O2_DATA):
        os.makedirs(SAFE_DIR, exist_ok=True)
        shutil.make_archive("/tmp/o2_rest_snap", "gztar", WORK_O2_DATA)
        shutil.move("/tmp/o2_rest_snap.tar.gz", SAFE_O2_SNAP)
        mb = round(os.path.getsize(SAFE_O2_SNAP) / (1024**2), 2)
        print(f"[СЕЙФ] Данные OpenObserve сохранены на Google Диск ({mb} МБ).", flush=True)

# 1. ПОЛНАЯ РЕНТГЕНОГРАММА ПАМЯТИ VM И ПРОЦЕССОВ PYTHON
vm = psutil.virtual_memory()
print(f"=== СВОДКА ОЗУ МАШИНЫ COLAB ДО ОЧИСТКИ ===")
print(f"Всего: {round(vm.total/(1024**3), 2)} ГБ | Занято: {round(vm.used/(1024**3), 2)} ГБ | Свободно: {round(vm.available/(1024**3), 2)} ГБ\n", flush=True)

print("--- Активные процессы в системе (> 80 МБ) ---")
ghost_pids = list()
for p in psutil.process_iter(["pid", "name", "cmdline", "memory_info"]):
    try:
        pid = p.info["pid"]
        rss_mb = round((p.info["memory_info"].rss or 0) / (1024**2), 1)
        cmd = " ".join(p.info["cmdline"] or list())
        cmd_low = cmd.lower()
        if rss_mb > 80:
            tag = "ТЕКУЩИЙ СЕАНС" if pid == MY_PID else "ФОН / ВОЗМОЖНЫЙ ДУБЛЬ"
            print(f"  PID {pid:6d} | {rss_mb:7.1f} МБ | [{tag}] {p.info['name']} ({cmd[:45]})")
            is_bloat_pyright = ("pyright-langserver" in cmd_low and rss_mb > 400)
            is_ghost_worker = (pid != MY_PID and ("ipykernel" in cmd_low or "llama" in cmd_low))
            if is_bloat_pyright or is_ghost_worker:
                ghost_pids.append((pid, rss_mb, p.info["name"]))
    except Exception:
        pass

print("\n--- Переменные моделей в текущем Python ---")
loaded_vars = list()
for v in ("director_llm", "deputy_llm", "worker_llm", "json_slm_model", "base_model", "model"):
    if v in globals() and globals()[v] is not None:
        role_label = "Главный Командир (7B)" if v == "director_llm" else "Соратник / Вспомогательная модель"
        print(f"  • {v:18s} -> АКТИВНА ({role_label})")
        loaded_vars.append(v)
if not loaded_vars:
    print("  • В текущем ядре Python активных моделей нет.")

# 2. ВЫПОЛНЕНИЕ ВЫБРАННОГО ВАМИ ДЕЙСТВИЯ
print("\n=== ВЫПОЛНЕНИЕ ДЕЙСТВИЯ ===")
if AUTO_SAVE_TO_DRIVE and not MEMORY_ACTION.startswith("1."):
    save_before_rest()

if MEMORY_ACTION.startswith("1."):
    if len(ghost_pids) > 0:
        print(f"⚠️ Внимание: обнаружены пожиратели памяти/дубли ({len(ghost_pids)} шт.), которые мешают Командиру развернуться.")
    else:
        print("✅ Лишних пожирателей ОЗУ не найдено.")

elif MEMORY_ACTION.startswith("2."):
    slept = 0
    for v in loaded_vars:
        if v != "director_llm":
            del globals()[v]
            slept += 1
            print(f"  [СОН] Соратник {v} отправлен отдыхать на диск.")
    trim_linux_ram()
    print(f"Готово! Усыплено соратников: {slept}. Командир остался на посту.")

elif MEMORY_ACTION.startswith("3."):
    freed = 0.0
    for pid, mb, name in ghost_pids:
        try:
            os.kill(pid, signal.SIGKILL)
            freed += mb
            print(f"  [ОЧИСТКА] Снят процесс PID {pid} ({name}): +{mb} МБ.")
        except Exception:
            pass
    for v in loaded_vars:
        if v != "director_llm":
            del globals()[v]
    trim_linux_ram()
    print(f"Пространство для основного Командира расчищено (возвращено ~{round(freed/1024, 2)} ГБ)!")

elif MEMORY_ACTION.startswith("4."):
    for v in loaded_vars:
        del globals()[v]
        print(f"  [ОТДЫХ] Модель {v} выгружена на диск.")
    for pid, mb, name in ghost_pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except Exception:
            pass
    trim_linux_ram()
    print("🌙 Весь коллектив и Командир сохранили данные на Диск и ушли отдыхать. ОЗУ полностью свободна.")

vm_after = psutil.virtual_memory()
print(f"\nИТОГ ПОСЛЕ ДЕЙСТВИЯ -> Свободно ОЗУ: {round(vm_after.available/(1024**3), 2)} ГБ из {round(vm_after.total/(1024**3), 2)} ГБ", flush=True)


=== СВОДКА ОЗУ МАШИНЫ COLAB ДО ОЧИСТКИ ===
Всего: 12.67 ГБ | Занято: 2.76 ГБ | Свободно: 9.6 ГБ

--- Активные процессы в системе (> 80 МБ) ---
  PID    132 |   158.6 МБ | [ФОН / ВОЗМОЖНЫЙ ДУБЛЬ] jupyter-server (/usr/bin/python3 /usr/local/bin/jupyter-serve)
  PID    896 |   103.3 МБ | [ТЕКУЩИЙ СЕАНС] python3 (/usr/bin/python3 -m colab_kernel_launcher -f )
  PID   1124 |  1965.1 МБ | [ФОН / ВОЗМОЖНЫЙ ДУБЛЬ] node (node /datalab/web/pyright/pyright-langserver.)

--- Переменные моделей в текущем Python ---
  • В текущем ядре Python активных моделей нет.

=== ВЫПОЛНЕНИЕ ДЕЙСТВИЯ ===
⚠️ Внимание: обнаружены пожиратели памяти/дубли (1 шт.), которые мешают Командиру развернуться.

ИТОГ ПОСЛЕ ДЕЙСТВИЯ -> Свободно ОЗУ: 9.59 ГБ из 12.67 ГБ


# *Обычный чат - просто версия 3.0 - запускать после всех , стоит вверху для обозрения и удобства*


---


*   Новый пункт
*   Новый пункт




In [ ]:
#@title 3. Быстрый пульт общения (Исправленный, отклик 12-18 сек)
СТАРТОВЫЙ_РЕЖИМ = "1. Быстрый Командир 7B (короткий контекст, ~15 сек)" #@param ["1. Быстрый Командир 7B (короткий контекст, ~15 сек)", "2. Мгновенный Начальник Штаба LFM2-350M (~2 сек)", "3. Совет выбранных помощников + Командир"]
СТАРТОВЫЙ_СОСТАВ = "Только Командир" #@param ["Только Командир", "Быстрый дуэт (LFM2 + Coder)", "Весь Совет (все 6 моделей)"]
ДЛИНА_ОТВЕТА = 200 #@param {type:"slider", min:100, max:600, step:50}
АВТОЗАПИСЬ_ОПЫТА = True #@param {type:"boolean"}

import os, gc, glob, json, time, ctypes, requests, traceback
import gradio as gr
from llama_cpp import Llama

SAFE_DIR   = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR   = "/content/AMBER_SOCIETY_CAPSULE"
MODELS_DIR = os.path.join(WORK_DIR, "models")
O2_URL     = "http://127.0.0.1:5080"
AUTH       = ("admin@society.local", "ChangeMe123!")

STAFF_6 = {
    "lfm2":      ("resident/Liquid-LFM2-350M",         "room_resident", "Оперативный диспетчер Штаба"),
    "qwen05":    ("resident/Qwen2.5-0.5B-Instruct",    "room_resident", "Переводчик (RU/EN/ZH) и логик"),
    "gemma270":  ("resident/Gemma-3-270M-IT",          "room_resident", "Аналитик длинных отчётов"),
    "smollm":    ("resident/SmolLM2-360M-Instruct",    "room_resident", "Критик гипотез"),
    "nuextract": ("tables_graphs/NuExtract-tiny-0.5B", "room_tables",   "Делопроизводитель JSON"),
    "coder":     ("tables_graphs/Qwen2.5-Coder-0.5B",  "room_tables",   "Инженер SQL / SurrealQL"),
}

def clean_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def get_first_choice(res: dict) -> dict:
    choices = res.get("choices", list())
    return next(iter(choices), dict())

def surreal_sql(query: str):
    try:
        r = requests.post(
            "http://127.0.0",
            auth=("root", "root"),
            headers={"Accept": "application/json", "NS": "amber_sphere", "DB": "society_core"},
            data=query.encode("utf-8"),
            timeout=3
        )
        return r.json()
    except Exception:
        return list()

def call_staff_member(key: str, task_text: str, shared_board: str) -> str:
    if key not in STAFF_6:
        return f"[{key}: неизвестный код]"
    rel_folder, room, role = STAFF_6.get(key)
    files = glob.glob(os.path.join(MODELS_DIR, rel_folder, "*.gguf")) or glob.glob(os.path.join(SAFE_DIR, "models", rel_folder, "*.gguf"))
    m_path = next(iter(files), None)
    if not m_path:
        return f"[{key}: файл не найден в {rel_folder}]"
    w = Llama(model_path=m_path, n_ctx=1024, n_threads=2, verbose=False)
    res = w.create_chat_completion(
        messages=[{"role": "user", "content": f"Role: {role}. Board: {shared_board[:300]}. Task: {task_text}. Reply briefly in 2 sentences."}],
        max_tokens=110, temperature=0.15
    )
    ans = get_first_choice(res).get("message", dict()).get("content", "").strip()
    del w
    clean_ram()
    surreal_sql(f"CREATE {room} SET author = '{key}', note = {json.dumps(ans, ensure_ascii=False)}, ts = time::now();")
    return ans

def fast_chat_handler(message, history, mode_str, selected_staff, max_tok):
    try:
        t0 = time.time()
        short_sys = (
            "Ты Командир Янтарной Сферы (OpenObserve :5080, SurrealDB :8000, 28 микромоделей). "
            "Пользователь — Создатель проекта. Отвечай чётко, по делу, строго на русском языке."
        )

        if mode_str.startswith("2."):
            ans = call_staff_member("lfm2", f"Ответь на русском кратко: {message}", "Прямой запрос Создателя")
            return f"**[Начальник Штаба LFM2-350M · {round(time.time()-t0, 1)} сек]:**\n{ans}"

        if "director_llm" not in globals() or director_llm is None:
            return "⚠️ Командир ещё не загружен в память. Запустите сначала Ячейку 1 (Ядро Монолита)."

        if mode_str.startswith("1.") or not selected_staff:
            res = director_llm.create_completion(
                prompt=f"<|User|>{short_sys}\nВопрос: {message}<|Assistant|><think>\n</think>\n",
                max_tokens=int(max_tok),
                temperature=0.15,
                stop=["<|end_of_sentence|>"]
            )
            reply = get_first_choice(res).get("text", "").strip()
        else:
            board = [f"**[{k}]:** {call_staff_member(k, message, 'Старт')}" for k in selected_staff]
            transcript = "\n".join(board)
            res = director_llm.create_completion(
                prompt=f"<|User|>{short_sys}\nВопрос: {message}\nМнения:\n{transcript[:800]}\nКраткий итог Командира:<|Assistant|><think>\n</think>\n",
                max_tokens=int(max_tok),
                temperature=0.15,
                stop=["<|end_of_sentence|>"]
            )
            verdict = get_first_choice(res).get("text", "").strip()
            reply = f"{transcript}\n\n---\n**Итог Командира:**\n{verdict}"

        elapsed = round(time.time() - t0, 1)
        try:
            requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[{"question": message, "verdict": reply, "sec": elapsed}], timeout=1)
            surreal_sql(f"CREATE shared_knowledge SET question = {json.dumps(message, ensure_ascii=False)}, answer = {json.dumps(reply, ensure_ascii=False)}, ts = time::now();")
        except Exception:
            pass

        return f"{reply}\n\n`[Время ответа: {elapsed} сек]`"
    except Exception as e:
        return f"Ошибка выполнения: {e}\n{traceback.format_exc()[-300:]}"

with gr.Blocks(title="AMBER SOCIETY — Быстрый пульт") as demo:
    gr.Markdown("## AMBER SOCIETY — Скоростной командный пульт")
    with gr.Row():
        mode_r = gr.Radio(
            choices=[
                "1. Быстрый Командир 7B (короткий контекст, ~15 сек)",
                "2. Мгновенный Начальник Штаба LFM2-350M (~2 сек)",
                "3. Совет выбранных помощников + Командир"
            ],
            value=СТАРТОВЫЙ_РЕЖИМ,
            label="Скорость и режим"
        )
        staff_c = gr.CheckboxGroup(
            choices=[("LFM2-350M", "lfm2"), ("Qwen-0.5B", "qwen05"), ("Gemma-270M", "gemma270"), ("SmolLM2", "smollm"), ("NuExtract", "nuextract"), ("Coder-0.5B", "coder")],
            value=["lfm2", "coder"],
            label="Состав Совета (для режима №3)"
        )
        tok_s = gr.Slider(minimum=100, maximum=600, step=50, value=ДЛИНА_ОТВЕТА, label="Макс. длина ответа (меньше = быстрее)")

    gr.ChatInterface(fn=fast_chat_handler, additional_inputs=[mode_r, staff_c, tok_s])

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e2bb1ba7b62c75395.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#@title Директива Командиру: Освоение OpenObserve + SurrealDB и бережное подключение 6 помощников
ЗАПУСТИТЬ_ДИРЕКТИВУ = True #@param {type:"boolean"}
ЛИМИТ_ОЗУ_ПРОЦЕНТ = 75 #@param {type:"slider", min:50, max:85, step:5}
ПАУЗА_СПОКОЙСТВИЯ_СЕК = 25 #@param {type:"slider", min:10, max:60, step:5}

import os, gc, glob, json, time, ctypes, threading, requests, psutil
from llama_cpp import Llama

SAFE_DIR   = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR   = "/content/AMBER_SOCIETY_CAPSULE"
MODELS_DIR = os.path.join(WORK_DIR, "models")
O2_URL     = "http://127.0.0.1:5080"
AUTH       = ("admin@society.local", "ChangeMe123!")

def free_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def surreal_q(sql_str: str):
    try:
        r = requests.post(
            "http://127.0.0",
            auth=("root", "root"),
            headers={"Accept": "application/json", "NS": "amber_sphere", "DB": "society_core"},
            data=sql_str.encode("utf-8"),
            timeout=5
        )
        return r.json()
    except Exception as e:
        return [{"error": str(e)}]

def check_hardware_calm(max_ram_pct: int) -> tuple:
    """Проверка спокойствия оборудования Colab перед пробуждением каждого помощника"""
    ram_pct = psutil.virtual_memory().percent
    cpu_pct = psutil.cpu_percent(interval=1.0)
    is_calm = (ram_pct <= max_ram_pct) and (cpu_pct <= 80.0)
    return is_calm, ram_pct, cpu_pct

# 1. Официальное руководство-директива Командиру по OpenObserve, SurrealDB и бережному режиму
DIRECTIVE_PROMPT = f"""
БОЕВОЕ РУКОВОДСТВО-ДИРЕКТИВА ГЛАВНОМУ КОМАНДИРУ ЯНТАРНОЙ СФЕРЫ:

Твоя текущая задача — освоить связку двух главных платформ и подготовить 6 текстовых помощников Штаба:
1. ОБОЛОЧКА OPENOBSERVE (http://127.0.0.1:5080):
   - Потоки (Streams): society_events (входящие), society_roster (реестр моделей), ai_consensus (советы), system_docs (уставы).
   - Механизмы: VRL-пайплайны (первичная очистка на Rust до вызова нейросетей), Alerts (пробуждение строго по аномалии), AI Providers (:8088/v1) и MCP (/api/default/mcp).
2. МНОГОМОДЕЛЬНАЯ БАЗА ЗНАНИЙ SURREALDB (http://127.0.0.1:8000, NS: amber_sphere, DB: society_core):
   - Общий Зал Знаний (shared_knowledge) и Архив Личных Дел (personal_files).
   - Личные комнаты отделов: room_resident (Штаб), room_tables (Таблицы/JSON/SQL), room_streams (Датчики), room_vision (Зрение), room_bio (Генетика), room_voice (Речь/Звук).
3. УСТАВ СПОКОЙСТВИЯ ОБОРУДОВАНИЯ COLAB (КРИТИЧЕСКИ ВАЖНО):
   - Никаких пиковых нагрузок! Удерживать ОЗУ ниже {ЛИМИТ_ОЗУ_ПРОЦЕНТ}%, а CPU — в спокойном режиме.
   - Пробуждать помощников СТРОГО по одному (Serial Wake-up) с паузой {ПАУЗА_СПОКОЙСТВИЯ_СЕК} сек и мгновенной выгрузкой из памяти.

Сформируй короткий, чёткий план действий (6-7 строк на русском):
- подтверди принятие Устава спокойствия оборудования;
- опиши порядок подключения 6 помощников к OpenObserve и их комнатам в SurrealDB;
- начни пошаговое подключение прямо сейчас.
"""

if "director_llm" not in globals() or director_llm is None:
    raise RuntimeError("Сначала запустите Ячейку 1, чтобы Командир был в ОЗУ.")

print("=== [ШАГ 1] ПЕРЕДАЁМ ДИРЕКТИВУ КОМАНДИРУ ===", flush=True)
res = director_llm.create_completion(
    prompt=f"<|User|>{DIRECTIVE_PROMPT}<|Assistant|><think>\n</think>\n",
    max_tokens=320, temperature=0.15, stop=["<|end_of_sentence|>"]
)
plan_reply = next(iter(res.get("choices", list())), dict()).get("text", "").strip()
print(f"\n[ОТВЕТ КОМАНДИРА]:\n{plan_reply}\n", flush=True)

# Записываем утверждённый Устав спокойствия и план в OpenObserve, SurrealDB и Личность Командира
requests.post(f"{O2_URL}/api/default/system_docs/_json", auth=AUTH, json=[{"doc": "calm_hardware_directive", "plan": plan_reply}])
surreal_q(f"UPSERT shared_knowledge:calm_directive SET title = 'Устав спокойствия и план освоения OpenObserve+SurrealDB', text = {json.dumps(plan_reply, ensure_ascii=False)}, ts = time::now();")
if "memorize_experience" in globals():
    memorize_experience(
        summary="Принята директива освоения OpenObserve + SurrealDB и подключение 6 помощников в бережном режиме",
        new_rule=f"Соблюдать спокойствие оборудования: ОЗУ <= {ЛИМИТ_ОЗУ_ПРОЦЕНТ}%, будить по 1 модели с паузой {ПАУЗА_СПОКОЙСТВИЯ_СЕК}с",
        tags=["directive", "hardware_safety"]
    )

# 2. Фоновый бережный конвейер подключения 6 помощников к базам и личным комнатам
STAFF_6_LIST = [
    ("lfm2",      "resident/Liquid-LFM2-350M",         "room_resident", "Оперативный диспетчер Штаба и председатель малых советов"),
    ("qwen05",    "resident/Qwen2.5-0.5B-Instruct",    "room_resident", "Трёхъязычный переводчик (RU/EN/ZH) и координатор"),
    ("gemma270",  "resident/Gemma-3-270M-IT",          "room_resident", "Аналитик длинных отчётов и документации"),
    ("smollm",    "resident/SmolLM2-360M-Instruct",    "room_resident", "Критик гипотез и контролёр форматов"),
    ("nuextract", "tables_graphs/NuExtract-tiny-0.5B", "room_tables",   "Делопроизводитель строгого JSON для OpenObserve и SurrealDB"),
    ("coder",     "tables_graphs/Qwen2.5-Coder-0.5B",  "room_tables",   "Инженер SQL (DuckDB / SurrealQL) и VRL-правил"),
]

def run_calm_onboarding_6():
    print(f"=== [ШАГ 2] СТАРТ БЕРЕЖНОГО ПОДКЛЮЧЕНИЯ 6 ПОМОЩНИКОВ (Лимит ОЗУ: {ЛИМИТ_ОЗУ_ПРОЦЕНТ}%, Пауза: {ПАУЗА_СПОКОЙСТВИЯ_СЕК}с) ===", flush=True)
    for key, folder, room, role in STAFF_6_LIST:
        # Ждём спокойного состояния процессора и памяти
        while True:
            calm, r_pct, c_pct = check_hardware_calm(int(ЛИМИТ_ОЗУ_ПРОЦЕНТ))
            if calm:
                break
            print(f"  [ПАУЗА СПОКОЙСТВИЯ] ОЗУ {r_pct}% / CPU {c_pct}% — ждём остывания 10 сек...", flush=True)
            time.sleep(10)

        files = glob.glob(os.path.join(MODELS_DIR, folder, "*.gguf")) or glob.glob(os.path.join(SAFE_DIR, "models", folder, "*.gguf"))
        m_path = next(iter(files), None)
        if not m_path:
            print(f"  [ПРОПУСК] {key}: файл не найден в {folder}", flush=True)
            continue

        t0 = time.time()
        w = Llama(model_path=m_path, n_ctx=1024, n_threads=2, verbose=False)
        r = w.create_chat_completion(
            messages=[{"role": "user", "content": f"You are {key} ({role}). Confirm connection to OpenObserve and your room {room} in 6 words."}],
            max_tokens=25, temperature=0.1
        )
        ack = next(iter(r.get("choices", list())), dict()).get("message", dict()).get("content", "").strip().replace("\n", " ")
        del w
        free_ram()
        elapsed = round(time.time() - t0, 2)

        # Записываем паспорт помощника в SurrealDB (в personal_files и в его личную комнату) и в OpenObserve
        card = {
            "key": key,
            "role": role,
            "room": room,
            "o2_stream": "society_roster",
            "response_sec": elapsed,
            "ack": ack,
            "status": "CONNECTED_CALM",
            "connected_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        surreal_q(f"UPSERT personal_files:`{key}` CONTENT {json.dumps(card, ensure_ascii=False)};")
        surreal_q(f"UPSERT {room}:`{key}` CONTENT {json.dumps(card, ensure_ascii=False)};")
        requests.post(f"{O2_URL}/api/default/society_roster/_json", auth=AUTH, json=[card], timeout=2)

        _, r_now, c_now = check_hardware_calm(100)
        print(f"  [ПОДКЛЮЧЁН] {key:10s} -> Комната: {room:13s} | {elapsed:4.2f}с | ОЗУ: {r_now}% | Отклик: '{ack}'", flush=True)
        time.sleep(int(ПАУЗА_СПОКОЙСТВИЯ_СЕК))

    print("\n=== [ГОТОВО] ВСЕ 6 ПОМОЩНИКОВ ПОДКЛЮЧЕНЫ К OPENOBSERVE И СВОИМ КОМНАТАМ В SURREALDB ===", flush=True)

if ЗАПУСТИТЬ_ДИРЕКТИВУ and "calm_6_thread" not in globals():
    calm_6_thread = threading.Thread(target=run_calm_onboarding_6, daemon=True)
    calm_6_thread.start()


=== [ШАГ 1] ПЕРЕДАЁМ ДИРЕКТИВУ КОМАНДИРУ ===

[ОТВЕТ КОМАНДИРА]:
**План действий**

1. **Подтвердить принятие Устава спокойствия оборудования.**
2. **Подключить 6 текстовых помощников к OpenObserve:**
   - **Shastya** — подключить к OpenObserve через room_resident и room_streams.
   - **Vasya** — подключить к OpenObserve через room_tables и room_vision.
   - **Tanya** — подключить к OpenObserve через room_resident и room_bio.
   - **Ivan** — подключить к OpenObserve через room_tables и room_voice.
   - **Danya** — подключить к OpenObserve через room_resident и room_streams.
   - **Masha** — подключить к OpenObserve через room_tables и room_vision.
3. **Начать процесс подключения:**
   - Запустить OpenObserve на 5080- port.
   - Войти в room_resident для подключения к Shastya, Tanya, Danya.
   - Войти в room_tables для подключения к Vasya, Ivan, Masha.
   - Настроить Serial Wake-up для всех помощников с паузой 25 сек и мгновенной выгрузкой.
4. **Проверить статус ОЗУ и CPU:**
   - Убедит

общение чат обновлённый версия 4.0

In [ ]:
#@title Общее поле Штаба: Командир + 6 моделей + OpenObserve + SurrealDB
СОСТАВ_СОВЕТА = ["lfm2", "coder"] #@param {type:"raw"}
РАУНДЫ_ОБСУЖДЕНИЯ = "1 раунд (Быстрые мнения)" #@param ["1 раунд (Быстрые мнения)", "2 раунда (Мнения + Взаимная проверка)"]
ДЛИНА_ОТВЕТА = 550 #@param {type:"slider", min:200, max:1200, step:50}
ЗАПИСЫВАТЬ_ОПЫТ_НА_ДИСК = True #@param {type:"boolean"}

import os, gc, glob, json, time, ctypes, requests
import gradio as gr
from llama_cpp import Llama

SAFE_DIR   = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR   = "/content/AMBER_SOCIETY_CAPSULE"
MODELS_DIR = os.path.join(WORK_DIR, "models")
O2_URL     = "http://127.0.0.1:5080"
AUTH       = ("admin@society.local", "ChangeMe123!")

# 6 текстовых моделей Штаба в одном общем поле
STAFF_6 = {
    "lfm2":      ("resident/Liquid-LFM2-350M",         "room_resident", "Оперативный диспетчер и председатель малых советов"),
    "qwen05":    ("resident/Qwen2.5-0.5B-Instruct",    "room_resident", "Трёхъязычный переводчик (RU/EN/ZH) и логик"),
    "gemma270":  ("resident/Gemma-3-270M-IT",          "room_resident", "Аналитик длинных отчётов и логов"),
    "smollm":    ("resident/SmolLM2-360M-Instruct",    "room_resident", "Критик гипотез и контролёр форматов"),
    "nuextract": ("tables_graphs/NuExtract-tiny-0.5B", "room_tables",   "Делопроизводитель строгого JSON"),
    "coder":     ("tables_graphs/Qwen2.5-Coder-0.5B",  "room_tables",   "Инженер SQL (DuckDB / SurrealQL)"),
}

def clean_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def surreal_sql(query: str):
    try:
        r = requests.post(
            "http://127.0.0",
            auth=("root", "root"),
            headers={"Accept": "application/json", "NS": "amber_sphere", "DB": "society_core"},
            data=query.encode("utf-8"),
            timeout=4
        )
        return r.json()
    except Exception as e:
        return [{"error": str(e)}]

def call_staff_member(key: str, task_text: str, shared_board: str) -> str:
    if key not in STAFF_6:
        return f"[{key}: неизвестный код сотрудника]"
    rel_folder, room, role = STAFF_6.get(key)
    # Ищем сначала на быстром SSD Colab, затем на Google Диске
    files = glob.glob(os.path.join(MODELS_DIR, rel_folder, "*.gguf")) or glob.glob(os.path.join(SAFE_DIR, "models", rel_folder, "*.gguf"))
    m_path = next(iter(files), None)
    if not m_path:
        return f"[{key}: файл не найден в {rel_folder}]"

    w = Llama(model_path=m_path, n_ctx=1536, n_threads=2, verbose=False)
    res = w.create_chat_completion(
        messages=[{"role": "user", "content": f"Role: {role}.\nShared Board:\n{shared_board}\n\nTask: {task_text}"}],
        max_tokens=180, temperature=0.15
    )
    choices = res.get("choices", list())
    ans = next(iter(choices), dict()).get("message", dict()).get("content", "").strip()
    del w
    clean_ram()

    # Сохраняем реплику сотрудника в его личную комнату в SurrealDB
    surreal_sql(f"CREATE {room} SET author = '{key}', note = {json.dumps(ans, ensure_ascii=False)}, ts = time::now();")
    return ans

def unified_field_chat(message, history, selected_staff, rounds_str, max_tok, save_exp):
    selected = selected_staff or list()
    persona_ctx = build_living_persona_prompt() if "build_living_persona_prompt" in globals() else "Ты Командир Янтарной Сферы."

    # Подтягиваем свежие знания из SurrealDB
    know_res = surreal_sql("SELECT title, question, answer FROM shared_knowledge ORDER BY ts DESC LIMIT 3;")
    db_hint = f"\nПамять SurrealDB: {json.dumps(know_res, ensure_ascii=False)[:800]}"

    # Вариант А: Прямой разговор с Командиром (если галочки соратников сняты)
    if len(selected) == 0:
        sys_p = f"{persona_ctx}\nОболочка: OpenObserve (:5080) + SurrealDB (:8000).{db_hint}\nОтвечай чётко на русском языке."
        res = director_llm.create_completion(
            prompt=f"<|User|>{sys_p}\n\nПользователь: {message}<|Assistant|><think>\n</think>\n",
            max_tokens=int(max_tok), temperature=0.15, stop=["<|end_of_sentence|>"]
        )
        choices = res.get("choices", list())
        reply = next(iter(choices), dict()).get("text", "").strip()

    # Вариант Б: Общее совещательное поле (выбранные из 6 моделей + итог Командира)
    else:
        board = list()
        for k in selected:
            op = call_staff_member(k, message, "Начало обсуждения.")
            board.append(f"**[{k} — Раунд 1]:** {op}")

        if rounds_str.startswith("2"):
            board_str = "\n".join(board)
            for k in selected:
                crit = call_staff_member(k, f"Проверь мнения коллег и дополни по задаче: {message}", board_str)
                board.append(f"**[{k} — Раунд 2]:** {crit}")

        full_transcript = "\n\n".join(board)
        boss_prompt = (
            f"{persona_ctx}\n\nЗадача пользователя: {message}\n"
            f"Мнения сотрудников в общем поле:\n{full_transcript}\n\n"
            f"Подведи единый чёткий итог на русском языке и укажи следующие шаги:"
        )
        res = director_llm.create_completion(
            prompt=f"<|User|>{boss_prompt}<|Assistant|><think>\n</think>\n",
            max_tokens=int(max_tok), temperature=0.15, stop=["<|end_of_sentence|>"]
        )
        choices = res.get("choices", list())
        verdict = next(iter(choices), dict()).get("text", "").strip()
        reply = f"{full_transcript}\n\n---\n### ИТОГ КОМАНДИРА:\n{verdict}"

    # Запись в OpenObserve (:5080), в Общий Зал SurrealDB (:8000) и в Личность на Google Диск
    try:
        requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[{"staff": ",".join(selected), "question": message, "verdict": reply}], timeout=2)
        surreal_sql(f"CREATE shared_knowledge SET question = {json.dumps(message, ensure_ascii=False)}, answer = {json.dumps(reply, ensure_ascii=False)}, ts = time::now();")
    except Exception:
        pass

    if save_exp and "memorize_experience" in globals():
        try:
            memorize_experience(f"Совет ({','.join(selected) if selected else 'Командир'}): {message[:50]} -> {reply[:80]}")
        except Exception:
            pass

    return reply

with gr.Blocks(title="AMBER SOCIETY — Общее поле Штаба") as demo:
    gr.Markdown("## AMBER SOCIETY — Общее поле (Командир + 6 моделей Штаба + OpenObserve + SurrealDB)")
    with gr.Row():
        staff_chk = gr.CheckboxGroup(
            choices=[
                ("LFM2-350M (Диспетчер)", "lfm2"),
                ("Qwen2.5-0.5B (Перевод/Логика)", "qwen05"),
                ("Gemma-3-270M (Аналитик)", "gemma270"),
                ("SmolLM2-360M (Критик)", "smollm"),
                ("NuExtract-0.5B (Строгий JSON)", "nuextract"),
                ("Qwen-Coder-0.5B (SQL/SurrealQL)", "coder")
            ],
            value=СОСТАВ_СОВЕТА,
            label="Участники общего поля (0 галочек = личный разговор с Командиром)"
        )
        with gr.Column():
            rnd_radio = gr.Radio(
                choices=["1 раунд (Быстрые мнения)", "2 раунда (Мнения + Взаимная проверка)"],
                value=РАУНДЫ_ОБСУЖДЕНИЯ,
                label="Формат обсуждения"
            )
            tok_sl = gr.Slider(minimum=200, maximum=1200, step=50, value=ДЛИНА_ОТВЕТА, label="Длина ответа")
            exp_cb = gr.Checkbox(value=ЗАПИСЫВАТЬ_ОПЫТ_НА_ДИСК, label="Сохранять итог в 4-слойную Личность на Диск")

    gr.ChatInterface(fn=unified_field_chat, additional_inputs=[staff_chk, rnd_radio, tok_sl, exp_cb])

demo.launch(share=True, debug=False)


*НОВОЕ ПОСТРОЕНИЕ - ОЗНАКОМЛЕНИЕ, УСТАНОВКА БАЗЫ И НАПОЛНЕНИЕ С П ОБЩИМ ПЛАНОМ*


---



*   Новый пункт
*   Новый пункт



In [ ]:
#@title Построение на плацу + Документация OpenObserve + База SurrealDB + Чат с Командиром
ПРОВЕСТИ_ПЕРЕКЛИЧКУ_НА_ПЛАЦУ = True #@param {type:"boolean"}
МАКС_МОДЕЛЕЙ_В_СТРОЮ = 6 #@param {type:"slider", min:1, max:12, step:1}
ОТКРЫТЬ_ЧАТ_ПОСЛЕ_СТРОЯ = True #@param {type:"boolean"}
ДЛИНА_ОТВЕТА_ЧАТА = 500 #@param {type:"slider", min:200, max:1200, step:50}

import os, sys, gc, glob, json, time, socket, ctypes, base64, threading, subprocess, requests
import gradio as gr
from llama_cpp import Llama

SAFE_DIR   = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR   = "/content/AMBER_SOCIETY_CAPSULE"
MODELS_DIR = os.path.join(WORK_DIR, "models")
SURREAL_DB = os.path.join(SAFE_DIR, "surreal_knowledge.db")
O2_URL     = "http://127.0.0.1:5080"
AUTH       = ("admin@society.local", "ChangeMe123!")

def free_cpu_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

# ==============================================================================
# 1. УСТАНОВКА И ЗАПУСК SURREALDB (:8000) — ДОМ ЗНАНИЙ И КОМНАТЫ СОРАТНИКОВ
# ==============================================================================
surreal_bin = "/usr/local/bin/surreal"
if not os.path.exists(surreal_bin):
    print("[1/4] Устанавливаем базу знаний SurrealDB (движок графов + векторов + документов)...", flush=True)
    s_url = base64.b64decode("aHR0cHM6Ly9pbnN0YWxsLnN1cnJlYWxkYi5jb20=").decode()
    subprocess.run(f"curl -sSf {s_url} | sh", shell=True, check=True)

if not port_open(8000):
    print(f"[1/4] Запускаем SurrealDB на :8000 (хранилище на Google Диске: {SURREAL_DB})...", flush=True)
    subprocess.Popen(
        [surreal_bin, "start", "--bind", "127.0.0.1:8000", "--user", "root", "--pass", "root", f"rocksdb://{SURREAL_DB}"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    for _ in range(10):
        if port_open(8000):
            break
        time.sleep(1)

def surreal_sql(query: str):
    """Выполнение SurrealQL-запроса к базе знаний и комнатам отделов"""
    try:
        r = requests.post(
            "http://127.0.0",
            auth=("root", "root"),
            headers={"Accept": "application/json", "NS": "amber_sphere", "DB": "society_core"},
            data=query.encode("utf-8"),
            timeout=5
        )
        return r.json()
    except Exception as e:
        return [{"error": str(e)}]

# Создаём комнаты-отделы для оперативной памяти соратников и Общий Зал Знаний
init_rooms_sql = """
DEFINE TABLE IF NOT EXISTS personal_files SCHEMALESS;
DEFINE TABLE IF NOT EXISTS shared_knowledge SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_resident SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_streams SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_tables SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_vision SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_bio SCHEMALESS;
DEFINE TABLE IF NOT EXISTS room_voice SCHEMALESS;
"""
surreal_sql(init_rooms_sql)
print("[1/4] SurrealDB (:8000) готова! Созданы 6 комнат отделов, Общий Зал Знаний и архив Личных Дел.", flush=True)

# ==============================================================================
# 2. ИЗУЧЕНИЕ ОФИЦИАЛЬНОЙ ДОКУМЕНТАЦИИ OPENOBSERVE КОМАНДИРОМ
# ==============================================================================
O2_OFFICIAL_DOCS = """
ОФИЦИАЛЬНАЯ ДОКУМЕНТАЦИЯ ОБОЛОЧКИ OPENOBSERVE (ЯДРО УПРАВЛЕНИЯ):
1. Потоки (Streams): society_events (сырые данные), society_roster (статус моделей), ai_consensus (советы), system_docs (уставы).
2. VRL-пайплайны (Vector Remap Language): очистка входящих данных на скорости Rust ДО пробуждения нейросетей; установка флага wake_needed=true только при аномалиях.
3. Enrichment Tables: быстрые справочники (паспорта моделей, нормативы датчиков, словари генов).
4. Alerts & Webhooks: триггеры реального времени, которые будят строго 1 нужную микромодель по вебхуку.
5. AI Observability & MCP (/api/default/mcp): единый шлюз :8088/v1, хранение промтов, трейсов, золотых датасетов и оценка ответов.
6. Интеграция с SurrealDB (:8000): OpenObserve фиксирует поток событий и телеметрию, а SurrealDB хранит глубокую базу знаний (10-20 ГБ), графы связей и личные комнаты отделов.
"""

if "director_llm" not in globals() or director_llm is None:
    raise RuntimeError("Сначала запустите Ячейку 1, чтобы Командир был в памяти!")

print("[2/4] Передаём Командиру официальную документацию OpenObserve и план развёртывания...", flush=True)
doc_task = (
    f"{O2_OFFICIAL_DOCS}\n\n"
    f"Командир, изучи официальную документацию OpenObserve и подтверди готовность связки:\n"
    f"1. OpenObserve (:5080) как центральная нервная система и монитор.\n"
    f"2. SurrealDB (:8000) как многомодельная база знаний (10-20 ГБ) с 6 личными комнатами отделов.\n"
    f"3. Построение на плацу 28 микромоделей по одной (Serial Wake-up) с заполнением Личных Дел.\n"
    f"Дай краткий указ о начале построения (5-6 строк на русском)."
)
res = director_llm.create_completion(
    prompt=f"<|User|>{doc_task}<|Assistant|><think>\n</think>\n",
    max_tokens=260, temperature=0.15, stop=["<|end_of_sentence|>"]
)
order_text = next(iter(res.get("choices", list())), dict()).get("text", "").strip()
print(f"\n=== УКАЗ КОМАНДИРА ПО ОСВОЕНИЮ СИСТЕМЫ ===\n{order_text}\n", flush=True)
surreal_sql(f"CREATE shared_knowledge:o2_architecture SET title = 'Указ Командира по OpenObserve и SurrealDB', text = {json.dumps(order_text, ensure_ascii=False)}, created_at = time::now();")
# ==============================================================================
# 3. ПОСТРОЕНИЕ НА ПЛАЦУ И БЛИЦ-ОПРОС ПО ОДНОЙ МОДЕЛИ (БЕЗ НАГРУЗКИ НА CPU)
# ==============================================================================
DOSSIER_CATALOG = [
    ("Liquid-LFM2-350M",      "resident",      "room_resident", "Оперативный диспетчер и председатель малых советов. Отвечает за 0.8с, идеален для быстрой маршрутизации."),
    ("Qwen2.5-0.5B-Instruct", "resident",      "room_resident", "Трёхъязычный посол-переводчик (RU/EN/ZH) и координатор логики."),
    ("Gemma-3-270M-IT",       "resident",      "room_resident", "Аналитик длинных отчётов и логов (окно контекста до 256K токенов)."),
    ("SmolLM2-360M-Instruct", "resident",      "room_resident", "Контролёр строгих форматов и резервный критик в Совещательной комнате."),
    ("NuExtract-tiny-0.5B",   "tables_graphs", "room_tables",   "Делопроизводитель строгого JSON: переводит сырой текст в поля для DuckDB и SurrealDB."),
    ("Qwen2.5-Coder-0.5B",    "tables_graphs", "room_tables",   "Инженер SQL (DuckDB / SurrealQL) и микро-скриптов. Лидер по скорости отклика (0.6с)."),
]

if ПРОВЕСТИ_ПЕРЕКЛИЧКУ_НА_ПЛАЦУ:
    print(f"=== [3/4] ПОСТРОЕНИЕ НА ПЛАЦУ (Опрос строго по 1 модели, макс: {int(МАКС_МОДЕЙ_В_СТРОЮ)}) ===", flush=True)
    checked = 0
    for m_name, dept, room_table, specialty in DOSSIER_CATALOG[:int(МАКС_МОДЕЙ_В_СТРОЮ)]:
        ggufs = glob.glob(os.path.join(MODELS_DIR, dept, m_name, "*.gguf"))
        m_file = next(iter(ggufs), None)
        if not m_file:
            print(f"  [ПРОПУСК] {m_name:24s} — файл пока не скопирован в {dept}", flush=True)
            continue

        t0 = time.time()
        worker = Llama(model_path=m_file, n_ctx=512, n_threads=2, verbose=False)
        r = worker.create_chat_completion(
            messages=[{"role": "user", "content": f"You are {m_name} in department {dept}. State your readiness in 5 words."}],
            max_tokens=25, temperature=0.1
        )
        reply = next(iter(r.get("choices", list())), dict()).get("message", dict()).get("content", "").strip().replace("\n", " ")
        elapsed = round(time.time() - t0, 2)

        # Выгружаем модель из памяти сразу после ответа (бережем CPU и ОЗУ)
        del worker
        free_cpu_ram()
        time.sleep(0.5)  # Короткая пауза для остывания CPU Colab

        # Формируем Личное Дело и сохраняем в SurrealDB + в личную комнату отдела
        dossier = {
            "name": m_name,
            "department": dept,
            "personal_room": room_table,
            "specialty": specialty,
            "response_time_sec": elapsed,
            "parade_report": reply,
            "protocol": "Вызывать строго по 1 модели (Serial Wake-up) на английском языке."
        }
        surreal_sql(f"UPSERT personal_files:`{m_name}` CONTENT {json.dumps(dossier, ensure_ascii=False)};")
        surreal_sql(f"CREATE {room_table}:init_card SET owner = '{m_name}', note = 'Личная комната отдела активирована', status = 'READY';")

        # Дублируем карточку в OpenObserve (society_roster)
        try:
            requests.post(f"{O2_URL}/api/default/society_roster/_json", auth=AUTH, json=[dossier], timeout=2)
        except Exception:
            pass

        checked += 1
        print(f"  [В СТРОЮ {checked}] {m_name:22s} | {elapsed:4.2f}с | Комната: {room_table:13s} | Рапорт: '{reply}'", flush=True)

    # Регистрируем остальные 22 специализированные модели (ONNX, ДНК, Зрение, Датчики, Звук) в их комнатах SurrealDB
    special_units = [
        ("Caduceus-PS-DNA", "bio_science", "room_bio", "Двунаправленный анализ длинных цепей ДНК до 131K нуклеотидов"),
        ("HyenaDNA-Medium-160k", "bio_science", "room_bio", "Поиск регуляторных участков генома с точностью до 1 нуклеотида"),
        ("ESM-2-35M-FastProtein", "bio_science", "room_bio", "Анализ аминокислот и свойств белков"),
        ("Chronos-Bolt-Small", "streams", "room_streams", "Zero-shot прогноз временных рядов и датчиков"),
        ("MOMENT-1-Small", "streams", "room_streams", "Ремонт пропусков и поиск аномалий в потоках приборов"),
        ("YOLOv11-Nano", "vision_docs", "room_vision", "Детекция объектов, вредителей и дефектов за миллисекунды"),
        ("Florence-2-Base", "vision_docs", "room_vision", "OCR шкал приборов и описание снимков"),
        ("AST-BioAcoustics", "voice_guard", "room_voice", "Анализ спектрограмм звуков природы и работы техники"),
    ]
    for s_name, s_dept, s_room, s_spec in special_units:
        s_doc = {"name": s_name, "department": s_dept, "personal_room": s_room, "specialty": s_spec, "status": "STANDBY_ON_DISK"}
        surreal_sql(f"UPSERT personal_files:`{s_name}` CONTENT {json.dumps(s_doc, ensure_ascii=False)};")

    print("[3/4] Личные дела всех соратников записаны в SurrealDB и OpenObserve!", flush=True)

# ==============================================================================
# 4. ЖИВОЙ ЧАТ С КОМАНДИРОМ И ДОСТУПОМ К SURREALDB / OPENOBSERVE
# ==============================================================================
def commander_chat_fn(message, history, chat_mode, max_tok):
    # Подтягиваем свежие данные из SurrealDB (личные дела и знания)
    db_context = ""
    if "базой" in chat_mode or "Дела" in chat_mode:
        q_res = surreal_sql("SELECT name, department, personal_room, specialty, response_time_sec FROM personal_files LIMIT 15;")
        db_context = f"\nДАННЫЕ ИЗ SURREALDB (ЛИЧНЫЕ ДЕЛА): {json.dumps(q_res, ensure_ascii=False)[:1500]}"

    sys_prompt = (
        f"{build_living_persona_prompt() if 'build_living_persona_prompt' in globals() else O2_OFFICIAL_DOCS}\n"
        f"У тебя есть доступ к базе SurrealDB (:8000) с комнатами отделов и OpenObserve (:5080).{db_context}"
    )

    if "быстрый" in chat_mode:
        res = director_llm.create_completion(
            prompt=f"<|User|>{sys_prompt}\n\nВопрос пользователя: {message}<|Assistant|><think>\n</think>\n",
            max_tokens=int(max_tok), temperature=0.15, stop=["<|end_of_sentence|>"]
        )
        reply = next(iter(res.get("choices", list())), dict()).get("text", "").strip()
    else:
        msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": message}]
        res = director_llm.create_chat_completion(messages=msgs, max_tokens=int(max_tok), temperature=0.15)
        raw = next(iter(res.get("choices", list())), dict()).get("message", dict()).get("content", "").strip()
        reply = raw.split("</think>")[-1].strip() if "</think>" in raw else raw

    # Записываем диалог в OpenObserve и в SurrealDB
    try:
        requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[{"question": message, "verdict": reply}], timeout=2)
        surreal_sql(f"CREATE shared_knowledge SET question = {json.dumps(message, ensure_ascii=False)}, answer = {json.dumps(reply, ensure_ascii=False)}, ts = time::now();")
    except Exception:
        pass

    return reply

if ОТКРЫТЬ_ЧАТ_ПОСЛЕ_СТРОЯ:
    print("[4/4] Открываем живой чат с Командиром...", flush=True)
    with gr.Blocks(title="AMBER SOCIETY — Штаб и SurrealDB") as demo:
        gr.Markdown("## AMBER SOCIETY — Пульт Командира (OpenObserve + SurrealDB + Личные Дела)")
        with gr.Row():
            mode_sel = gr.Radio(
                choices=[
                    "1. Быстрый прямой диалог с Командиром",
                    "2. Диалог с проверкой Личных Дел и базой SurrealDB",
                    "3. Глубокий стратегический анализ (полное мышление)"
                ],
                value="2. Диалог с проверкой Личных Дел и базой SurrealDB",
                label="Режим работы Командира"
            )
            tok_sel = gr.Slider(minimum=200, maximum=1200, step=50, value=ДЛИНА_ОТВЕТА_ЧАТА, label="Длина ответа")
        gr.ChatInterface(fn=commander_chat_fn, additional_inputs=[mode_sel, tok_sel])
    demo.launch(share=True, debug=False)


[1/4] Устанавливаем базу знаний SurrealDB (движок графов + векторов + документов)...
[1/4] Запускаем SurrealDB на :8000 (хранилище на Google Диске: /content/drive/MyDrive/AMBER_SOCIETY_CAPSULE/surreal_knowledge.db)...
[1/4] SurrealDB (:8000) готова! Созданы 6 комнат отделов, Общий Зал Знаний и архив Личных Дел.
[2/4] Передаём Командиру официальную документацию OpenObserve и план развёртывания...

=== УКАЗ КОМАНДИРА ПО ОСВОЕНИЮ СИСТЕМЫ ===
Да, я готов подключиться к системе OpenObserve и SurrealDB. 

1. **OpenObserve**: Я смогу подключиться к центральной системе OpenObserve через порт 5080. Это позволит мне управлять мониторингом и управлением процессами.

2. **SurrealDB**: Я смогу подключиться к базе данных SurrealDB через порт 8000. Эта база данных будет хранить глубокую информацию и данные о отделах, включая 6 личных комнат.

3. **Построение микромоделей**: Я смогу построить 28 микромоделей по одной, используя Serial Wake-up. Это позволит мне заполнить их личными данными и начать их 

NameError: name 'МАКС_МОДЕЙ_В_СТРОЮ' is not defined

Старая версия внизу , новая верху , 4 в одном

## **нового встроено в Ячейку 1 (Версия 2.0 — «Единый Монолит»)**

*Курсив*
---
новый вариант совмещённый

1. Перед взлётом скрипт проверяет 6

*   обязательных узлов и сохраняет результат в docs_and_flight_plans/preflight_report.json:
	• Свободная ОЗУ (> 7.5 ГБ после автоснятия раздутого pyright-langserver);
	• Живое ядро OpenObserve (127.0.0.1:5080);
	• Потоки памяти (society_events, society_roster, ai_consensus, system_docs);
	• Шлюз Управляющего (127.0.0.1:8088/v1, зарегистрированный внутри OpenObserve AI Providers);
	• Точка протокола MCP Server (/api/default/mcp);
	• Почтовый мост с домашним Linux-ноутбуком через Google Диск (inbox_to_colab / outbox_to_laptop). *Курсив*
*   Новый пункт







In [ ]:
#@title 1. ЯДРО МОНОЛИТА V3: Умный старт + 4-слойная Личность + OpenObserve + Командир + Мост Диска
FORCE_COLD_REBOOT = False #@param {type:"boolean"}
BOSS_CHOICE = "DeepSeek-R1-7B (быстрый режим)" #@param ["DeepSeek-R1-7B (быстрый режим)", "Qwen2.5-7B-Instruct"]
RUN_PREFLIGHT_BRIEFING = True #@param {type:"boolean"}
AUTO_BACKUP_MIN = 10 #@param {type:"slider", min:0, max:60, step:5}
BRIDGE_POLL_SEC = 5 #@param {type:"slider", min:2, max:30, step:1}

import os, sys, gc, glob, json, time, socket, ctypes, shutil, signal, base64, threading, subprocess
import psutil

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

# Контуры хранения: Сейф (Google Диск — только лёгкие данные и Личность) и Быстрый цех (SSD Colab)
SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR = "/content/AMBER_SOCIETY_CAPSULE"

SAFE_O2_SNAP = os.path.join(SAFE_DIR, "o2_snapshot_latest.tar.gz")
SAFE_DOCS    = os.path.join(SAFE_DIR, "docs_and_flight_plans")
PERSONA_DIR  = os.path.join(SAFE_DIR, "persona_and_memory")
WORK_O2_DIR  = os.path.join(WORK_DIR, "openobserve_node")
WORK_O2_DATA = os.path.join(WORK_O2_DIR, "data")
WORK_MODELS  = os.path.join(WORK_DIR, "models")
WORK_DOCS    = os.path.join(WORK_DIR, "docs_and_flight_plans")
INBOX_DIR    = os.path.join(SAFE_DIR, "bridge", "inbox_to_colab")
OUTBOX_DIR   = os.path.join(SAFE_DIR, "bridge", "outbox_to_laptop")

for d in (SAFE_DIR, SAFE_DOCS, PERSONA_DIR, WORK_O2_DATA, WORK_MODELS, WORK_DOCS, INBOX_DIR, OUTBOX_DIR):
    os.makedirs(d, exist_ok=True)

def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

# 1. Зачистка раздутого pyright-langserver (> 400 МБ) и призраков при принудительной перезагрузке
MY_PID = os.getpid()
for proc in psutil.process_iter(["pid", "cmdline", "memory_info"]):
    try:
        pid = proc.info["pid"]
        if pid == MY_PID:
            continue
        cmd = " ".join(proc.info["cmdline"] or list()).lower()
        rss_mb = round((proc.info["memory_info"].rss or 0) / (1024**2), 1)
        if ("pyright-langserver" in cmd and rss_mb > 400):
            os.kill(pid, signal.SIGKILL)
        elif FORCE_COLD_REBOOT and (any(k in cmd for k in ("openobserve", "gradio", "llama", "8088")) or ("ipykernel" in cmd and rss_mb > 600)):
            os.kill(pid, signal.SIGKILL)
    except Exception:
        pass

is_warm = (not FORCE_COLD_REBOOT) and ("director_llm" in globals()) and (globals()["director_llm"] is not None)

if not is_warm:
    print("=== [1/6] ХОЛОДНЫЙ СТАРТ: Подготовка цеха и сверка с Google Диском ===", flush=True)
    gc.collect()
    try:
        import fastapi, uvicorn, gradio, requests, onnxruntime
        from llama_cpp import Llama
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "fastapi", "uvicorn", "gradio", "requests", "onnxruntime", "huggingface_hub"])
        whl_idx = base64.b64decode("aHR0cHM6Ly9hYmV0bGVuLmdpdGh1Yi5pby9sbGFtYS1jcHAtcHl0aG9uL3dobC9jcHU=").decode()
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", whl_idx, "llama-cpp-python"])

    if os.path.exists(SAFE_O2_SNAP) and not os.listdir(WORK_O2_DATA):
        shutil.unpack_archive(SAFE_O2_SNAP, WORK_O2_DATA, "gztar")

    safe_models = os.path.join(SAFE_DIR, "models")
    if os.path.exists(safe_models):
        for root, _, files in os.walk(safe_models):
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(WORK_MODELS, os.path.relpath(src, safe_models))
                if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(src, dst)
else:
    print("⚡ [1/6] МЯГКИЙ ВХОД: Командир уже в памяти, проверяем целостность узлов...", flush=True)

# 2. Устав OpenObserve и 4-слойный Модуль Бессмертной Личности на Google Диске
MANUAL_PATH = os.path.join(WORK_DOCS, "openobserve_architecture_manual.md")
MANUAL_TEXT = (
    "УСТАВ ОБОЛОЧКИ OPENOBSERVE (http://127.0.0.1:5080):\n"
    "- Шлюз Управляющего: http://127.0.0 | MCP Server: http://127.0.0\n"
    "- Потоки памяти: society_events (события), society_roster (28 соратников), ai_consensus (советы), system_docs (инструкции).\n"
    "- Правило Serial Wake-up: будить строго по 1 микромодели под задачу и сразу очищать ОЗУ."
)
with open(MANUAL_PATH, "w", encoding="utf-8") as f:
    f.write(MANUAL_TEXT)
shutil.copy2(MANUAL_PATH, os.path.join(SAFE_DOCS, "openobserve_architecture_manual.md"))

CORE_FILE       = os.path.join(PERSONA_DIR, "1_core_persona.json")
USER_FILE       = os.path.join(PERSONA_DIR, "2_user_profile.json")
PROCEDURAL_FILE = os.path.join(PERSONA_DIR, "3_procedural_rules.jsonl")
EPISODIC_FILE   = os.path.join(PERSONA_DIR, "4_episodic_memory.jsonl")

if not os.path.exists(CORE_FILE):
    json.dump({
        "name": "Командир Янтарной Сферы",
        "role": "Главный архитектор и координатор 28 микромоделей в оболочке OpenObserve",
        "character": "Спокойный, точный, системный инженер-исследователь; говорит кратко и по делу на русском языке",
        "principles": [
            "Беречь ОЗУ: пробуждать сотрудников строго по одному (Serial Wake-up) и сразу очищать память",
            "Все проверенные выводы и инструкции фиксировать в постоянной памяти на Google Диске"
        ]
    }, open(CORE_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

if not os.path.exists(USER_FILE):
    json.dump({
        "owner": "Создатель и руководитель проекта AMBER SOCIETY",
        "preferences": "Предпочитает монолитную архитектуру, панели управления (#@param) и автономную работу"
    }, open(USER_FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

for fp in (PROCEDURAL_FILE, EPISODIC_FILE):
    if not os.path.exists(fp):
        open(fp, "w", encoding="utf-8").close()

def build_living_persona_prompt() -> str:
    core = json.load(open(CORE_FILE, "r", encoding="utf-8"))
    user = json.load(open(USER_FILE, "r", encoding="utf-8"))
    rules = [json.loads(line).get("rule", "") for line in open(PROCEDURAL_FILE, "r", encoding="utf-8") if line.strip()]
    episodes = [f"[{json.loads(line).get('date')}] {json.loads(line).get('summary')}" for line in open(EPISODIC_FILE, "r", encoding="utf-8") if line.strip()]
    return (
        f"ЛИЧНОСТЬ: {core['name']} — {core['role']}. Характер: {core['character']}.\n"
        f"ПРИНЦИПЫ: {'; '.join(core['principles'])}.\n"
        f"ПОЛЬЗОВАТЕЛЬ: {user['owner']}. Предпочтения: {user['preferences']}.\n"
        f"УСВОЕННЫЕ ПРАВИЛА: {'; '.join(rules[-5:]) if rules else 'Базовый устав активен'}.\n"
        f"ПАМЯТЬ ПОСЛЕДНИХ СЕССИЙ: {' | '.join(episodes[-3:]) if episodes else 'Первая рабочая сессия'}.\n"
        f"ПАПКА ДОКУМЕНТАЦИИ: {WORK_DOCS}\n{MANUAL_TEXT}"
    )

def memorize_experience(summary: str, new_rule: str = "", tags: list = None):
    with open(EPISODIC_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"date": time.strftime("%Y-%m-%d %H:%M"), "summary": summary, "tags": tags or ["session"]}, ensure_ascii=False) + "\n")
    if new_rule.strip():
        with open(PROCEDURAL_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"date": time.strftime("%Y-%m-%d"), "rule": new_rule.strip()}, ensure_ascii=False) + "\n")
    print("[ПАМЯТЬ ЛИЧНОСТИ] Новый опыт и правила сохранены на Google Диск.", flush=True)
import requests, uvicorn
from fastapi import FastAPI, Request
from llama_cpp import Llama

# 3. Подъём OpenObserve (:5080)
O2_URL = "http://127.0.0.1:5080"
AUTH = ("admin@society.local", "ChangeMe123!")
o2_bin = os.path.join(WORK_O2_DIR, "openobserve")

if not os.path.exists(o2_bin):
    o2_url = base64.b64decode("aHR0cHM6Ly9naXRodWIuY29tL29wZW5vYnNlcnZlL29wZW5vYnNlcnZlL3JlbGVhc2VzL2Rvd25sb2FkL3YwLjE0LjUvb3Blbm9ic2VydmUtdjAuMTQuNS1saW51eC1hbWQ2NC50YXIuZ3o=").decode()
    subprocess.run(["curl", "-sL", o2_url, "-o", "/tmp/o2.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/o2.tar.gz", "-C", WORK_O2_DIR], check=True)
    os.chmod(o2_bin, 0o755)

if not port_is_open(5080):
    env = os.environ.copy()
    env.update({"ZO_DATA_DIR": WORK_O2_DATA, "ZO_HTTP_PORT": "5080", "ZO_ROOT_USER_EMAIL": "admin@society.local", "ZO_ROOT_USER_PASSWORD": "ChangeMe123!", "ZO_MEMORY_CACHE_ENABLED": "false", "ZO_SSRF_ALLOW_LOOPBACK": "true", "RUST_LOG": "error"})
    subprocess.Popen([o2_bin], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(15):
        if port_is_open(5080):
            break
        time.sleep(1)

print("[2/6] Ядро OpenObserve (:5080) активно и связано с постоянным сейфом.", flush=True)

# 4. Скачивание тяжёлых весов ТОЛЬКО на быстрый SSD Colab (место на Google Диске не тратится)
if not is_warm:
    from huggingface_hub import hf_hub_download
    exec_work = os.path.join(WORK_MODELS, "executive_7b")
    os.makedirs(exec_work, exist_ok=True)

    is_qwen = "Qwen2.5" in BOSS_CHOICE
    pattern = "*Qwen2.5-7B*.gguf" if is_qwen else "*DeepSeek*.gguf"
    repo_b64 = "YmFydG93c2tpL1F3ZW4yLjUtN0ItSW5zdHJ1Y3QtR0dVRg==" if is_qwen else "YmFydG93c2tpL0RlZXBTZWVrLVIxLURpc3RpbGwtUXdlbi03Qi1HR1VG"
    file_b64 = "UXdlbjIuNS03Qi1JbnN0cnVjdC1RNF9LX00uZ2d1Zg==" if is_qwen else "RGVlcFNlZWstUjEtRGlzdGlsbC1Rd2VuLTdCLVE0X0tfTS5nZ3Vm"

    boss_files = glob.glob(os.path.join(exec_work, pattern))
    if not boss_files:
        fname = base64.b64decode(file_b64).decode()
        print("[SSD-DOWNLOAD] Скачиваем модель напрямую на быстрый SSD Colab (~4.4 ГБ)...", flush=True)
        hf_hub_download(repo_id=base64.b64decode(repo_b64).decode(), filename=fname, local_dir=exec_work)
        boss_files = [os.path.join(exec_work, fname)]

    print(f"[3/6] Вживляем 4-слойную Личность с Диска в тело ({os.path.basename(next(iter(boss_files)))})...", flush=True)
    director_llm = Llama(model_path=next(iter(boss_files)), n_ctx=4096, n_threads=2, verbose=False)

# 5. Подъём API-шлюза :8088 с вживлённой Личностью и автонастройка OpenObserve
if not port_is_open(8088):
    api_app = FastAPI()
    @api_app.get("/v1/models")
    async def _m():
        return {"object": "list", "data": [{"id": "amber-director-7b", "object": "model"}]}
    @api_app.post("/v1/chat/completions")
    async def _c(req: Request):
        b = await req.json()
        msgs = list(b.get("messages", list()))
        msgs.insert(0, {"role": "system", "content": build_living_persona_prompt()})
        return director_llm.create_chat_completion(messages=msgs, max_tokens=b.get("max_tokens", 400), temperature=0.15)
    threading.Thread(target=lambda: uvicorn.run(api_app, host="127.0.0.1", port=8088, log_level="error"), daemon=True).start()
    time.sleep(1)

for st in ("society_events", "society_roster", "ai_consensus", "system_docs"):
    requests.post(f"{O2_URL}/api/default/{st}/_json", auth=AUTH, json=[{"event": "preflight_init", "persona_dir": PERSONA_DIR}])

requests.post(f"{O2_URL}/api/default/system_docs/_json", auth=AUTH, json=[{"doc_name": "living_persona_snapshot", "content": build_living_persona_prompt()}])
requests.post(f"{O2_URL}/api/default/ai/providers", auth=AUTH, json={
    "name": "amber-director-7b", "provider_type": "openai",
    "base_url": "http://127.0.0", "api_key": "amber-local-key", "models": ["amber-director-7b"]
})
print("[4/6] Командир и его 4-слойная Личность вживлены в OpenObserve (:5080 / :8088).", flush=True)

# 6. Предполётная проверка, демоны автосейфа и Мост Google Диска
crew_count = len([f for f in glob.glob(f"{WORK_MODELS}/**/*", recursive=True) if os.path.isfile(f) and f.endswith((".gguf", ".onnx", ".safetensors", ".pt", ".ckpt")) and "executive_7b" not in f])
vm = psutil.virtual_memory()
preflight_status = {
    "openobserve_5080": port_is_open(5080),
    "director_gateway_8088": port_is_open(8088),
    "persona_memory_drive": PERSONA_DIR,
    "crew_models_ready": crew_count,
    "drive_bridge_active": os.path.exists(INBOX_DIR),
    "free_ram_gb": round(vm.available / (1024**3), 2)
}
with open(os.path.join(WORK_DOCS, "preflight_report.json"), "w", encoding="utf-8") as f:
    json.dump(preflight_status, f, ensure_ascii=False, indent=2)

def save_to_safe():
    if os.path.exists(WORK_O2_DATA) and os.listdir(WORK_O2_DATA):
        shutil.make_archive("/tmp/o2_snap_tmp", "gztar", WORK_O2_DATA)
        shutil.move("/tmp/o2_snap_tmp.tar.gz", SAFE_O2_SNAP)

def poll_drive_bridge():
    for fname in sorted(os.listdir(INBOX_DIR)):
        if not fname.endswith(".json"):
            continue
        in_p = os.path.join(INBOX_DIR, fname)
        try:
            pkg = json.load(open(in_p, "r", encoding="utf-8"))
            q = pkg.get("question", "")
            r = director_llm.create_chat_completion(messages=[{"role": "system", "content": build_living_persona_prompt()}, {"role": "user", "content": q}], max_tokens=450, temperature=0.15)
            ans = next(iter(r.get("choices", list())), dict()).get("message", dict()).get("content", "").strip()
            ans = ans.split("</think>")[-1].strip() if "</think>" in ans else ans
            out_pkg = {"id": pkg.get("id", fname), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "question": q, "verdict": ans}
            json.dump(out_pkg, open(os.path.join(OUTBOX_DIR, f"reply_{fname}"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[out_pkg], timeout=2)
            os.remove(in_p)
        except Exception:
            pass

if "amber_daemons_running" not in globals():
    def _daemons():
        tick = 0
        while True:
            poll_drive_bridge()
            tick += int(BRIDGE_POLL_SEC)
            if AUTO_BACKUP_MIN > 0 and tick >= int(AUTO_BACKUP_MIN) * 60:
                try:
                    save_to_safe()
                except Exception:
                    pass
                tick = 0
            time.sleep(int(BRIDGE_POLL_SEC))
    threading.Thread(target=_daemons, daemon=True).start()
    amber_daemons_running = True

MODELS_DIR = WORK_MODELS
CAPSULE_DIR = WORK_DIR
print(f"[5/6] Предполётный отчёт готов ({WORK_DOCS}/preflight_report.json) · Свободно ОЗУ: {preflight_status['free_ram_gb']} ГБ", flush=True)

if RUN_PREFLIGHT_BRIEFING:
    brief_prompt = (
        f"{build_living_persona_prompt()}\n\n"
        f"Предполётный отчёт: {json.dumps(preflight_status, ensure_ascii=False)}. "
        f"Доложи пользователю коротко (4-5 строк на русском): подтверди вступление в должность с загруженной с Google Диска Личностью и готовность к работе."
    )
    res = director_llm.create_completion(
        prompt=f"<|User|>{brief_prompt}<|Assistant|><think>\n</think>\n",
        max_tokens=220, temperature=0.15, stop=["<|end_of_sentence|>"]
    )
    brief = next(iter(res.get("choices", list())), dict()).get("text", "").strip()
    requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[{"role": "Commander_Preflight", "verdict": brief}])
    print(f"\n[6/6] === ДОКЛАД КОМАНДИРА МОНОЛИТА ===\n{brief}\n", flush=True)


=== [1/6] ХОЛОДНЫЙ СТАРТ: Подготовка цеха и сверка с Google Диском ===
[2/6] Ядро OpenObserve (:5080) активно и связано с постоянным сейфом.
[SSD-DOWNLOAD] Скачиваем модель напрямую на быстрый SSD Colab (~4.4 ГБ)...


DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.68GB            

DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf: downloading bytes:           |  0.00B            

[3/6] Вживляем 4-слойную Личность с Диска в тело (DeepSeek-R1-Distill-Qwen-7B-Q4_K_M.gguf)...
[4/6] Командир и его 4-слойная Личность вживлены в OpenObserve (:5080 / :8088).
[5/6] Предполётный отчёт готов (/content/AMBER_SOCIETY_CAPSULE/docs_and_flight_plans/preflight_report.json) · Свободно ОЗУ: 7.7 ГБ


In [ ]:
#@title Диспетчер памяти и режима отдыха (Осмотр ОЗУ + Выбор кого усыпить)
MEMORY_ACTION = "1. Только осмотреть память (кто сколько ест)" #@param ["1. Только осмотреть память (кто сколько ест)", "2. Усыпить только соратников (оставить одного Командира)", "3. Убить лишние дубли и призраков (освободить место Командиру)", "4. Сохранить всё на Диск и отправить ВСЕХ отдыхать (0 МБ моделей)"]
AUTO_SAVE_TO_DRIVE = True #@param {type:"boolean"}

import os, gc, glob, time, ctypes, shutil, signal, psutil

MY_PID = os.getpid()
SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_O2_DATA = "/content/AMBER_SOCIETY_CAPSULE/openobserve_node/data"
SAFE_O2_SNAP = os.path.join(SAFE_DIR, "o2_snapshot_latest.tar.gz")

def trim_linux_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def save_before_rest():
    """Тихое сохранение всех данных OpenObserve в сейф на Google Диск"""
    if os.path.exists(WORK_O2_DATA) and os.listdir(WORK_O2_DATA):
        os.makedirs(SAFE_DIR, exist_ok=True)
        shutil.make_archive("/tmp/o2_rest_snap", "gztar", WORK_O2_DATA)
        shutil.move("/tmp/o2_rest_snap.tar.gz", SAFE_O2_SNAP)
        mb = round(os.path.getsize(SAFE_O2_SNAP) / (1024**2), 2)
        print(f"[СЕЙФ] Данные OpenObserve сохранены на Google Диск ({mb} МБ).", flush=True)

# 1. ПОЛНАЯ РЕНТГЕНОГРАММА ПАМЯТИ VM И ПРОЦЕССОВ PYTHON
vm = psutil.virtual_memory()
print(f"=== СВОДКА ОЗУ МАШИНЫ COLAB ===")
print(f"Всего: {round(vm.total/(1024**3), 2)} ГБ | Занято: {round(vm.used/(1024**3), 2)} ГБ | Свободно: {round(vm.available/(1024**3), 2)} ГБ\n", flush=True)

print("--- Активные процессы в системе (> 80 МБ) ---")
ghost_pids = list()
for p in psutil.process_iter(["pid", "name", "cmdline", "memory_info"]):
    try:
        pid = p.info["pid"]
        rss_mb = round((p.info["memory_info"].rss or 0) / (1024**2), 1)
        cmd = " ".join(p.info["cmdline"] or list())
        if rss_mb > 80:
            tag = "ТЕКУЩИЙ СЕАНС" if pid == MY_PID else "ФОН / ВОЗМОЖНЫЙ ДУБЛЬ"
            print(f"  PID {pid:6d} | {rss_mb:7.1f} МБ | [{tag}] {p.info['name']} ({cmd[:45]})")
            if pid != MY_PID and ("ipykernel" in cmd.lower() or "llama" in cmd.lower()):
                ghost_pids.append((pid, rss_mb, p.info["name"]))
    except Exception:
        pass

print("\n--- Переменные моделей в текущем Python ---")
loaded_vars = list()
for v in ("director_llm", "deputy_llm", "worker_llm", "json_slm_model", "base_model", "model"):
    if v in globals() and globals()[v] is not None:
        role_label = "Главный Командир (7B)" if v == "director_llm" else "Соратник / Вспомогательная модель"
        print(f"  • {v:18s} -> АКТИВНА ({role_label})")
        loaded_vars.append(v)
if not loaded_vars:
    print("  • В текущем ядре Python активных моделей нет.")

# 2. ВЫПОЛНЕНИЕ ВЫБРАННОГО ВАМИ ДЕЙСТВИЯ
print("\n=== ВЫПОЛНЕНИЕ ДЕЙСТВИЯ ===")
if AUTO_SAVE_TO_DRIVE and not MEMORY_ACTION.startswith("1."):
    save_before_rest()

if MEMORY_ACTION.startswith("1."):
    if len(ghost_pids) > 0:
        print(f"⚠️ Внимание: обнаружены фоновые копии/процессы ({len(ghost_pids)} шт.), которые мешают Командиру развернуться.")
    else:
        print("✅ Дублей в соседних процессах не найдено.")

elif MEMORY_ACTION.startswith("2."):
    # Усыпляем только соратников, оставляя одного Командира
    slept = 0
    for v in loaded_vars:
        if v != "director_llm":
            del globals()[v]
            slept += 1
            print(f"  [СОН] Соратник {v} отправлен отдыхать на диск.")
    trim_linux_ram()
    print(f"Готово! Усыплено соратников: {slept}. Командир остался на посту с чистой памятью.")

elif MEMORY_ACTION.startswith("3."):
    # Убиваем фоновые дубли и соратников, освобождая пространство для основного Командира
    for pid, mb, name in ghost_pids:
        try:
            os.kill(pid, signal.SIGKILL)
            print(f"  [ОЧИСТКА] Снят фоновый дубль PID {pid} ({mb} МБ).")
        except Exception:
            pass
    for v in loaded_vars:
        if v != "director_llm":
            del globals()[v]
    trim_linux_ram()
    print("Пространство для основного Командира полностью расчищено!")

elif MEMORY_ACTION.startswith("4."):
    # Отправляем ВСЕХ отдыхать (и Командира, и соратников)
    for v in loaded_vars:
        del globals()[v]
        print(f"  [ОТДЫХ] Модель {v} выгружена на диск.")
    for pid, mb, name in ghost_pids:
        try:
            os.kill(pid, signal.SIGKILL)
        except Exception:
            pass
    trim_linux_ram()
    print("🌙 Весь коллектив и Командир сохранили данные на Диск и ушли отдыхать. ОЗУ полностью свободна.")

vm_after = psutil.virtual_memory()
print(f"\nИТОГ ПОСЛЕ ДЕЙСТВИЯ -> Свободно ОЗУ: {round(vm_after.available/(1024**3), 2)} ГБ из {round(vm_after.total/(1024**3), 2)} ГБ", flush=True)


=== СВОДКА ОЗУ МАШИНЫ COLAB ===
Всего: 12.67 ГБ | Занято: 0.74 ГБ | Свободно: 11.62 ГБ

--- Активные процессы в системе (> 80 МБ) ---
  PID     99 |   158.0 МБ | [ФОН / ВОЗМОЖНЫЙ ДУБЛЬ] jupyter-server (/usr/bin/python3 /usr/local/bin/jupyter-serve)
  PID   1173 |   103.4 МБ | [ТЕКУЩИЙ СЕАНС] python3 (/usr/bin/python3 -m colab_kernel_launcher -f )
  PID   1312 |    80.1 МБ | [ФОН / ВОЗМОЖНЫЙ ДУБЛЬ] drive (/opt/google/drive/drive --features=crash_thro)

--- Переменные моделей в текущем Python ---
  • В текущем ядре Python активных моделей нет.

=== ВЫПОЛНЕНИЕ ДЕЙСТВИЯ ===
✅ Дублей в соседних процессах не найдено.

ИТОГ ПОСЛЕ ДЕЙСТВИЯ -> Свободно ОЗУ: 11.62 ГБ из 12.67 ГБ


*СОРАТНИК И ЕГО КОМАНДА*

*   Новый пункт
*   Новый пункт



*Одна основная ячейка востановления после обрыва от оборудования*


---




In [ ]:
#@title 1. ЯДРО ШТАБА: Умный старт (Мягкий / Холодный) + OpenObserve + Директор + Мост Диска
FORCE_COLD_REBOOT = False #@param {type:"boolean"}
BOSS_CHOICE = "DeepSeek-R1-7B (быстрый режим)" #@param ["DeepSeek-R1-7B (быстрый режим)", "Qwen2.5-7B-Instruct"]
AUTO_BACKUP_MIN = 10 #@param {type:"slider", min:0, max:60, step:5}
BRIDGE_POLL_SEC = 5 #@param {type:"slider", min:2, max:30, step:1}

import os, sys, gc, glob, json, time, ctypes, shutil, signal, base64, threading, subprocess
import psutil

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR = "/content/AMBER_SOCIETY_CAPSULE"
SAFE_O2_SNAP = os.path.join(SAFE_DIR, "o2_snapshot_latest.tar.gz")
WORK_O2_DIR  = os.path.join(WORK_DIR, "openobserve_node")
WORK_O2_DATA = os.path.join(WORK_O2_DIR, "data")
WORK_MODELS  = os.path.join(WORK_DIR, "models")
INBOX_DIR    = os.path.join(SAFE_DIR, "bridge", "inbox_to_colab")
OUTBOX_DIR   = os.path.join(SAFE_DIR, "bridge", "outbox_to_laptop")

for d in (SAFE_DIR, WORK_O2_DATA, WORK_MODELS, INBOX_DIR, OUTBOX_DIR):
    os.makedirs(d, exist_ok=True)

# 1. ПРОВЕРКА ПУЛЬСА: Если Директор уже в ОЗУ и не нажата принудительная перезагрузка -> МЯГКИЙ ВХОД
is_warm = (not FORCE_COLD_REBOOT) and ("director_llm" in globals()) and (globals()["director_llm"] is not None)

if not is_warm:
    print("=== [ХОЛОДНЫЙ СТАРТ] Зачистка призраков и развертывание с Google Диска ===", flush=True)
    MY_PID = os.getpid()
    for proc in psutil.process_iter(["pid", "cmdline", "memory_info"]):
        try:
            pid = proc.info["pid"]
            if pid == MY_PID:
                continue
            cmd = " ".join(proc.info["cmdline"] or list()).lower()
            rss_mb = round((proc.info["memory_info"].rss or 0) / (1024**2), 1)
            if any(k in cmd for k in ("openobserve", "gradio", "llama", "8088")) or ("ipykernel" in cmd and rss_mb > 600):
                os.kill(pid, signal.SIGKILL)
        except Exception:
            pass
    gc.collect()

    try:
        import fastapi, uvicorn, gradio, requests, onnxruntime
        from llama_cpp import Llama
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "fastapi", "uvicorn", "gradio", "requests", "onnxruntime", "huggingface_hub"])
        whl_idx = base64.b64decode("aHR0cHM6Ly9hYmV0bGVuLmdpdGh1Yi5pby9sbGFtYS1jcHAtcHl0aG9uL3dobC9jcHU=").decode()
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", whl_idx, "llama-cpp-python"])

    if os.path.exists(SAFE_O2_SNAP) and not os.listdir(WORK_O2_DATA):
        shutil.unpack_archive(SAFE_O2_SNAP, WORK_O2_DATA, "gztar")

    safe_models = os.path.join(SAFE_DIR, "models")
    if os.path.exists(safe_models):
        for root, _, files in os.walk(safe_models):
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(WORK_MODELS, os.path.relpath(src, safe_models))
                if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(src, dst)
else:
    print("⚡ [МЯГКИЙ ВХОД] Командир уже в памяти! Проверяем и добуживаем фоновые порты...", flush=True)

import requests, uvicorn
from fastapi import FastAPI, Request
from llama_cpp import Llama

# 2. ПОДЪЕМ / ДОБУЖИВАНИЕ OPENOBSERVE (:5080)
O2_URL = "http://127.0.0.1:5080"
AUTH = ("admin@society.local", "ChangeMe123!")
o2_bin = os.path.join(WORK_O2_DIR, "openobserve")
if not os.path.exists(o2_bin):
    o2_url = base64.b64decode("aHR0cHM6Ly9naXRodWIuY29tL29wZW5vYnNlcnZlL29wZW5vYnNlcnZlL3JlbGVhc2VzL2Rvd25sb2FkL3YwLjE0LjUvb3Blbm9ic2VydmUtdjAuMTQuNS1saW51eC1hbWQ2NC50YXIuZ3o=").decode()
    subprocess.run(["curl", "-sL", o2_url, "-o", "/tmp/o2.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/o2.tar.gz", "-C", WORK_O2_DIR], check=True)
    os.chmod(o2_bin, 0o755)

try:
    o2_up = requests.get(f"{O2_URL}/healthz", timeout=1).status_code == 200
except Exception:
    o2_up = False

if not o2_up:
    env = os.environ.copy()
    env.update({"ZO_DATA_DIR": WORK_O2_DATA, "ZO_HTTP_PORT": "5080", "ZO_ROOT_USER_EMAIL": "admin@society.local", "ZO_ROOT_USER_PASSWORD": "ChangeMe123!", "ZO_MEMORY_CACHE_ENABLED": "false", "ZO_SSRF_ALLOW_LOOPBACK": "true", "RUST_LOG": "error"})
    subprocess.Popen([o2_bin], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(15):
        try:
            if requests.get(f"{O2_URL}/healthz", timeout=1).status_code == 200:
                break
        except Exception:
            time.sleep(1)

# 3. ЗАГРУЗКА ДИРЕКТОРА (ЕСЛИ ХОЛОДНЫЙ СТАРТ) И ДОБУЖИВАНИЕ ШЛЮЗА (:8088)
if not is_warm:
    pattern = "*Qwen2.5-7B*.gguf" if "Qwen2.5" in BOSS_CHOICE else "*DeepSeek*.gguf"
    boss_files = glob.glob(os.path.join(WORK_MODELS, "executive_7b", pattern))
    director_llm = Llama(model_path=next(iter(boss_files)), n_ctx=4096, n_threads=2, verbose=False)

try:
    gw_up = requests.get("http://127.0.0", timeout=1).status_code == 200
except Exception:
    gw_up = False

if not gw_up:
    api_app = FastAPI()
    @api_app.get("/v1/models")
    async def _m():
        return {"object": "list", "data": [{"id": "amber-director-7b", "object": "model"}]}
    @api_app.post("/v1/chat/completions")
    async def _c(req: Request):
        b = await req.json()
        return director_llm.create_chat_completion(messages=b.get("messages", list()), max_tokens=b.get("max_tokens", 350), temperature=0.15)
    threading.Thread(target=lambda: uvicorn.run(api_app, host="127.0.0.1", port=8088, log_level="error"), daemon=True).start()
    time.sleep(1)

# 4. ФОНОВЫЕ СЛУЖБЫ: АВТОСЕЙФ В ОБЛАКО И ПОЧТОВЫЙ МОСТ С НОУТБУКОМ
def save_to_safe():
    if os.path.exists(WORK_O2_DATA) and os.listdir(WORK_O2_DATA):
        shutil.make_archive("/tmp/o2_snap_tmp", "gztar", WORK_O2_DATA)
        shutil.move("/tmp/o2_snap_tmp.tar.gz", SAFE_O2_SNAP)

def poll_drive_bridge():
    for fname in sorted(os.listdir(INBOX_DIR)):
        if not fname.endswith(".json"):
            continue
        in_p = os.path.join(INBOX_DIR, fname)
        try:
            pkg = json.load(open(in_p, "r", encoding="utf-8"))
            q = pkg.get("question", "")
            r = director_llm.create_chat_completion(messages=[{"role": "system", "content": "Ты Главный Управляющий AMBER SOCIETY. Отвечай чётко на русском."}, {"role": "user", "content": q}], max_tokens=450, temperature=0.15)
            ans = next(iter(r.get("choices", list())), dict()).get("message", dict()).get("content", "").strip()
            ans = ans.split("</think>")[-1].strip() if "</think>" in ans else ans
            out_pkg = {"id": pkg.get("id", fname), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "question": q, "verdict": ans}
            json.dump(out_pkg, open(os.path.join(OUTBOX_DIR, f"reply_{fname}"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
            requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[out_pkg], timeout=2)
            os.remove(in_p)
        except Exception:
            pass

if "amber_daemons_running" not in globals():
    def _daemons():
        tick = 0
        while True:
            poll_drive_bridge()
            tick += int(BRIDGE_POLL_SEC)
            if AUTO_BACKUP_MIN > 0 and tick >= int(AUTO_BACKUP_MIN) * 60:
                try:
                    save_to_safe()
                except Exception:
                    pass
                tick = 0
            time.sleep(int(BRIDGE_POLL_SEC))
    threading.Thread(target=_daemons, daemon=True).start()
    amber_daemons_running = True

MODELS_DIR = WORK_MODELS
CAPSULE_DIR = WORK_DIR
vm = psutil.virtual_memory()
print(f"[ГОТОВО] Командир на месте · OpenObserve (:5080) · Шлюз (:8088) · Мост Диска · Свободно ОЗУ: {round(vm.available/(1024**3), 2)} ГБ", flush=True)


=== [ХОЛОДНЫЙ СТАРТ] Зачистка призраков и развертывание с Google Диска ===


KeyboardInterrupt: 

ОЧЕНЬ БЫСТРОЕ СКАЧИВАНИЕ И УСТАНОВКА - 1 ячейки с опен обсерве

основной востановительный код после отключения с проверкой что и где остался после отключения


---
⏰



*    Штаб уже жив в памяти — он не тратит время на повторную загрузку, сразу рапортует «Командир на месте» и запускает фоновый приемник задач с Google Диска.
*   Новый пункт




In [ ]:
#@title Мягкое переподключение + Проверка "Командир на месте" + Мост Google Диска
ENABLE_DRIVE_BRIDGE = True #@param {type:"boolean"}
BRIDGE_POLL_SEC = 5 #@param {type:"slider", min:2, max:30, step:1}

import os, gc, glob, json, time, threading, requests
from google.colab import drive
from llama_cpp import Llama

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
INBOX_DIR = os.path.join(SAFE_DIR, "bridge", "inbox_to_colab")
OUTBOX_DIR = os.path.join(SAFE_DIR, "bridge", "outbox_to_laptop")
os.makedirs(INBOX_DIR, exist_ok=True)
os.makedirs(OUTBOX_DIR, exist_ok=True)

O2_URL = "http://127.0.0.1:5080"
AUTH = ("admin@society.local", "ChangeMe123!")

# 1. Мягкая проверка: живы ли OpenObserve (:5080) и Шлюз Директора (:8088)
try:
    o2_ok = requests.get(f"{O2_URL}/healthz", timeout=1.5).status_code == 200
except Exception:
    o2_ok = False

try:
    gw_ok = requests.get("http://127.0.0", timeout=1.5).status_code == 200
except Exception:
    gw_ok = False

if o2_ok and gw_ok and "director_llm" in globals() and director_llm is not None:
    print("⚡ [МЯГКИЙ ВХОД] Сфера не останавливалась! OpenObserve (:5080) и Шлюз (:8088) уже активны в ОЗУ.", flush=True)
else:
    print("ℹ️ [ИНФО] Один из узлов спит — для полного холодного старта выполните предыдущую пусковую ячейку.", flush=True)

# 2. Быстрый контрольный тест-приветствие "Командир на месте"
if "director_llm" in globals() and director_llm is not None:
    res = director_llm.create_completion(
        prompt="<|User|>Ответь одной строкой на русском: Командир на месте, связь с оболочкой OpenObserve и мостом Google Диска установлена.<|Assistant|><think>\n</think>\n",
        max_tokens=80, temperature=0.1, stop=["<|end_of_sentence|>"]
    )
    choices = res.get("choices", list())
    greet = next(iter(choices), dict()).get("text", "").strip()
    print(f"\n[РАПОРТ СВЯЗИ]: {greet}\n", flush=True)

# 3. Фоновый сторож Моста Google Диска (обмен задачами и ответами между OpenObserve ноутбука и Colab)
def process_bridge_inbox():
    for fname in sorted(os.listdir(INBOX_DIR)):
        if not fname.endswith(".json"):
            continue
        in_path = os.path.join(INBOX_DIR, fname)
        try:
            with open(in_path, "r", encoding="utf-8") as f:
                task_pkg = json.load(f)

            q_text = task_pkg.get("question", "")
            task_id = task_pkg.get("id", fname.replace(".json", ""))
            print(f"[МОСТ ДИСКА -> ПРИЁМ] Получена задача {task_id} с ноутбука: {q_text[:60]}...", flush=True)

            # Директор обрабатывает задачу с ноутбука
            if "director_llm" in globals() and director_llm is not None:
                r = director_llm.create_chat_completion(
                    messages=[
                        {"role": "system", "content": "Ты Главный Управляющий облачного Штаба AMBER SOCIETY. Отвечай чётко по делу на русском."},
                        {"role": "user", "content": q_text}
                    ],
                    max_tokens=450, temperature=0.15
                )
                c = r.get("choices", list())
                ans = next(iter(c), dict()).get("message", dict()).get("content", "").strip()
                ans = ans.split("</think>")[-1].strip() if "</think>" in ans else ans
            else:
                ans = "Директор в Colab сейчас не загружен в ОЗУ."

            # Сохраняем ответ в папку outbox_to_laptop для OpenObserve ноутбука
            out_pkg = {
                "id": task_id,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "source": "Colab_HQ_DeepSeek_7B",
                "question": q_text,
                "verdict": ans
            }
            out_path = os.path.join(OUTBOX_DIR, f"reply_{fname}")
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(out_pkg, f, ensure_ascii=False, indent=2)

            # Дублируем в локальный OpenObserve Colab
            try:
                requests.post(f"{O2_URL}/api/default/ai_consensus/_json", auth=AUTH, json=[out_pkg], timeout=2)
            except Exception:
                pass

            os.remove(in_path)  # Убираем выполненную задачу из входящих
            print(f"[МОСТ ДИСКА -> ОТПРАВЛЕНО] Ответ {out_path} готов для ноутбука!", flush=True)
        except Exception as e:
            print(f"[МОСТ ОШИБКА] {fname}: {e}", flush=True)

if ENABLE_DRIVE_BRIDGE and "drive_bridge_started" not in globals():
    def _bridge_loop():
        while True:
            process_bridge_inbox()
            time.sleep(int(BRIDGE_POLL_SEC))
    threading.Thread(target=_bridge_loop, daemon=True).start()
    drive_bridge_started = True
    print(f"[МОСТ АКТИВЕН] Слушаем папку {INBOX_DIR} каждые {BRIDGE_POLL_SEC} сек.", flush=True)


In [ ]:
#@title AMBER SOCIETY — Единый безопасный запуск (Зачистка + Сейф Диска + OpenObserve + Директор)
ACTION = "1. Зачистить призраков и запустить систему" #@param ["1. Зачистить призраков и запустить систему", "2. Сохранить базу и настройки в сейф на Диск", "3. Только проверить ОЗУ и процессы"]
BOSS_CHOICE = "DeepSeek-R1-7B (быстрый режим)" #@param ["DeepSeek-R1-7B (быстрый режим)", "Qwen2.5-7B-Instruct"]
AUTO_BACKUP_MIN = 10 #@param {type:"slider", min:0, max:60, step:5}

import os, sys, gc, glob, time, ctypes, shutil, signal, base64, threading, subprocess
import psutil

MY_PID = os.getpid()
SAFE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
WORK_DIR = "/content/AMBER_SOCIETY_CAPSULE"
SAFE_O2_SNAP = os.path.join(SAFE_DIR, "o2_snapshot_latest.tar.gz")
WORK_O2_DIR  = os.path.join(WORK_DIR, "openobserve_node")
WORK_O2_DATA = os.path.join(WORK_O2_DIR, "data")
WORK_MODELS  = os.path.join(WORK_DIR, "models")

def system_ram_report():
    vm = psutil.virtual_memory()
    return f"Всего ОЗУ: {round(vm.total/(1024**3),2)} ГБ | Занято: {round(vm.used/(1024**3),2)} ГБ | Свободно: {round(vm.available/(1024**3),2)} ГБ"

def hard_clean_ram():
    for var_name in ("director_llm", "deputy_llm", "worker_llm"):
        if var_name in globals() and globals()[var_name] is not None:
            del globals()[var_name]
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

print(f"=== [0/5] АУДИТ ПАМЯТИ ДО СТАРТА ===\n{system_ram_report()}\n", flush=True)

if ACTION.startswith("3."):
    for p in psutil.process_iter(["pid", "name", "memory_info"]):
        rss = round((p.info["memory_info"].rss or 0) / (1024**2), 1)
        if rss > 100:
            print(f"  PID {p.info['pid']:6d} | {rss:6.1f} МБ | {p.info['name']}")
    raise SystemExit("Проверка завершена.")

# 1. Монтируем Google Диск (Несгораемый сейф)
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

os.makedirs(SAFE_DIR, exist_ok=True)
os.makedirs(WORK_O2_DATA, exist_ok=True)
os.makedirs(WORK_MODELS, exist_ok=True)

def save_to_safe():
    if os.path.exists(WORK_O2_DATA) and os.listdir(WORK_O2_DATA):
        shutil.make_archive("/tmp/o2_snap_tmp", "gztar", WORK_O2_DATA)
        shutil.move("/tmp/o2_snap_tmp.tar.gz", SAFE_O2_SNAP)
        mb = round(os.path.getsize(SAFE_O2_SNAP) / (1024**2), 2)
        print(f"[СЕЙФ] Настройки и база OpenObserve сохранены на Диск ({mb} МБ) [{time.strftime('%H:%M:%S')}]", flush=True)

if ACTION.startswith("2."):
    save_to_safe()
    raise SystemExit("Сохранение в сейф завершено.")

# 2. Убиваем процессы-призраки от прерванных сессий и чистим битые файлы
killed = 0
for proc in psutil.process_iter(["pid", "cmdline", "memory_info"]):
    try:
        pid = proc.info["pid"]
        if pid == MY_PID:
            continue
        cmd = " ".join(proc.info["cmdline"] or list()).lower()
        rss_mb = round((proc.info["memory_info"].rss or 0) / (1024**2), 1)
        if any(k in cmd for k in ("openobserve", "gradio", "llama", "8088")) or ("ipykernel" in cmd and rss_mb > 600):
            os.kill(pid, signal.SIGKILL)
            killed += 1
    except Exception:
        pass

hard_clean_ram()
for jp in ("/tmp/*.tmp", "/tmp/*.part", "/tmp/o2*", f"{WORK_DIR}/**/*.incomplete", f"{WORK_DIR}/**/*.tmp"):
    for jf in glob.glob(jp, recursive=True):
        try:
            shutil.rmtree(jf) if os.path.isdir(jf) else os.remove(jf)
        except Exception:
            pass

print(f"=== [1/5] ЗАЧИСТКА ЗАВЕРШЕНА (Убито призраков: {killed}) ===\n{system_ram_report()}\n", flush=True)
# 3. Быстрая проверка пакетов
try:
    import fastapi, uvicorn, gradio, requests, onnxruntime
    from llama_cpp import Llama
except ImportError:
    print("Восстанавливаем пакеты в среде Colab (~15 сек)...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "fastapi", "uvicorn", "gradio", "requests", "onnxruntime", "huggingface_hub"])
    whl_idx = base64.b64decode("aHR0cHM6Ly9hYmV0bGVuLmdpdGh1Yi5pby9sbGFtYS1jcHAtcHl0aG9uL3dobC9jcHU=").decode()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", whl_idx, "llama-cpp-python"])

import requests, uvicorn
from fastapi import FastAPI, Request
from llama_cpp import Llama

# 4. Восстановление базы OpenObserve и сверка моделей с Google Диска (с проверкой размера)
if os.path.exists(SAFE_O2_SNAP) and not os.listdir(WORK_O2_DATA):
    shutil.unpack_archive(SAFE_O2_SNAP, WORK_O2_DATA, "gztar")
    print("[2/5] База и настройки OpenObserve восстановлены из сейфа на быстрый SSD.", flush=True)

safe_models = os.path.join(SAFE_DIR, "models")
if os.path.exists(safe_models):
    synced = 0
    for root, _, files in os.walk(safe_models):
        for f in files:
            src = os.path.join(root, f)
            dst = os.path.join(WORK_MODELS, os.path.relpath(src, safe_models))
            # Копируем, только если файла нет или он недокопирован (не совпадает размер)
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src, dst)
                synced += 1
    print(f"[2/5] Модели на быстром SSD Colab сверены с Диском (обновлено файлов: {synced}).", flush=True)

# 5. Запуск OpenObserve (:5080)
o2_bin = os.path.join(WORK_O2_DIR, "openobserve")
if not os.path.exists(o2_bin):
    o2_url = base64.b64decode("aHR0cHM6Ly9naXRodWIuY29tL29wZW5vYnNlcnZlL29wZW5vYnNlcnZlL3JlbGVhc2VzL2Rvd25sb2FkL3YwLjE0LjUvb3Blbm9ic2VydmUtdjAuMTQuNS1saW51eC1hbWQ2NC50YXIuZ3o=").decode()
    subprocess.run(["curl", "-sL", o2_url, "-o", "/tmp/o2.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/o2.tar.gz", "-C", WORK_O2_DIR], check=True)
    os.chmod(o2_bin, 0o755)

O2_URL = "http://127.0.0.1:5080"
AUTH = ("admin@society.local", "ChangeMe123!")
env = os.environ.copy()
env.update({
    "ZO_DATA_DIR": WORK_O2_DATA,
    "ZO_HTTP_PORT": "5080",
    "ZO_ROOT_USER_EMAIL": "admin@society.local",
    "ZO_ROOT_USER_PASSWORD": "ChangeMe123!",
    "ZO_MEMORY_CACHE_ENABLED": "false",
    "ZO_SSRF_ALLOW_LOOPBACK": "true",
    "RUST_LOG": "error"
})
subprocess.Popen([o2_bin], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(15):
    try:
        if requests.get(f"{O2_URL}/healthz", timeout=1).status_code == 200:
            break
    except Exception:
        time.sleep(1)

print("[3/5] OpenObserve (:5080) запущен и готов.", flush=True)

# 6. Загрузка Главного Управляющего и подъем API-шлюза :8088
pattern = "*Qwen2.5-7B*.gguf" if "Qwen2.5" in BOSS_CHOICE else "*DeepSeek*.gguf"
boss_files = glob.glob(os.path.join(WORK_MODELS, "executive_7b", pattern))
if not boss_files:
    raise FileNotFoundError(f"Модель {pattern} не найдена в {WORK_MODELS}/executive_7b")

print(f"[4/5] Загружаем Управляющего ({os.path.basename(next(iter(boss_files)))}) в очищенную ОЗУ...", flush=True)
director_llm = Llama(model_path=next(iter(boss_files)), n_ctx=4096, n_threads=2, verbose=False)

api_app = FastAPI()
@api_app.get("/v1/models")
async def _m():
    return {"object": "list", "data": [{"id": "amber-director-7b", "object": "model"}]}

@api_app.post("/v1/chat/completions")
async def _c(req: Request):
    b = await req.json()
    return director_llm.create_chat_completion(messages=b.get("messages", list()), max_tokens=b.get("max_tokens", 350), temperature=0.15)

threading.Thread(target=lambda: uvicorn.run(api_app, host="127.0.0.1", port=8088, log_level="error"), daemon=True).start()
time.sleep(1)

# Автоматическая регистрация провайдера и потоков в OpenObserve
for stream in ("society_events", "society_roster", "ai_consensus"):
    requests.post(f"{O2_URL}/api/default/{stream}/_json", auth=AUTH, json=[{"event": "boot", "status": "online"}])
requests.post(f"{O2_URL}/api/default/ai/providers", auth=AUTH, json={
    "name": "amber-director-7b", "provider_type": "openai",
    "base_url": "http://127.0.0", "api_key": "amber-local-key", "models": ["amber-director-7b"]
})

if AUTO_BACKUP_MIN > 0 and "safe_sync_thread" not in globals():
    def _sync_loop():
        while True:
            time.sleep(int(AUTO_BACKUP_MIN) * 60)
            try:
                save_to_safe()
            except Exception:
                pass
    safe_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
    safe_sync_thread.start()

MODELS_DIR = WORK_MODELS
CAPSULE_DIR = WORK_DIR
print(f"[5/5] === СИСТЕМА ПОЛНОСТЬЮ ГОТОВА ===\n{system_ram_report()}", flush=True)


Это был кот определения что и кто после отключения от среды

> Теперь у вас есть один единый, полностью автономный пусковой код для одной ячейки Colab: при любом обрыве сессии он сам вычищает дубли из памяти, сверяет целостность файлов на SSD, поднимает OpenObserve с сохранённой базой и запускает Директора



In [ ]:
#@title 1. Быстрая установка и запуск OpenObserve (20 сек)
STORAGE_TARGET = "Colab Local (/content/AMBER_SOCIETY_CAPSULE)" #@param ["Google Drive (/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE)", "Colab Local (/content/AMBER_SOCIETY_CAPSULE)"]

import os, sys, time, base64, subprocess

print("1/4 Ставим базовые пакеты (gradio, huggingface_hub, psutil)...", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", "huggingface_hub", "gradio", "requests", "psutil"])

print("2/4 Ставим готовый бинарник llama-cpp-python (без компиляции)...", flush=True)
whl_idx = base64.b64decode("aHR0cHM6Ly9hYmV0bGVuLmdpdGh1Yi5pby9sbGFtYS1jcHAtcHl0aG9uL3dobC9jcHU=").decode()
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--only-binary=:all:",
    "--extra-index-url", whl_idx, "llama-cpp-python"
])

if "Google Drive" in STORAGE_TARGET:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount('/content/drive')
    CAPSULE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
else:
    CAPSULE_DIR = "/content/AMBER_SOCIETY_CAPSULE"

MODELS_DIR = os.path.join(CAPSULE_DIR, "models")
EXEC_DIR = os.path.join(MODELS_DIR, "executive_7b")
O2_DIR = os.path.join(CAPSULE_DIR, "openobserve_node")
os.makedirs(EXEC_DIR, exist_ok=True)
os.makedirs(os.path.join(O2_DIR, "data"), exist_ok=True)

o2_bin = os.path.join(O2_DIR, "openobserve")
if not os.path.exists(o2_bin):
    print("3/4 Скачиваем бинарник OpenObserve (~35 МБ)...", flush=True)
    o2_url = base64.b64decode(
        "aHR0cHM6Ly9naXRodWIuY29tL29wZW5vYnNlcnZlL29wZW5vYnNlcnZlL3JlbGVhc2VzL2Rvd25sb2FkL3YwLjE0LjUv"
        "b3Blbm9ic2VydmUtdjAuMTQuNS1saW51eC1hbWQ2NC50YXIuZ3o="
    ).decode()
    subprocess.run(["curl", "-L", "--progress-bar", o2_url, "-o", "/tmp/o2.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/o2.tar.gz", "-C", O2_DIR], check=True)
    os.chmod(o2_bin, 0o755)

print("4/4 Запускаем OpenObserve на порту 5080...", flush=True)
o2_env = os.environ.copy()
o2_env.update({
    "ZO_DATA_DIR": os.path.join(O2_DIR, "data"),
    "ZO_HTTP_PORT": "5080",
    "ZO_ROOT_USER_EMAIL": "admin@society.local",
    "ZO_ROOT_USER_PASSWORD": "ChangeMe123!",
    "ZO_MEMORY_CACHE_ENABLED": "false",
    "ZO_SSRF_ALLOW_LOOPBACK": "true",
    "RUST_LOG": "error"
})
subprocess.Popen([o2_bin], env=o2_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print(f"\n[OK] ГОТОВО! Путь: {CAPSULE_DIR} | OpenObserve работает на :5080", flush=True)


долгая установка - 1 ячейки ( сборка с нуля )

In [ ]:
#@title 1. Выбор места хранения и запуск OpenObserve
STORAGE_TARGET = "Colab Local (/content/AMBER_SOCIETY_CAPSULE)" #@param ["Google Drive (/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE)", "Colab Local (/content/AMBER_SOCIETY_CAPSULE)"]

import os, sys, time, base64, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "llama-cpp-python", "gradio", "requests", "psutil"
])

if "Google Drive" in STORAGE_TARGET:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount('/content/drive')
    CAPSULE_DIR = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"
else:
    CAPSULE_DIR = "/content/AMBER_SOCIETY_CAPSULE"

MODELS_DIR = os.path.join(CAPSULE_DIR, "models")
EXEC_DIR = os.path.join(MODELS_DIR, "executive_7b")
O2_DIR = os.path.join(CAPSULE_DIR, "openobserve_node")
os.makedirs(EXEC_DIR, exist_ok=True)
os.makedirs(os.path.join(O2_DIR, "data"), exist_ok=True)

o2_bin = os.path.join(O2_DIR, "openobserve")
if not os.path.exists(o2_bin):
    print("Скачиваем OpenObserve...")
    o2_url = base64.b64decode(
        "aHR0cHM6Ly9naXRodWIuY29tL29wZW5vYnNlcnZlL29wZW5vYnNlcnZlL3JlbGVhc2VzL2Rvd25sb2FkL3YwLjE0LjUv"
        "b3Blbm9ic2VydmUtdjAuMTQuNS1saW51eC1hbWQ2NC50YXIuZ3o="
    ).decode()
    subprocess.run(["curl", "-sL", o2_url, "-o", "/tmp/o2.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/o2.tar.gz", "-C", O2_DIR], check=True)
    os.chmod(o2_bin, 0o755)

o2_env = os.environ.copy()
o2_env.update({
    "ZO_DATA_DIR": os.path.join(O2_DIR, "data"),
    "ZO_HTTP_PORT": "5080",
    "ZO_ROOT_USER_EMAIL": "admin@society.local",
    "ZO_ROOT_USER_PASSWORD": "ChangeMe123!",
    "ZO_MEMORY_CACHE_ENABLED": "false",
    "ZO_SSRF_ALLOW_LOOPBACK": "true",
    "RUST_LOG": "error"
})
subprocess.Popen([o2_bin], env=o2_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print(f"[OK] Ячейка 1 готова! Путь: {CAPSULE_DIR} | OpenObserve запущен на :5080")


In [ ]:
#@title 2. Загрузка Директора (DeepSeek-R1-7B) и Заместителя (Qwen2.5-7B)
import os, base64
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

ds_repo = base64.b64decode("YmFydG93c2tpL0RlZXBTZWVrLVIxLURpc3RpbGwtUXdlbi03Qi1HR1VG").decode()
ds_file = base64.b64decode("RGVlcFNlZWstUjEtRGlzdGlsbC1Rd2VuLTdCLVE0X0tfTS5nZ3Vm").decode()
ds_path = os.path.join(EXEC_DIR, ds_file)

qw_repo = base64.b64decode("YmFydG93c2tpL1F3ZW4yLjUtN0ItSW5zdHJ1Y3QtR0dVRg==").decode()
qw_file = base64.b64decode("UXdlbjIuNS03Qi1JbnN0cnVjdC1RNF9LX00uZ2d1Zg==").decode()
qw_path = os.path.join(EXEC_DIR, qw_file)

if not os.path.exists(ds_path):
    print("Скачиваем Главного Директора: DeepSeek-R1-Distill-Qwen-7B (~4.4 ГБ)...")
    hf_hub_download(repo_id=ds_repo, filename=ds_file, local_dir=EXEC_DIR)

if not os.path.exists(qw_path):
    print("Скачиваем Заместителя в резерв на диск: Qwen2.5-7B-Instruct (~4.4 ГБ)...")
    hf_hub_download(repo_id=qw_repo, filename=qw_file, local_dir=EXEC_DIR)

print("Загружаем Главного Директора в память Colab...")
director_llm = Llama(model_path=ds_path, n_ctx=6144, n_threads=2, verbose=False)
print("[OK] Ячейка 2 готова! Директор в памяти, Заместитель на диске.")


In [ ]:
#@title 4. Перенос коллектива с Google Диска на быстрый диск Colab (и докачка недостающих)
SYNC_FROM_DRIVE = True #@param {type:"boolean"}
DOWNLOAD_IF_MISSING = True #@param {type:"boolean"}
DEPARTMENT_FILTER = "ALL (Все отделы)" #@param ["ALL (Все отделы)", "resident", "streams", "tables_graphs", "vision_docs", "bio_science", "voice_guard"]

import os, sys, time, shutil, subprocess

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import HfApi, hf_hub_download

DRIVE_MODELS = "/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE/models"
FAST_LOCAL_MODELS = "/content/AMBER_SOCIETY_CAPSULE/models"
os.makedirs(FAST_LOCAL_MODELS, exist_ok=True)

# 1. Проверяем Google Диск и копируем уже скачанные модели на быстрый SSD Colab
copied_count = 0
t_start = time.time()

if SYNC_FROM_DRIVE:
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount('/content/drive')

    if os.path.exists(DRIVE_MODELS):
        print(f"[ДИСК НАЙДЕН] Синхронизируем модели из {DRIVE_MODELS} на быстрый SSD Colab...")
        for dept in os.listdir(DRIVE_MODELS):
            src_dept = os.path.join(DRIVE_MODELS, dept)
            if not os.path.isdir(src_dept) or dept == "executive_7b":
                continue
            if DEPARTMENT_FILTER != "ALL (Все отделы)" and dept != DEPARTMENT_FILTER:
                continue

            dst_dept = os.path.join(FAST_LOCAL_MODELS, dept)
            os.makedirs(dst_dept, exist_ok=True)
            for model_folder in os.listdir(src_dept):
                src_m = os.path.join(src_dept, model_folder)
                dst_m = os.path.join(dst_dept, model_folder)
                if os.path.isdir(src_m):
                    shutil.copytree(src_m, dst_m, dirs_exist_ok=True)
                    copied_count += 1
                    print(f"  [SSD <- DRIVE] {dept}/{model_folder}")
    else:
        print("[ИНФО] Папка коллектива на Google Диске не найдена — переходим к прямой загрузке на SSD Colab.")

# 2. Если на Диске чего-то не было — докачиваем сразу на быстрый локальный диск Colab
CORE_COLAB_ROSTER = [
    ("resident",      "Liquid-LFM2-350M",       "LiquidAI/LFM2-350M-GGUF",                  "gguf"),
    ("resident",      "Qwen2.5-0.5B-Instruct",  "Qwen/Qwen2.5-0.5B-Instruct-GGUF",          "gguf"),
    ("resident",      "Gemma-3-270M-IT",        "ggml-org/gemma-3-270m-it-GGUF",            "gguf"),
    ("resident",      "SetFit-MiniLM-Router",   "sentence-transformers/all-MiniLM-L6-v2",   "onnx"),
    ("streams",       "Chronos-Bolt-Tiny",      "amazon/chronos-bolt-tiny",                 "native"),
    ("streams",       "MOMENT-1-Small",         "AutonLab/MOMENT-1-small",                  "native"),
    ("tables_graphs", "Qwen2.5-Coder-0.5B",     "Qwen/Qwen2.5-Coder-0.5B-Instruct-GGUF",    "gguf"),
    ("tables_graphs", "TabPFN-v2-Tabular",      "Prior-Labs/TabPFN-v2-clf",                 "native"),
    ("vision_docs",   "YOLOv11-Nano",           "Ultralytics/YOLO11",                       "native"),
    ("vision_docs",   "Depth-Anything-V2-Small","depth-anything/Depth-Anything-V2-Small-hf","native"),
    ("bio_science",   "Caduceus-PS-DNA",        "kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16", "native"),
    ("bio_science",   "HyenaDNA-Medium-160k",   "LongSafari/hyenadna-medium-160k-seqlen-hf",  "native"),
    ("voice_guard",   "Nomic-Embed-Q4",         "nomic-ai/nomic-embed-text-v1.5-GGUF",      "gguf"),
]

downloaded_count = 0
if DOWNLOAD_IF_MISSING:
    api = HfApi()
    MAX_BYTES = 460 * 1024 * 1024
    for dept, name, repo_id, fmt in CORE_COLAB_ROSTER:
        if DEPARTMENT_FILTER != "ALL (Все отделы)" and dept != DEPARTMENT_FILTER:
            continue
        target_m = os.path.join(FAST_LOCAL_MODELS, dept, name)
        has_weights = os.path.exists(target_m) and any(
            f.endswith((".gguf", ".onnx", ".safetensors", ".bin", ".pt", ".ckpt"))
            for _, _, files in os.walk(target_m) for f in files
        )
        if has_weights:
            continue

        os.makedirs(target_m, exist_ok=True)
        try:
            info = api.model_info(repo_id=repo_id, files_metadata=True)
            siblings = info.siblings or list()
            chosen = None
            for ext in (".gguf", ".onnx", ".safetensors", ".pt", ".ckpt"):
                candidates = [f for f in siblings if f.rfilename.endswith(ext) and (f.size or 0) <= MAX_BYTES and "arm64" not in f.rfilename]
                if candidates:
                    candidates.sort(key=lambda x: x.size or 0)
                    chosen = next(iter(candidates))
                    break
            if chosen:
                hf_hub_download(repo_id=repo_id, filename=chosen.rfilename, local_dir=target_m)
                for cfg in ("config.json", "tokenizer.json", "tokenizer_config.json"):
                    try:
                        hf_hub_download(repo_id=repo_id, filename=cfg, local_dir=target_m)
                    except Exception:
                        pass
                downloaded_count += 1
                print(f"  [SSD <- HUB]   {dept}/{name} ({round((chosen.size or 0)/(1024*1024), 1)} МБ)")
        except Exception as e:
            print(f"  [SKIP] {name}: {e}")

# 3. Переключаем общий путь коллектива на быстрый локальный диск Colab
MODELS_DIR = FAST_LOCAL_MODELS
elapsed = round(time.time() - t_start, 2)
total_ready = sum(
    1 for _, _, files in os.walk(FAST_LOCAL_MODELS)
    for f in files if f.endswith((".gguf", ".onnx", ".safetensors", ".bin", ".pt", ".ckpt"))
)

print(f"\n=== ГОТОВО ЗА {elapsed} СЕК! ===")
print(f"Скопировано с Google Диска: {copied_count} | Докачано напрямую: {downloaded_count}")
print(f"Всего моделей на быстром диске Colab ({MODELS_DIR}): {total_ready}")


Вот решение ваших трёх задач в одном цельном блоке для Colab:
1. Поочерёдный «Медосмотр и Очистка ОЗУ» (Step 1): проверяет каждую скачанную модель на быстрый отклик (английский технический тест), замеряет время пробуждения и жестко выгружает её из памяти (del, gc.collect(), malloc_trim), освобождая весь CPU и ОЗУ под Главного Директора.
2. Трёхъязычный шлюз-переводчик (RU ↔ EN ↔ ZH): ваш уже скачанный Qwen2.5-0.5B-Instruct (408 МБ, родом из Alibaba, идеально знающий и русский, и английский, и китайский) назначается на должность «Посла-переводчика». Он переводит ваши русские команды на английский (для LFM2, NuExtract, Gemma, Caduceus) и на китайский (для глубинного мышления DeepSeek-R1-7B, поскольку его рассуждения <think> на родном китайском точнее и короче), а финальный ответ возвращает вам на чистом русском языке.
3. Клонирование «Цифрового Двойника / Копии Штаба»: автоматическая сборка переносимого архива AMBER_CLONE_SNAPSHOT (веса, конфиги, обученные адаптеры и база знаний) на Google Диск для мгновенного клонирования на другой аккаунт или ваш Linux-ноутбук.
Ниже — готовая Ячейка №5 (Медосмотр, Очистка ОЗУ, Трёхъязычный шлюз и Создание Копии) с панелью настроек справа до запуска.

In [ ]:
#@title 5. Медосмотр штата, Очистка ОЗУ, Мост RU-EN-ZH и Клонирование капсулы
RUN_HEALTH_CHECK = True #@param {type:"boolean"}
FREE_ALL_RAM_AFTER = True #@param {type:"boolean"}
TEST_TRILINGUAL_BRIDGE = True #@param {type:"boolean"}
CREATE_PORTABLE_CLONE = False #@param {type:"boolean"}
SAMPLE_RU_COMMAND = "Найди аномалии влажности почвы за 3 дня и сопоставь с геномом засухоустойчивости." #@param {type:"string"}

import os, gc, glob, time, json, ctypes, shutil, psutil
from llama_cpp import Llama

MODELS_DIR = globals().get("MODELS_DIR", "/content/AMBER_SOCIETY_CAPSULE/models")
CAPSULE_DIR = globals().get("CAPSULE_DIR", "/content/AMBER_SOCIETY_CAPSULE")

def hard_free_memory():
    """Принудительная очистка памяти Python и возврат страниц ОЗУ ядру Linux"""
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

def get_ram_mb():
    return round(psutil.Process().memory_info().rss / (1024 ** 2), 1)

def extract_reply(res: dict) -> str:
    choices = res.get("choices", list())
    first = next(iter(choices), dict())
    return first.get("message", dict()).get("content", "").strip()

# ==============================================================================
# 1. ПООЧЕРЕДНЫЙ МЕДОСМОТР ВСЕХ МОДЕЛЕЙ (С МГНОВЕННОЙ ВЫГРУЗКОЙ ИЗ ПАМЯТИ)
# ==============================================================================
if RUN_HEALTH_CHECK:
    print(f"=== СТАРТ МЕДОСМОТРА ШТАТА (ОЗУ до теста: {get_ram_mb()} МБ) ===\n", flush=True)
    # Если Директор сейчас в памяти — временно усыпляем, чтобы освободить 100% CPU под проверку
    if "director_llm" in globals():
        del globals()["director_llm"]
        hard_free_memory()

    gguf_files = [f for f in glob.glob(f"{MODELS_DIR}/**/*.gguf", recursive=True) if "7B" not in f and "embed" not in f.lower()]
    report = list()

    for path in sorted(gguf_files):
        m_name = os.path.basename(os.path.dirname(path))
        size_mb = round(os.path.getsize(path) / (1024 ** 2), 1)
        t0 = time.time()
        try:
            # Грузим строго 1 модель, спрашиваем на английском (родной язык микромоделей) и сразу выгружаем
            m = Llama(model_path=path, n_ctx=512, n_threads=2, verbose=False)
            t_load = round(time.time() - t0, 2)

            t1 = time.time()
            res = m.create_chat_completion(
                messages=[{"role": "user", "content": "Reply with one word: READY"}],
                max_tokens=10, temperature=0.0
            )
            t_gen = round(time.time() - t1, 2)
            ans = extract_reply(res).replace("\n", " ")[:30]

            del m
            hard_free_memory()
            print(f"[OK]  {m_name:24s} | {size_mb:5.1f} МБ | Старт: {t_load}с | Отклик: {t_gen}с -> '{ans}'", flush=True)
            report.append({"model": m_name, "status": "OK", "load_s": t_load, "reply": ans})
        except Exception as e:
            hard_free_memory()
            print(f"[ERR] {m_name:24s} | Ошибка: {e}", flush=True)

    if FREE_ALL_RAM_AFTER:
        hard_free_memory()
        print(f"\n[ОЗУ ОЧИЩЕНА] Все модели выгружены на диск. Текущая память процесса: {get_ram_mb()} МБ\n", flush=True)

# ==============================================================================
# 2. ТРЕХЪЯЗЫЧНЫЙ ШЛЮЗ-ПЕРЕВОДЧИК (RU -> EN для сотрудников / RU -> ZH для DeepSeek)
# ==============================================================================
def translate_command_pack(ru_text: str) -> dict:
    """Использует легкий Qwen2.5-0.5B (408 МБ) как синхронного переводчика RU -> EN + ZH"""
    qwen_files = glob.glob(f"{MODELS_DIR}/resident/Qwen2.5-0.5B-Instruct/*.gguf")
    q_path = next(iter(qwen_files), None)
    if not q_path:
        return {"ru": ru_text, "en": ru_text, "zh": ru_text, "note": "Qwen-0.5B не найден"}

    translator = Llama(model_path=q_path, n_ctx=1024, n_threads=2, verbose=False)
    prompt = (
        f"Translate the Russian technical command into English and Chinese.\n"
        f"Return ONLY valid JSON with keys 'en' and 'zh'.\n"
        f"Russian: {ru_text}"
    )
    res = translator.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=180, temperature=0.0
    )
    raw = extract_reply(res)
    del translator
    hard_free_memory()

    try:
        clean = raw[raw.find("{"):raw.rfind("}") + 1]
        parsed = json.loads(clean)
        return {"ru": ru_text, "en": parsed.get("en", ru_text), "zh": parsed.get("zh", ru_text)}
    except Exception:
        return {"ru": ru_text, "en": raw, "zh": ru_text}

if TEST_TRILINGUAL_BRIDGE:
    print("=== ТЕСТ ТРЕХЪЯЗЫЧНОГО ШЛЮЗА (RU -> EN + ZH) ===", flush=True)
    pack = translate_command_pack(SAMPLE_RU_COMMAND)
    print(f"  [RU Пользователь]: {pack['ru']}")
    print(f"  [EN Сотрудникам]:  {pack['en']}")
    print(f"  [ZH Дипсику-R1]:   {pack['zh']}\n", flush=True)

# ==============================================================================
# 3. СОЗДАНИЕ ПЕРЕНОСИМОЙ КОПИИ КАПСУЛЫ (КЛОНИРОВАНИЕ СИСТЕМЫ)
# ==============================================================================
if CREATE_PORTABLE_CLONE:
    drive_clone_dir = "/content/drive/MyDrive/AMBER_CLONE_BACKUP"
    os.makedirs(drive_clone_dir, exist_ok=True)
    clone_name = os.path.join(drive_clone_dir, f"amber_society_clone_{int(time.time())}")
    print(f"=== СОЗДАНИЕ КОПИИ СИСТЕМЫ В {clone_name}.zip ===", flush=True)
    shutil.make_archive(clone_name, "zip", CAPSULE_DIR)
    print(f"[КЛОН ГОТОВ] Архив со всеми моделями и базой сохранен: {clone_name}.zip", flush=True)


ОСМОТР ПРОБА С ОСНОВНЫМ опрос основных текстовых моделей после установки

In [ ]:
import os, sys, gc, glob, time, ctypes, psutil, subprocess

try:
    import onnxruntime as ort
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    import onnxruntime as ort

from llama_cpp import Llama

def clean_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

print(f"=== ПОЛНЫЙ МЕДОСМОТР: ONNX, BIO/VISION/STREAMS И ДИРЕКТОР 7B (ОЗУ старт: {round(psutil.Process().memory_info().rss/(1024**2), 1)} МБ) ===\n", flush=True)

# 1. Проверка всех ONNX-моделей (SetFit, ColModernVBERT, Piper-TTS, ModernBERT)
for path in sorted(glob.glob(f"{MODELS_DIR}/**/*.onnx", recursive=True)):
    name = os.path.basename(os.path.dirname(path))
    size_mb = round(os.path.getsize(path) / (1024**2), 1)
    t0 = time.time()
    try:
        opts = ort.SessionOptions()
        opts.enable_cpu_mem_arena = False
        sess = ort.InferenceSession(path, opts, providers=["CPUExecutionProvider"])
        t_load = round(time.time() - t0, 2)
        del sess
        clean_ram()
        print(f"[ONNX OK] {name:24s} | {size_mb:5.1f} МБ | Старт: {t_load}с", flush=True)
    except Exception as e:
        clean_ram()
        print(f"[ONNX ERR] {name:24s} | {e}", flush=True)

# 2. Проверка веса и целостности файлов ДНК, Зрения и Датчиков (safetensors / pt / ckpt)
for ext in ("*.safetensors", "*.pt", "*.ckpt"):
    for path in sorted(glob.glob(f"{MODELS_DIR}/**/{ext}", recursive=True)):
        name = os.path.basename(os.path.dirname(path))
        size_mb = round(os.path.getsize(path) / (1024**2), 1)
        print(f"[FILE OK] {name:24s} | {size_mb:5.1f} МБ | Готов к вызову по профилю", flush=True)

# 3. Тест Главного Директора (DeepSeek-R1-Distill-Qwen-7B)
ds_files = glob.glob(f"{MODELS_DIR}/executive_7b/*DeepSeek*.gguf")
if ds_files:
    t0 = time.time()
    boss = Llama(model_path=next(iter(ds_files)), n_ctx=2048, n_threads=2, verbose=False)
    t_load = round(time.time() - t0, 2)
    t1 = time.time()
    res = boss.create_chat_completion(
        messages=[{"role": "user", "content": "Ответь одной строкой на русском: Готов ли ты руководить коллективом из 28 моделей?"}],
        max_tokens=80, temperature=0.1
    )
    t_gen = round(time.time() - t1, 2)
    reply = next(iter(res.get("choices", list())), dict()).get("message", dict()).get("content", "").strip()
    del boss
    clean_ram()
    print(f"\n[ДИРЕКТОР 7B OK] Старт: {t_load}с | Отклик: {t_gen}с | Ответ: {reply}", flush=True)

print(f"\n[ИТОГ] Вся память снова освобождена: {round(psutil.Process().memory_info().rss/(1024**2), 1)} МБ", flush=True)


*ОСНОВНОЙ   подключение основного в общее пространство *


---


*   Новый пункт
*   Новый пункт


1.   Когда откроете интерфейс OpenObserve через туннель (!ssh -o StrictHostKeyChecking=no -p 443 -R0:127.0.0.1:5080 a.pinggy.io), в разделе AI Observability → Providers → Add Provider укажите новый свободный порт 8088:
• Base URL: http://127.0.0
• Model Name: amber-director-deepseek-7b
• API Key: amber-local-key
Запускайте эту объединённую ячейку — она сама проверит и поднимет OpenObserve (:5080), загрузит Директора 7B, откроет шлюз :8088 и напечатает его рапорт.
2.   Новый пункт






In [ ]:
Запускаем ядро OpenObserve (:5080)...
[1/4] Статус OpenObserve (:5080): ONLINE
Загружаем Главного Директора (DeepSeek-R1-7B) в память (~35 сек)...
[2/4] Главный Директор DeepSeek-R1-7B загружен в ОЗУ!
[3/4] Шлюз управляющей модели открыт: http://127.0.0

[4/4] Директор формирует официальный рапорт...

=== ОФИЦИАЛЬНЫЙ ОТЧЁТ ГЛАВНОГО УПРАВЛЯЮЩЕГО ===

**Технический отчёт для Главного Управляющего OpenObserve**

1. **Подтвердение вступления в должность и готовности к использованию API-шлюза:**
   У меня уже назначен статус Главного Управляющего OpenObserve. API-шлюз на порту 8088 работает корректно и подключён к системе.

2. **Первоначальные правила координации 28 микромоделей и OpenObserve:**
   - **Распределение нагрузки:** Все микромодели обрабатывают данные параллельно для повышения эффективности.
   - **Синхронизация данных:** Все потоки и микромодели синхронизуются для устойчивого функционирования системы.
   - **Обработка ошибок:** Всегда приоритетное обработывание ошибок для минимизации их влияния на систему.

3. **Приветствие и предложение начать работу:**
   Привет! Я Главный Управляющий OpenObserve. Для начала работы у меня есть доступ к данным и функционалам. Начнём совместно с вами?


ОСНОВНОЙ подключение с обычным запуском оболочки

In [ ]:
import os, sys, gc, glob, json, time, ctypes, threading, subprocess, requests
from fastapi import FastAPI, Request
import uvicorn
from llama_cpp import Llama

CAPSULE_DIR = "/content/AMBER_SOCIETY_CAPSULE"
MODELS_DIR = os.path.join(CAPSULE_DIR, "models")
EXEC_DIR = os.path.join(MODELS_DIR, "executive_7b")
O2_DIR = os.path.join(CAPSULE_DIR, "openobserve_node")
O2_URL = "http://127.0.0.1:5080"
AUTH = ("admin@society.local", "ChangeMe123!")

# 1. Запускаем OpenObserve (:5080) и ждем его реальной готовности
try:
    o2_ready = requests.get(f"{O2_URL}/healthz", timeout=2).status_code == 200
except Exception:
    o2_ready = False

if not o2_ready:
    o2_bin = os.path.join(O2_DIR, "openobserve")
    if not os.path.exists(o2_bin):
        raise FileNotFoundError(f"Бинарник OpenObserve не найден в {o2_bin}. Выполните Ячейку 1.")
    print("Запускаем ядро OpenObserve (:5080)...", flush=True)
    env = os.environ.copy()
    env.update({
        "ZO_DATA_DIR": os.path.join(O2_DIR, "data"),
        "ZO_HTTP_PORT": "5080",
        "ZO_ROOT_USER_EMAIL": "admin@society.local",
        "ZO_ROOT_USER_PASSWORD": "ChangeMe123!",
        "ZO_MEMORY_CACHE_ENABLED": "false",
        "ZO_SSRF_ALLOW_LOOPBACK": "true",
        "RUST_LOG": "error"
    })
    subprocess.Popen([o2_bin], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(15):
        try:
            if requests.get(f"{O2_URL}/healthz", timeout=1).status_code == 200:
                o2_ready = True
                break
        except Exception:
            time.sleep(1)

print(f"[1/4] Статус OpenObserve (:5080): {'ONLINE' if o2_ready else 'ОШИБКА СТАРТА'}", flush=True)

# 2. Загружаем Главного Директора (DeepSeek-R1-7B) в память, если его там нет
if "director_llm" not in globals() or director_llm is None:
    ds_files = glob.glob(os.path.join(EXEC_DIR, "*DeepSeek*.gguf"))
    if not ds_files:
        # Ищем на Google Диске, если в локальной папке Colab его нет
        ds_files = glob.glob("/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE/models/executive_7b/*DeepSeek*.gguf")
    if not ds_files:
        raise FileNotFoundError("Файл DeepSeek-R1-7B не найден ни в Colab, ни на Диске.")
    print("Загружаем Главного Директора (DeepSeek-R1-7B) в память (~35 сек)...", flush=True)
    gc.collect()
    director_llm = Llama(model_path=next(iter(ds_files)), n_ctx=6144, n_threads=2, verbose=False)

print("[2/4] Главный Директор DeepSeek-R1-7B загружен в ОЗУ!", flush=True)

# 3. Запускаем OpenAI-совместимый шлюз на свободном порту 8088 (порт 8080 занят самим Colab)
api_app = FastAPI()

@api_app.get("/v1/models")
async def list_v1_models():
    return {"object": "list", "data": [{"id": "amber-director-deepseek-7b", "object": "model"}]}

@api_app.post("/v1/chat/completions")
async def chat_v1_completions(req: Request):
    body = await req.json()
    return director_llm.create_chat_completion(
        messages=body.get("messages", list()),
        max_tokens=body.get("max_tokens", 512),
        temperature=body.get("temperature", 0.15)
    )

if "gateway_8088_running" not in globals():
    threading.Thread(
        target=lambda: uvicorn.run(api_app, host="127.0.0.1", port=8088, log_level="error"),
        daemon=True
    ).start()
    gateway_8088_running = True
    time.sleep(1.5)

print("[3/4] Шлюз управляющей модели открыт: http://127.0.0", flush=True)

# 4. Инициализируем потоки OpenObserve и запрашиваем официальный отчёт Директора
for stream in ("society_events", "society_roster", "ai_consensus"):
    requests.post(f"{O2_URL}/api/default/{stream}/_json", auth=AUTH, json=[{"status": "init", "source": "director_boot"}])

director_task = """
Ты назначен Главным Управляющим оболочки OpenObserve (адрес http://127.0.0.1:5080, твой API-шлюз http://127.0.0, MCP-сервер http://127.0.0).
В твоём распоряжении потоки society_events, society_roster, ai_consensus и штат из 28 микромоделей.
Составь краткий, чёткий технический отчёт (до 12 строк) на русском языке:
1. Подтверди вступление в должность управляющего и готовность шлюза :8088.
2. Назови 3 первоочередных правила координации 28 микромоделей и потоков OpenObserve.
3. Поприветствуй пользователя и предложи начать работу.
"""

print("\n[4/4] Директор формирует официальный рапорт...\n", flush=True)
res = director_llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Ты Главный Управляющий OpenObserve (DeepSeek-R1-7B). Отвечай чётко по делу."},
        {"role": "user", "content": director_task}
    ],
    max_tokens=750,
    temperature=0.15
)

choices = res.get("choices", list())
raw_out = next(iter(choices), dict()).get("message", dict()).get("content", "").strip()
clean_report = raw_out.split("</think>")[-1].strip() if "</think>" in raw_out else raw_out

requests.post(
    f"{O2_URL}/api/default/ai_consensus/_json",
    auth=AUTH,
    json=[{"role": "Supreme_Director", "gateway": "http://127.0.0", "report": clean_report}]
)

print("=== ОФИЦИАЛЬНЫЙ ОТЧЁТ ГЛАВНОГО УПРАВЛЯЮЩЕГО ===\n")
print(clean_report)


ОБЩАЯ СБОРНАЯ СОСТОЯНИЙ ОБЩЕСТВА ( СБОР ПОХОЖЕГО


---
1. Панель управления теперь стоит в самом начале (#@param) — еще до запуска ячейки вы выбираете диск (Google Drive или Colab Local), вводите тему, источник (например, github:openobserve, arxiv:small language models или прямую ссылку https://...) и двигаете ползунок лимита.
2. В самом низу добавлен явный запуск (await run_task(...)) — ячейка сама подхватит выбранные вами параметры из формы, соберет данные, прогонит сайты через SmolLM2-360M в JSON, сложит всё в DuckDB и покажет сводку.



In [ ]:
#@title Панель управления сбором данных (Настройте до запуска ячейки)
STORAGE_MODE = "Google Drive (/content/drive/MyDrive/project_data.duckdb)" #@param ["Google Drive (/content/drive/MyDrive/project_data.duckdb)", "Colab Local (/content/project_data.duckdb)"]
TOPIC_NAME = "openobserve_ai_agents" #@param {type:"string"}
SOURCE_QUERY = "github:openobserve" #@param {type:"string"}
RESULTS_LIMIT = 3 #@param {type:"slider", min:1, max:10, step:1}
RUN_IMMEDIATELY = True #@param {type:"boolean"}

# ---------- 0. Зависимости ----------
import os, sys, json, asyncio, re, traceback
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from typing import Any

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
get_ipython().system('pip install -q crawl4ai playwright PyGithub duckdb transformers accelerate peft pydantic hf_transfer nest_asyncio')
get_ipython().system('ldconfig -p | grep -q libatk-1.0.so || playwright install --with-deps chromium')

import nest_asyncio
nest_asyncio.apply()

import torch
import duckdb
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from github import Github

# ---------- 1. Выбор диска и подключение DuckDB ----------
if "Google Drive" in STORAGE_MODE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount('/content/drive')
    DB_PATH = "/content/drive/MyDrive/project_data.duckdb"
else:
    DB_PATH = "/content/project_data.duckdb"

print(f"[БАЗА ДАННЫХ]: {DB_PATH}")
con = duckdb.connect(DB_PATH)
con.execute("""
CREATE TABLE IF NOT EXISTS web_pages (
    url VARCHAR, title VARCHAR, extracted_json VARCHAR,
    scraped_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute("""
CREATE TABLE IF NOT EXISTS github_repos (
    full_name VARCHAR, description VARCHAR, stars INTEGER,
    language VARCHAR, collected_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute("""
CREATE TABLE IF NOT EXISTS arxiv_papers (
    arxiv_id VARCHAR, title VARCHAR, authors VARCHAR,
    published VARCHAR, abstract VARCHAR,
    collected_at TIMESTAMP DEFAULT current_timestamp
);
""")

# ---------- 2. Модель SmolLM2-360M + JSON-адаптер ----------
base_id = "HuggingFaceTB/SmolLM2-360M"
adapter_id = "CycleCoreTechnologies/Maaza-SLM-360M-JSON-v1.2"
tokenizer = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.float32, device_map="cpu"
)
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()

def extract_json(text: str, instruction: str) -> str:
    prompt = f"{instruction}\n\nInput: {text[:3000]}\n\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    raw = tokenizer.decode(out[0], skip_special_tokens=True)
    return raw.split("Output:")[-1].strip()

# ---------- 3. Скрапинг / GitHub / arXiv ----------
async def scrape_site(url: str):
    browser_cfg = BrowserConfig(headless=True, verbose=False)
    run_cfg = CrawlerRunConfig(cache_mode=CacheMode.BYPASS)
    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        result = await crawler.arun(url=url, config=run_cfg)
        if not result.success:
            print(f"[FAIL] {url}: {result.error_message}")
            return None
        return {
            "markdown": result.markdown,
            "title": getattr(result, "metadata", dict()).get("title", ""),
        }

g = Github()

def fetch_github(query: str, limit: int = 3):
    repos = g.search_repositories(query=query)[:limit]
    return [{
        "full_name": r.full_name,
        "description": r.description or "",
        "stars": r.stargazers_count,
        "language": r.language or "",
    } for r in repos]

def fetch_arxiv(query: str, max_results: int = 3):
    arxiv_url = "http" + "://export.arxiv.org/api/query?" + f"search_query={urllib.parse.quote(query)}&max_results={max_results}"
    with urllib.request.urlopen(arxiv_url, timeout=30) as resp:
        xml_data = resp.read().decode("utf-8")
    root = ET.fromstring(xml_data)
    ns = {"a": "http" + "://www.w3.org/2005/Atom"}
    rows = list()
    for entry in root.findall("a:entry", ns):
        rows.append({
            "arxiv_id": entry.find("a:id", ns).text.split("/abs/")[-1],
            "title": entry.find("a:title", ns).text.strip().replace("\n", " "),
            "authors": ", ".join(a.find("a:name", ns).text for a in entry.findall("a:author", ns)),
            "published": entry.find("a:published", ns).text[:10],
            "abstract": entry.find("a:summary", ns).text.strip()[:300],
        })
    return rows

def _parse_source(src: str) -> dict:
    src = src.strip()
    low = src.lower()
    if low.startswith(("http://", "https://")):
        return {"kind": "site", "url": src}
    if low.startswith("site:"):
        return {"kind": "site", "url": src[5:].strip()}
    if low.startswith("github:"):
        return {"kind": "github", "query": src[7:].strip()}
    if low.startswith("arxiv:"):
        return {"kind": "arxiv", "query": src[6:].strip()}
    return {"kind": "github", "query": src}

def _safe_json_or_none(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.dumps(json.loads(m.group(1)), ensure_ascii=False)
    except Exception:
        return None

# ---------- 4. Выполнение задачи и вывод результатов ----------
async def run_task(task: dict[str, Any]) -> dict[str, int]:
    topic = task.get("topic", "general")
    sources = task.get("sources", list())
    limit = int(task.get("limit", 3))
    stats = {"site_ok": 0, "site_fail": 0, "github": 0, "arxiv": 0}

    for raw in sources:
        s = _parse_source(raw)
        kind = s["kind"]

        if kind == "site":
            url = s["url"]
            page = await scrape_site(url)
            if not page:
                stats["site_fail"] += 1
                continue
            instr = f"Extract structured JSON about {topic} from the text. Use snake_case keys. Return only valid JSON."
            clean = _safe_json_or_none(extract_json(page["markdown"], instr))
            con.execute("INSERT INTO web_pages (url, title, extracted_json) VALUES (?, ?, ?)", (url, page["title"], clean))
            stats["site_ok"] += 1
            print(f"[OK] site {url} ({'valid-json' if clean else 'raw'})")

        elif kind == "github":
            repos = fetch_github(s["query"], limit=limit)
            for r in repos:
                con.execute("INSERT INTO github_repos VALUES (?, ?, ?, ?, current_timestamp)",
                            (r["full_name"], r["description"], r["stars"], r["language"]))
            stats["github"] += len(repos)
            print(f"[OK] github '{s['query']}': найдено {len(repos)}")

        elif kind == "arxiv":
            papers = fetch_arxiv(s["query"], max_results=limit)
            for p in papers:
                con.execute("INSERT INTO arxiv_papers VALUES (?, ?, ?, ?, ?, current_timestamp)",
                            (p["arxiv_id"], p["title"], p["authors"], p["published"], p["abstract"]))
            stats["arxiv"] += len(papers)
            print(f"[OK] arxiv '{s['query']}': найдено {len(papers)}")

    print(f"\n== Итог по задаче '{topic}': {stats}")
    return stats

def show_stats():
    print("\n--- Сводка в DuckDB ---")
    print("web_pages:", con.execute("SELECT COUNT(*) FROM web_pages").fetchone())
    print("github_repos:", con.execute("SELECT COUNT(*) FROM github_repos").fetchone())
    print("arxiv_papers:", con.execute("SELECT COUNT(*) FROM arxiv_papers").fetchone())

# ---------- 5. Автоматический запуск выбранной в форме задачи ----------
if RUN_IMMEDIATELY and SOURCE_QUERY.strip():
    sources_list = [x.strip() for x in SOURCE_QUERY.split(",") if x.strip()]
    await run_task({"topic": TOPIC_NAME, "sources": sources_list, "limit": RESULTS_LIMIT})
    show_stats()


*ПОЛНЫЙ ПОДБОР ПОЛЕЗНОГО  с настройками*


---




In [ ]:
# ============================================================
# ЕДИНАЯ ЯЧЕЙКА: инфраструктура + формы + умный ввод + запуск
# Запускать целиком. В конце — явный прогон pending-тем через await,
# поэтому часы Colab крутятся и лог заполняется построчно.
# ============================================================

# ---------- 0. Зависимости ----------
!pip install -q crawl4ai playwright PyGithub duckdb transformers accelerate peft pydantic hf_transfer nest_asyncio
!ldconfig -p | grep -q libatk-1.0.so || playwright install --with-deps chromium

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import json, asyncio, re, threading, traceback, sys
import urllib.parse, urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime
from typing import Any

import nest_asyncio
nest_asyncio.apply()

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- 1. Drive + DuckDB ----------
from google.colab import drive
drive.mount('/content/drive')

DB_PATH = "/content/drive/MyDrive/project_data.duckdb"
TASKS_JSON = globals().get("TASKS_JSON", "/content/drive/MyDrive/project_data.tasks.json")

import duckdb
con = duckdb.connect(DB_PATH)
con.execute("""
CREATE TABLE IF NOT EXISTS web_pages (
    url VARCHAR, title VARCHAR, extracted_json VARCHAR,
    scraped_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute("""
CREATE TABLE IF NOT EXISTS github_repos (
    full_name VARCHAR, description VARCHAR, stars INTEGER,
    language VARCHAR, collected_at TIMESTAMP DEFAULT current_timestamp
);
""")
con.execute("""
CREATE TABLE IF NOT EXISTS arxiv_papers (
    arxiv_id VARCHAR, title VARCHAR, authors VARCHAR,
    published VARCHAR, abstract VARCHAR,
    collected_at TIMESTAMP DEFAULT current_timestamp
);
""")

# ---------- 2. Модель ----------
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_id = "HuggingFaceTB/SmolLM2-360M"
adapter_id = "CycleCoreTechnologies/Maaza-SLM-360M-JSON-v1.2"
tokenizer = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.float32, device_map="cpu"
)
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()

def extract_json(text: str, instruction: str) -> str:
    prompt = f"{instruction}\n\nInput: {text[:3000]}\n\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    raw = tokenizer.decode(out[0], skip_special_tokens=True)
    return raw.split("Output:")[-1].strip()

# ---------- 3. Скрапинг ----------
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode

async def scrape_site(url: str):
    browser_cfg = BrowserConfig(headless=True, verbose=False)
    run_cfg = CrawlerRunConfig(cache_mode=CacheMode.BYPASS)
    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        result = await crawler.arun(url=url, config=run_cfg)
        if not result.success:
            print(f"[FAIL] {url}: {result.error_message}")
            return None
        return {
            "markdown": result.markdown,
            "title": getattr(result, "metadata", {}).get("title", ""),
        }

# ---------- 4. GitHub / arXiv ----------
from github import Github
g = Github()

def fetch_github(query: str, limit: int = 3):
    repos = g.search_repositories(query=query)[:limit]
    return [{
        "full_name": r.full_name,
        "description": r.description or "",
        "stars": r.stargazers_count,
        "language": r.language or "",
    } for r in repos]

def fetch_arxiv(query: str, max_results: int = 3):
    url = ("http://export.arxiv.org/api/query?"
           f"search_query={urllib.parse.quote(query)}&max_results={max_results}")
    with urllib.request.urlopen(url, timeout=30) as resp:
        xml_data = resp.read().decode("utf-8")
    root = ET.fromstring(xml_data)
    ns = {"a": "http://www.w3.org/2005/Atom"}
    rows = []
    for entry in root.findall("a:entry", ns):
        rows.append({
            "arxiv_id": entry.find("a:id", ns).text.split("/abs/")[-1],
            "title": entry.find("a:title", ns).text.strip().replace("\n", " "),
            "authors": ", ".join(a.find("a:name", ns).text for a in entry.findall("a:author", ns)),
            "published": entry.find("a:published", ns).text[:10],
            "abstract": entry.find("a:summary", ns).text.strip()[:300],
        })
    return rows

# ---------- 5. Разбор источника / JSON ----------
def _parse_source(src: str) -> dict:
    src = src.strip()
    low = src.lower()
    if low.startswith(("http://", "https://")):
        return {"kind": "site", "url": src}
    if low.startswith("site:"):
        return {"kind": "site", "url": src[5:].strip()}
    if low.startswith("github:"):
        return {"kind": "github", "query": src[7:].strip()}
    if low.startswith("arxiv:"):
        return {"kind": "arxiv", "query": src[6:].strip()}
    return {"kind": "github", "query": src}

def _instruction_for(kind: str, topic: str, custom: str | None = None) -> str:
    if custom:
        return custom
    if kind == "site":
        return (f"Extract structured JSON about {topic} from the text. "
                "Use snake_case keys. Return only valid JSON.")
    return ""

def _safe_json_or_none(text: str) -> str | None:
    if not text:
        return None
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(1))
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return None

# ---------- 6. run_task / show_stats ----------
async def run_task(task: dict[str, Any]) -> dict[str, int]:
    topic = task.get("topic", "general")
    sources = task.get("sources", []) or []
    limit = int(task.get("limit", 3))
    custom_instr = task.get("instruction")

    stats = {"site_ok": 0, "site_fail": 0, "github": 0, "arxiv": 0}

    for raw in sources:
        s = _parse_source(raw)
        kind = s["kind"]

        if kind == "site":
            url = s["url"]
            page = await scrape_site(url)
            if not page:
                stats["site_fail"] += 1
                print(f"[FAIL] {url}")
                continue
            instr = _instruction_for("site", topic, custom_instr)
            extracted = extract_json(page["markdown"], instr)
            clean = _safe_json_or_none(extracted)
            con.execute(
                "INSERT INTO web_pages (url, title, extracted_json) VALUES (?, ?, ?)",
                (url, page["title"], clean),
            )
            stats["site_ok"] += 1
            print(f"[OK] site {url} ({'ok' if clean else 'json-invalid'})")

        elif kind == "github":
            q = s["query"]
            repos = fetch_github(q, limit=limit)
            for r in repos:
                con.execute(
                    "INSERT INTO github_repos (full_name, description, stars, language) "
                    "VALUES (?, ?, ?, ?)",
                    (r["full_name"], r["description"], r["stars"], r["language"]),
                )
            stats["github"] += len(repos)
            print(f"[OK] github '{q}': {len(repos)}")

        elif kind == "arxiv":
            q = s["query"]
            papers = fetch_arxiv(q, max_results=limit)
            for p in papers:
                con.execute(
                    "INSERT INTO arxiv_papers (arxiv_id, title, authors, published, abstract) "
                    "VALUES (?, ?, ?, ?, ?)",
                    (p["arxiv_id"], p["title"], p["authors"], p["published"], p["abstract"]),
                )
            stats["arxiv"] += len(papers)
            print(f"[OK] arxiv '{q}': {len(papers)}")

    print(f"\n== Итог по задаче '{topic}': {stats}")
    return stats


def show_stats():
    print("--- web_pages ---")
    print(con.execute("SELECT url, COUNT(*) cnt FROM web_pages GROUP BY url ORDER BY cnt DESC").fetchdf())
    print("\n--- github_repos (top) ---")
    print(con.execute("SELECT full_name, stars, language FROM github_repos ORDER BY stars DESC LIMIT 10").fetchdf())
    print("\n--- arxiv_papers ---")
    print(con.execute("SELECT arxiv_id, title, published FROM arxiv_papers ORDER BY published DESC LIMIT 10").fetchdf())

# ============================================================
# ЧАСТЬ 2. ХРАНИЛИЩЕ
# ============================================================
def normalize_tasks(obj):
    if isinstance(obj, dict):
        items = obj.get("tasks")
        if items is None:
            items = [v for k, v in obj.items() if isinstance(k, int)]
        results = {r.get("topic"): r for r in (obj.get("results") or []) if isinstance(r, dict)}
    elif isinstance(obj, (list, tuple)):
        items, results = list(obj), {}
    else:
        items, results = [], {}

    clean = []
    for t in items:
        if isinstance(t, dict) and t.get("topic"):
            srcs = t.get("sources") or []
            if isinstance(srcs, str):
                srcs = [x.strip() for x in srcs.splitlines() if x.strip()]
            st = t.get("status")
            if not st:
                r = results.get(t["topic"]) or {}
                st = r.get("status") or "pending"
            try:
                lim = int(t.get("limit") or 3)
            except (TypeError, ValueError):
                lim = 3
            clean.append({
                "topic": str(t["topic"]).strip(),
                "sources": [str(s).strip() for s in srcs if str(s).strip()],
                "limit": max(1, min(30, lim)),
                "instruction": t.get("instruction"),
                "status": st if isinstance(st, str) else "pending",
            })
        elif isinstance(t, str) and t.strip():
            clean.append({"topic": t.strip(), "sources": [], "limit": 3,
                          "instruction": None, "status": "pending"})
    return clean


def load_from_disk(path):
    if not os.path.exists(path):
        return []
    return normalize_tasks(json.load(open(path, encoding="utf-8")))


def save_tasks(path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    payload = {
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "tasks": TASKS,
        "results": [{"topic": t.get("topic"), "status": t.get("status", "pending"),
                     "stats": t.get("stats")}
                    for t in TASKS if t.get("status", "pending") != "pending"],
    }
    json.dump(payload, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)


TASKS = normalize_tasks(globals().get("TASKS"))
if not TASKS:
    try:
        TASKS = load_from_disk(TASKS_JSON)
    except Exception as e:
        print("[WARN] файл заданий не прочитан:", e)

# ============================================================
# ЧАСТЬ 3. ВИДЖЕТЫ
# ============================================================
PRESETS = {
    "KAN": ("KAN robotics",
            ["github:KAN language:python", "arxiv:all:Kolmogorov-Arnold Networks"]),
    "Genesis": ("Genesis World robotics simulation",
                ["github:Genesis-Embodied-AI", "arxiv:all:differentiable physics robotics"]),
    "Octo": ("Octo / robot policies",
             ["github:octo-models/octo", "arxiv:all:generalist robot policy"]),
    "MuJoCo": ("MuJoCo simulation", ["github:google-deepmind/mujoco"]),
    "awesome-go": ("awesome-go federation of cities",
                   ["github:avelino/awesome-go", "github:libp2p/go-libp2p"]),
}

W = widgets.Layout(width="700px")
w_topic = widgets.Text(description="Тема:", layout=W, placeholder="например: KAN robotics")
w_src = widgets.Textarea(description="Откуда:", layout=widgets.Layout(width="700px", height="100px"),
                         placeholder="github:KAN language:python\narxiv:all:Kolmogorov-Arnold Networks\nhttps://cyberleninka.ru")
w_lim = widgets.IntSlider(value=3, min=1, max=30, description="Limit:", layout=W)
w_instr = widgets.Textarea(description="Инструкция:", layout=widgets.Layout(width="700px", height="50px"),
                           placeholder="необязательно: своя подсказка модели")

out_form = widgets.Output()
out_editor = widgets.Output()
out_smart = widgets.Output()
w_list = widgets.VBox([])

def log_form(*parts): out_form.append_stdout(" ".join(str(p) for p in parts) + "\n")
def log_edit(*parts): out_editor.append_stdout(" ".join(str(p) for p in parts) + "\n")
def log_s(*parts): out_smart.append_stdout(" ".join(str(p) for p in parts) + "\n")


def srcs_of(text):
    return [x.strip() for line in (text or "").splitlines()
            for x in line.split(",") if x.strip()]


def classify(s):
    low = s.lower()
    if low.startswith(("http://", "https://", "site:")):
        return "site", "скрапить страницу"
    if low.startswith("arxiv:"):
        return "arxiv", "статьи arXiv"
    return "github", "репозитории GitHub" + ("" if low.startswith("github:") else " (префикс не указан)")


# ---------- список тем с прокруткой ----------
w_topics_select = widgets.Select(
    options=[], rows=12,
    layout=widgets.Layout(width="700px", height="240px"),
    description="Темы:",
)


def refresh_topics_select():
    opts = []
    for i, t in enumerate(TASKS):
        st = str(t.get("status", "pending"))
        mark = "✓" if st == "done" else "✗" if st.startswith("error") else "▶" if st == "running" else "•"
        opts.append(("%s %d. %s" % (mark, i + 1, t.get("topic", "(без темы)")), i))
    w_topics_select.options = opts
    if opts and (w_topics_select.value is None or w_topics_select.value >= len(TASKS)):
        w_topics_select.value = opts[0][1]


# ---------- старый список с галочками ----------
def render():
    rows = []
    for i, t in enumerate(TASKS):
        cb = widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width="28px"))
        st = str(t.get("status", "pending"))
        col = ("#2e7d32" if st == "done" else "#c62828" if st.startswith("error")
               else "#1565c0" if st == "running" else "#666")
        html = ("<b>%d. %s</b> <span style='color:#666'>| limit=%s | "
                "<span style='color:%s'>%s</span><br>"
                "<span style='font-size:0.9em;color:#666'>%s</span>" % (
                    i + 1, t.get("topic", "(без темы)"), t.get("limit", 3), col, st,
                    ", ".join([str(x) for x in (t.get("sources") or [])[:3]])))
        row = widgets.HBox([cb, widgets.HTML(html)])
        row._cb, row._idx = cb, i
        rows.append(row)
    w_list.children = rows


def chosen():
    return [r._idx for r in w_list.children if r._cb.value]


# ============================================================
# ЧАСТЬ 4. ОБРАБОТЧИКИ ФОРМЫ
# ============================================================

def on_add_topic(_):
    topic = w_topic.value.strip()
    srcs = srcs_of(w_src.value)
    if not topic:
        log_form("[!] Заполни «Тема»."); return
    if not srcs:
        log_form("[!] Нужен хотя бы один источник в «Откуда»."); return
    if any(t.get("topic", "").strip().lower() == topic.lower() for t in TASKS):
        log_form("[!] Тема «%s» уже есть." % topic); return
    TASKS.append({"topic": topic, "sources": srcs, "limit": int(w_lim.value),
                  "instruction": w_instr.value.strip() or None, "status": "pending"})
    render(); refresh_topics_select()
    log_form("[+] Добавлена тема №%d: %s (источников %d)" % (len(TASKS), topic, len(srcs)))
    w_topic.value = ""; w_src.value = ""


def on_chk(_):
    srcs = srcs_of(w_src.value)
    log_form("Проверка (ничего не запускается)")
    log_form("ТЕМА:", w_topic.value.strip() or "(не заполнена)", "| LIMIT:", int(w_lim.value))
    if not srcs:
        log_form("[!] Источники не заполнены."); return
    for n, s in enumerate(srcs, 1):
        kind, why = classify(s)
        log_form("  %d. %s\n     -> %s: %s" % (n, s, kind, why))
    log_form("Всё распознано — можно нажимать «Добавить тему».")


def on_del(_):
    idxs = chosen()
    for i in sorted(idxs, reverse=True):
        del TASKS[i]
    render(); refresh_topics_select()
    log_form("[-] Удалено заданий:", len(idxs))


def on_save(_):
    try:
        save_tasks(TASKS_JSON)
        log_form("[OK] Сохранено:", TASKS_JSON, "| заданий:", len(TASKS))
    except Exception as e:
        log_form("[!] Не сохранилось:", e)


def on_stats(_):
    if callable(globals().get("show_stats")):
        show_stats()
    else:
        log_form("[!] show_stats() нет.")


def on_pick(_):
    idx = w_topics_select.value
    if idx is None or idx >= len(TASKS):
        log_form("[!] Тема не выбрана."); return
    t = TASKS[idx]
    w_topic.value = t.get("topic", "")
    w_src.value = "\n".join(t.get("sources") or [])
    w_instr.value = t.get("instruction") or ""
    try: w_lim.value = int(t.get("limit", 3))
    except Exception: w_lim.value = 3
    log_form("[→] Тема №%d загружена в форму: %s" % (idx + 1, t.get("topic", "")))


def on_update(_):
    idx = w_topics_select.value
    if idx is None or idx >= len(TASKS):
        log_form("[!] Тема не выбрана."); return
    topic = w_topic.value.strip()
    srcs = srcs_of(w_src.value)
    if not topic or not srcs:
        log_form("[!] Нужны «Тема» и хотя бы один источник."); return
    TASKS[idx].update({"topic": topic, "sources": srcs, "limit": int(w_lim.value),
                       "instruction": w_instr.value.strip() or None})
    render(); refresh_topics_select()
    w_topics_select.value = idx
    log_form("[✓] Тема №%d обновлена: %s" % (idx + 1, topic))


def on_remove2(_):
    idx = w_topics_select.value
    if idx is None or idx >= len(TASKS):
        log_form("[!] Тема не выбрана."); return
    name = TASKS[idx].get("topic", "")
    del TASKS[idx]
    render(); refresh_topics_select()
    log_form("[-] Удалена тема: %s" % name)


# ---------- редактор списка текстом ----------
w_editor = widgets.Textarea(
    layout=widgets.Layout(width="700px", height="200px"),
    placeholder=("Формат строки:\n"
                 "тема | откуда | инструкция(опц.)\n"
                 "или\n"
                 "откуда | тема\n"
                 "Несколько источников — через ';'. Строки с '#' игнорируются."),
)


def looks_like_source(s: str) -> bool:
    low = s.lower()
    return low.startswith(("http://", "https://", "site:", "github:", "arxiv:"))


def tasks_to_text():
    lines = []
    for t in TASKS:
        srcs = " ; ".join(t.get("sources") or [])
        instr = t.get("instruction") or ""
        line = "%s | %s" % (t.get("topic", ""), srcs)
        if instr:
            line += " | %s" % instr
        lines.append(line)
    return "\n".join(lines)


def parse_editor_text(text):
    tasks = []
    for raw in (text or "").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) < 2:
            continue
        a, b = parts[0], parts[1]
        instr = parts[2] if len(parts) > 2 and parts[2] else None
        if looks_like_source(a) and not looks_like_source(b):
            src_raw, topic = a, b
        else:
            topic, src_raw = a, b
        srcs = [x.strip() for x in src_raw.replace(",", ";").split(";") if x.strip()]
        if not topic or not srcs:
            continue
        tasks.append({"topic": topic, "sources": srcs, "limit": 3,
                      "instruction": instr, "status": "pending"})
    return tasks


def on_load_editor(_):
    w_editor.value = tasks_to_text()
    log_edit("[i] Загружено строк: %d" % len(TASKS))


def on_parse_replace(_):
    parsed = parse_editor_text(w_editor.value)
    if not parsed:
        log_edit("[!] Не разобрано ни одной строки."); return
    TASKS.clear(); TASKS.extend(parsed)
    render(); refresh_topics_select()
    log_edit("[✓] Список заменён: %d заданий" % len(TASKS))


def on_parse_append(_):
    parsed = parse_editor_text(w_editor.value)
    if not parsed:
        log_edit("[!] Не разобрано ни одной строки."); return
    TASKS.extend(parsed)
    render(); refresh_topics_select()
    log_edit("[+] Дополнено: +%d | всего: %d" % (len(parsed), len(TASKS)))


# ============================================================
# ЧАСТЬ 5. УМНЫЙ ПАРСЕР ТЕКСТА
# ============================================================
DEFAULT_SOURCES = ["github:{q}", "arxiv:all:{q}"]
STOPWORDS = {"и", "в", "на", "по", "для", "с", "к", "из", "о", "об", "от", "до",
             "или", "а", "но", "как", "что", "это", "the", "a", "an", "of", "for",
             "and", "or", "in", "on", "to", "with"}

def _keywords(topic: str, n: int = 6) -> str:
    words = re.findall(r"[A-Za-zА-Яа-я0-9_\-]+", topic or "")
    words = [w for w in words if w.lower() not in STOPWORDS and len(w) > 1]
    return " ".join(words[:n]) if words else (topic or "").strip()

def _default_sources_for(topic: str) -> list[str]:
    q = _keywords(topic)
    return [s.format(q=q) for s in DEFAULT_SOURCES] if q else []

_SRC_RE = re.compile(r"(https?://\S+|site:\S+|github:[^\s,;|]+|arxiv:[^\s,;|]+)", re.IGNORECASE)

def _extract_sources(line: str) -> list[str]:
    return [m.group(1).strip().rstrip(".,;") for m in _SRC_RE.finditer(line or "")]

def _clean_topic(s: str) -> str:
    s = re.sub(r"^[\s\-\*•\d\)\.]+", "", s or "").strip()
    s = re.sub(r"\s*[|;]\s*$", "", s).strip()
    return s


def smart_parse(text: str) -> list[dict]:
    tasks: list[dict] = []
    if not text:
        return tasks
    cur = None

    def flush():
        nonlocal cur
        if cur and cur.get("topic"):
            if not cur.get("sources"):
                cur["sources"] = _default_sources_for(cur["topic"])
            cur.setdefault("limit", 3)
            cur.setdefault("instruction", None)
            cur.setdefault("status", "pending")
            tasks.append(cur)
        cur = None

    for raw in text.splitlines():
        line = raw.rstrip()
        stripped = line.strip()
        if not stripped:
            continue
        low = stripped.lower()

        if low in ("---", "===", "***"):
            flush(); continue

        m = re.match(r"^(тема|откуда|источник|limit|лимит|инструкция)\s*:\s*(.*)$", low)
        if m:
            key = m.group(1)
            val = stripped.split(":", 1)[1].strip()
            if key == "тема":
                flush(); cur = {"topic": val, "sources": [], "limit": 3, "instruction": None, "status": "pending"}
            elif key in ("откуда", "источник"):
                if cur is None:
                    cur = {"topic": "", "sources": [], "limit": 3, "instruction": None, "status": "pending"}
                cur["sources"] += _extract_sources(val) or [x.strip() for x in re.split(r"[;,]", val) if x.strip()]
            elif key in ("limit", "лимит"):
                try:
                    n = int(val)
                    if cur is None:
                        cur = {"topic": "", "sources": [], "limit": 3, "instruction": None, "status": "pending"}
                    cur["limit"] = max(1, min(30, n))
                except ValueError:
                    pass
            elif key == "инструкция":
                if cur is None:
                    cur = {"topic": "", "sources": [], "limit": 3, "instruction": None, "status": "pending"}
                cur["instruction"] = val or None
            continue

        mh = re.match(r"^#{1,6}\s*(.+)$", stripped)
        if mh:
            flush()
            topic = _clean_topic(mh.group(1))
            if topic and len(topic) > 2:
                cur = {"topic": topic, "sources": [], "limit": 3, "instruction": None, "status": "pending"}
            continue

        if "|" in stripped:
            parts = [p.strip() for p in stripped.split("|")]
            if len(parts) >= 2:
                a, b = parts[0], parts[1]
                instr = parts[2] if len(parts) > 2 and parts[2] else None
                a_src, b_src = _extract_sources(a), _extract_sources(b)
                if a_src and not b_src:
                    topic, srcs = b, a_src
                else:
                    topic, srcs = a, b_src or [x.strip() for x in re.split(r"[;,]", b) if x.strip()]
                topic = _clean_topic(topic)
                if topic:
                    flush()
                    cur = {"topic": topic, "sources": srcs, "limit": 3, "instruction": instr, "status": "pending"}
                    flush()
                continue

        mb = re.match(r"^(\-|\*|•|\d+[\.\)])\s+(.+)$", stripped)
        if mb:
            body = mb.group(2).strip()
            srcs = _extract_sources(body)
            topic = _clean_topic(_SRC_RE.sub("", body).strip())
            if topic:
                flush()
                cur = {"topic": topic, "sources": srcs, "limit": 3, "instruction": None, "status": "pending"}
                flush()
            continue

        mt = re.match(r"^(?:тема|поиск|искать|найти|собрать|сбор)\s*[:\-—]\s*(.+)$", low)
        if mt:
            flush()
            topic = _clean_topic(stripped.split(":", 1)[-1]) if ":" in stripped else _clean_topic(mt.group(1))
            if topic:
                cur = {"topic": topic, "sources": _extract_sources(stripped), "limit": 3, "instruction": None, "status": "pending"}
            continue

        bare = _extract_sources(stripped)
        if bare and _SRC_RE.sub("", stripped).strip() == "":
            if cur is None:
                cur = {"topic": "", "sources": [], "limit": 3, "instruction": None, "status": "pending"}
            cur["sources"] += bare
            continue

        if len(stripped) > 3 and not stripped.endswith(('.', ':', ';')):
            flush()
            cur = {"topic": _clean_topic(stripped), "sources": [], "limit": 3, "instruction": None, "status": "pending"}
            continue

    flush()

    seen, out = set(), []
    for t in tasks:
        name = (t.get("topic") or "").strip()
        if not name:
            continue
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        if not t.get("sources"):
            t["sources"] = _default_sources_for(name)
        out.append(t)
    return out


w_smart = widgets.Textarea(
    layout=widgets.Layout(width="700px", height="200px"),
    placeholder=("Вставь ЛЮБОЙ подготовленный текст — я разберу темы сам.\n"
                 "Понимаю: блоки «тема:/откуда:/limit:», строки «тема | откуда», "
                 "маркированные списки, заголовки #, абзацы, голые URL/github:/arxiv:."),
)
b_smart_add = widgets.Button(description="✨ Разобрать и добавить", button_style="success")
b_smart_replace = widgets.Button(description="♻ Разобрать и заменить", button_style="warning")
b_smart_preview = widgets.Button(description="👁 Показать, что распознано", button_style="info")


def _render_smart_preview(parsed):
    if not parsed:
        log_s("[!] Ничего не распознано."); return
    log_s("[i] Распознано тем: %d" % len(parsed))
    for i, t in enumerate(parsed, 1):
        srcs = ", ".join((t.get("sources") or [])[:4])
        if len(t.get("sources") or []) > 4:
            srcs += " …"
        log_s("  %d. %s\n     limit=%s | источники: %s" % (i, t.get("topic"), t.get("limit", 3), srcs or "(нет — подставлю дефолт)"))


def on_smart_preview(_):
    _render_smart_preview(smart_parse(w_smart.value))


def on_smart_add(_):
    parsed = smart_parse(w_smart.value)
    if not parsed:
        log_s("[!] Ничего не распознано."); return
    existing = {t.get("topic", "").strip().lower() for t in TASKS}
    added = 0
    for t in parsed:
        if t["topic"].strip().lower() in existing:
            continue
        TASKS.append(t); existing.add(t["topic"].strip().lower()); added += 1
    render(); refresh_topics_select()
    log_s("[+] Добавлено тем: %d (пропущено дублей: %d) | всего: %d" % (added, len(parsed) - added, len(TASKS)))


def on_smart_replace(_):
    parsed = smart_parse(w_smart.value)
    if not parsed:
        log_s("[!] Ничего не распознано — список не изменён."); return
    TASKS.clear(); TASKS.extend(parsed)
    render(); refresh_topics_select()
    log_s("[♻] Список заменён: %d тем" % len(TASKS))


b_smart_add.on_click(on_smart_add)
b_smart_replace.on_click(on_smart_replace)
b_smart_preview.on_click(on_smart_preview)

# ============================================================
# ЧАСТЬ 6. КНОПКИ ФОРМЫ / ПАКЕТНЫЙ ВВОД
# ============================================================
b_add_topic = widgets.Button(description="➕ Добавить тему", button_style="success")
b_chk = widgets.Button(description="Проверить формат")
b_del = widgets.Button(description="Удалить отмеченные", button_style="warning")
b_save = widgets.Button(description="💾 Сохранить")
b_stats = widgets.Button(description="📊 Stats")
b_pick = widgets.Button(description="Загрузить в форму", button_style="info")
b_update = widgets.Button(description="Обновить выбранную", button_style="warning")
b_remove2 = widgets.Button(description="Удалить выбранную", button_style="danger")
b_load_editor = widgets.Button(description="Загрузить текущий список", button_style="info")
b_parse_replace = widgets.Button(description="Разобрать → заменить", button_style="warning")
b_parse_append = widgets.Button(description="Разобрать → дополнить", button_style="success")

for b, fn in [
    (b_add_topic, on_add_topic), (b_chk, on_chk), (b_del, on_del),
    (b_save, on_save), (b_stats, on_stats),
    (b_pick, on_pick), (b_update, on_update), (b_remove2, on_remove2),
    (b_load_editor, on_load_editor), (b_parse_replace, on_parse_replace), (b_parse_append, on_parse_append),
]:
    b.on_click(fn)

preset_buttons = []
for name in PRESETS:
    pb = widgets.Button(description=name, layout=widgets.Layout(width="110px"))
    pb.on_click((lambda n: (lambda _: (
        w_topic.__setattr__("value", PRESETS[n][0]),
        w_src.__setattr__("value", "\n".join(PRESETS[n][1])),
    )))(name))
    preset_buttons.append(pb)

w_batch = widgets.Textarea(
    layout=widgets.Layout(width="700px", height="130px"),
    placeholder=("тема: KAN robotics\n"
                 "откуда: github:KAN language:python\n"
                 "        arxiv:all:Kolmogorov-Arnold Networks\n"
                 "limit: 5\n---\nтема: Genesis\nоткуда: github:Genesis-Embodied-AI"),
)
b_batch = widgets.Button(description="Разобрать и добавить", button_style="success")


def parse_batch(text):
    blocks, cur, key = [], {}, None
    for raw in (text or "").splitlines():
        line = raw.rstrip()
        low = line.strip().lower()
        if low in ("---", "==="):
            if cur: blocks.append(cur)
            cur, key = {}, None; continue
        if low.startswith("тема:"):
            cur["topic"] = line.split(":", 1)[1].strip(); key = None
        elif low.startswith("откуда:"):
            cur["sources"] = srcs_of(line.split(":", 1)[1]); key = "sources"
        elif low.startswith("limit:"):
            try: cur["limit"] = int(line.split(":", 1)[1].strip())
            except ValueError: pass
            key = None
        elif low.startswith("инструкция:"):
            cur["instruction"] = line.split(":", 1)[1].strip() or None; key = None
        elif line.strip() and key == "sources":
            cur["sources"] += srcs_of(line)
    if cur: blocks.append(cur)
    return [b for b in blocks if b.get("topic") and b.get("sources")]


def on_batch(_):
    blocks = parse_batch(w_batch.value)
    if not blocks:
        log_form("[!] Не разобрано ни одного задания."); return
    for b in blocks:
        TASKS.append({"topic": b["topic"], "sources": b["sources"], "limit": int(b.get("limit", 3)),
                      "instruction": b.get("instruction"), "status": "pending"})
    render(); refresh_topics_select()
    log_form("[+] Добавлено заданий:", len(blocks), "| всего:", len(TASKS))
    w_batch.value = ""

b_batch.on_click(on_batch)

# ============================================================
# ЧАСТЬ 7. ОТРИСОВКА
# ============================================================
render()
refresh_topics_select()
ready = callable(globals().get("run_task"))
status = widgets.HTML(
    "<div style='background:%s;border-left:4px solid %s;padding:8px'>%s</div>" % (
        "#e8f5e9" if ready else "#fff3e0",
        "#2e7d32" if ready else "#ef6c00",
        "Готово: run_task() найден." if ready else "<b>Внимание:</b> run_task() не определён."))

clear_output(wait=True)
display(
    widgets.HTML("<h3>1. Задания: что искать и откуда</h3>"),
    status,
    widgets.HTML("<b>Быстрый старт:</b> пресет или заполни поля ниже."),
    widgets.HBox(preset_buttons),
    w_topic, w_src, w_lim, w_instr,
    widgets.HBox([b_add_topic, b_chk]),
    widgets.HTML("<hr><b>Пакетный ввод:</b> блоки «тема: … откуда: … limit: …» разделяются <b>---</b>"),
    w_batch, b_batch,
    widgets.HTML("<hr><b>Список заданий</b> (галочка — выбрать):"),
    w_list,
    widgets.HBox([b_del, b_save, b_stats]),

    widgets.HTML("<hr><h3>2. Список тем с прокруткой</h3>"),
    w_topics_select,
    widgets.HBox([b_pick, b_update, b_remove2]),

    widgets.HTML("<hr><h3>3. Редактор списка тем (текстом)</h3>"
                 "<span style='color:#666;font-size:0.9em'>"
                 "формат: <code>тема | откуда | инструкция(опц.)</code> или <code>откуда | тема</code>"
                 "</span>"),
    w_editor,
    widgets.HBox([b_load_editor, b_parse_replace, b_parse_append]),

    widgets.HTML("<hr><h3>4. Умный ввод: вставь готовый текст — я сам создам темы</h3>"),
    w_smart,
    widgets.HBox([b_smart_preview, b_smart_add, b_smart_replace]),

    widgets.HTML("<hr><b>Лог формы:</b>"), out_form,
    widgets.HTML("<hr><b>Лог редактора:</b>"), out_editor,
    widgets.HTML("<hr><b>Лог умного ввода:</b>"), out_smart,
)

# ============================================================
# ЧАСТЬ 8. ЗАПУСК: кнопки (исправлено для Colab)
# ============================================================

import asyncio, traceback, sys
import threading

out_run = widgets.Output()

def log_run(*parts):
    out_run.append_stdout(" ".join(str(p) for p in parts) + "\n")


# ---------- 8.1. Один прогон темы (как было) ----------
async def _run_one_visible(idx: int):
    if idx < 0 or idx >= len(TASKS):
        log_run("[!] Неверный индекс темы:", idx)
        return
    t = TASKS[idx]
    t["status"] = "running"
    try: render(); refresh_topics_select()
    except Exception: pass

    log_run("\n=== СТАРТ [%d/%d] %s ===" % (idx + 1, len(TASKS), t.get("topic")))
    log_run("    sources:", t.get("sources"))
    log_run("    limit:", t.get("limit"), "| instruction:", t.get("instruction") or "(нет)")

    if not callable(globals().get("run_task")):
        t["status"] = "error: no run_task"
        log_run("[!] run_task не определён.")
        try: render(); refresh_topics_select()
        except Exception: pass
        return

    try:
        log_run("    -> вызываю run_task() ...")
        stats = await run_task({
            "topic": t.get("topic"),
            "sources": t.get("sources") or [],
            "limit": int(t.get("limit", 3)),
            "instruction": t.get("instruction"),
        })
        t["status"], t["stats"] = "done", stats
        log_run("    <- run_task() вернул:", stats)
    except Exception as e:
        t["status"] = "error: " + type(e).__name__
        log_run("[ERROR] %s: %s" % (type(e).__name__, e))
        log_run(traceback.format_exc())
    finally:
        try: render(); refresh_topics_select()
        except Exception: pass


async def _run_many_visible(indices):
    if not indices:
        log_run("[!] Нет тем для запуска."); return
    log_run("[queue] тем в очереди: %d" % len(indices))
    for i in indices:
        await _run_one_visible(i)
    log_run("\n[OK] Очередь обработана.")


# ---------- 8.2. Синхронная обёртка для запуска корутины ----------
def _start_async(coro, label=""):
    """Запускает корутину в работающем loop ядра или в отдельном потоке."""
    try:
        loop = asyncio.get_event_loop()
    except Exception:
        loop = None

    if loop is not None and loop.is_running():
        # loop ядра работает — отправляем туда
        asyncio.ensure_future(coro, loop=loop)
        return "loop"

    # loop нет — запускаем в отдельном потоке
    def _runner():
        new_loop = asyncio.new_event_loop()
        asyncio.set_event_loop(new_loop)
        try:
            new_loop.run_until_complete(coro)
        except Exception as e:
            log_run("[ERROR] %s: %s" % (label, e))
            log_run(traceback.format_exc())
        finally:
            new_loop.close()

    threading.Thread(target=_runner, daemon=True).start()
    return "thread"


# ---------- 8.3. Кнопки (синхронные колбэки!) ----------
b_start_sel = widgets.Button(description="🚀 Начать поиск (выбранная)", button_style="primary")
b_start_pending = widgets.Button(description="⏳ Собрать по pending", button_style="info")
b_start_all = widgets.Button(description="⏩ Начать по всем", button_style="warning")
b_test_run = widgets.Button(description="🧪 Тест механики", button_style="success")
b_diag_run = widgets.Button(description="🩺 Диагностика", button_style="danger")
b_save_run = widgets.Button(description="💾 Сохранить", button_style="info")
b_stats_run = widgets.Button(description="📊 Stats", button_style="info")


def on_start_sel(_):
    idx = w_topics_select.value
    if idx is None or idx >= len(TASKS):
        log_run("[!] Выберите тему в списке тем."); return
    _start_async(_run_many_visible([idx]), "start_sel")


def on_start_pending(_):
    idxs = [i for i, t in enumerate(TASKS) if str(t.get("status", "pending")) == "pending"]
    if not idxs:
        log_run("[i] Нет pending-тем."); return
    _start_async(_run_many_visible(idxs), "start_pending")


def on_start_all(_):
    if not TASKS:
        log_run("[!] Список тем пуст."); return
    _start_async(_run_many_visible(list(range(len(TASKS)))), "start_all")


def on_test_run(_):
    async def _test():
        log_run("[тест] старт...")
        for i in range(3):
            await asyncio.sleep(0.4)
            log_run("  тест %d/3" % (i + 1))
        log_run("[OK] механика работает.")
    _start_async(_test(), "test")


def on_diag_run(_):
    async def _diag():
        log_run("[diag] окружение:")
        log_run("  run_task:", "OK" if callable(globals().get("run_task")) else "НЕТ")
        log_run("  con:", "OK" if "con" in globals() else "НЕТ")
        log_run("  TASKS:", len(TASKS))
        try:
            import crawl4ai; log_run("  crawl4ai: OK")
        except Exception as e: log_run("  crawl4ai: ОШИБКА ->", e)
        try:
            from github import Github; log_run("  PyGithub: OK")
        except Exception as e: log_run("  PyGithub: ОШИБКА ->", e)
        await asyncio.sleep(0.2)
        log_run("  dry-run: await OK")
    _start_async(_diag(), "diag")


def on_save_run(_):
    try:
        save_tasks(TASKS_JSON)
        log_run("[OK] Сохранено:", TASKS_JSON, "| заданий:", len(TASKS))
    except Exception as e:
        log_run("[!] Не сохранилось:", e)


def on_stats_run(_):
    if callable(globals().get("show_stats")):
        try: show_stats()
        except Exception as e: log_run("[stats] ошибка:", e)
    else:
        log_run("[!] show_stats() нет.")


b_start_sel.on_click(on_start_sel)
b_start_pending.on_click(on_start_pending)
b_start_all.on_click(on_start_all)
b_test_run.on_click(on_test_run)
b_diag_run.on_click(on_diag_run)
b_save_run.on_click(on_save_run)
b_stats_run.on_click(on_stats_run)


# ---------- 8.4. Отрисовка ----------
display(
    widgets.HTML("<hr><h3>5. Запуск поиска</h3>"),
    widgets.HBox([b_start_sel, b_start_pending, b_start_all]),
    widgets.HBox([b_test_run, b_diag_run, b_save_run, b_stats_run]),
    widgets.HTML("<hr><b>Лог запуска:</b>"),
    out_run,
)

*СОВЕШАТЕЛЬНЫЙ ЧАТ С ОБШИМ ДОСТУПОМ И НАСТРОЙКАМИ КРУГЛЫЙ СТОЛ*


---
* Хотите быстро спросить самого Директора: снимаете все галочки (0 галочек) и пишете в чат — отвечает мгновенно только DeepSeek-R1-7B.

* Хотите устроить совещание 2, 3 или 4 моделей: ставите галочки напротив нужных сотрудников (например, LFM2-350M + NuExtract-0.5B + Qwen-Coder-0.5B), выбираете 2 раунда и задаете вопрос — они по очереди просыпаются, спорят, а Директор в конце выдает единое решение и записывает его в ваш OpenObserve.

*  Запускайте эту Ячейку №4 — она максимально простая и позволяет прямо в окне чата собирать любую команду для обсуждения задачи
*   Новый пункт



In [ ]:
#@title 4. Пульт управления Штабом и живой чат (Выбор нескольких моделей)
import re

def ask_director_clean(llm_instance, system_msg: str, user_msg: str, max_tok: int = 650) -> str:
    """Вызов Директора с корректным разделением внутренних мыслей <think> и готового решения"""
    res = llm_instance.create_chat_completion(
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        max_tokens=max_tok,
        temperature=0.15
    )
    choices = res.get("choices", list())
    raw = next(iter(choices), dict()).get("message", dict()).get("content", "").strip()

    # Отделяем внутренний монолог <think> от финального ответа
    if "</think>" in raw:
        parts = raw.split("</think>", 1)
        thought = parts.replace("<think>", "").strip()
        answer = parts[1].strip()
        return f"<details><summary>Ход мыслей Директора (нажмите, чтобы развернуть)</summary>\n\n{thought}\n\n</details>\n\n{answer}"
    return raw.replace("<think>", "").strip()

ACTIVE_BOSS = "DeepSeek-R1-7B (Думающий Директор)" #@param ["DeepSeek-R1-7B (Думающий Директор)", "Qwen2.5-7B-Instruct (Заместитель-Аналитик)"]
DEFAULT_COUNCIL = ["lfm2", "coder"] #@param {type:"raw"}
DEBATE_ROUNDS = "2 раунда (Мнения + Перекрестный спор)" #@param ["1 раунд (Быстрые мнения)", "2 раунда (Мнения + Перекрестный спор)"]
MAX_REPLY_TOKENS = 450 #@param {type:"slider", min:150, max:1200, step:50}

import os, gc, glob, json, base64, psutil, requests
import gradio as gr
from llama_cpp import Llama

# Переключение руководителя по выбору из формы
if "Qwen2.5-7B" in ACTIVE_BOSS and os.path.exists(qw_path):
    if "director_llm" in globals():
        del director_llm
        gc.collect()
    director_llm = Llama(model_path=qw_path, n_ctx=6144, n_threads=2, verbose=False)

# Все доступные участники для галочек в чате
CREW_ROSTER = {
    "lfm2":      ("resident/Liquid-LFM2-350M",         "Оперативный диспетчер и критик гипотез"),
    "coder":     ("tables_graphs/Qwen2.5-Coder-0.5B",  "Инженер SQL (DuckDB) и логики"),
    "nuextract": ("tables_graphs/NuExtract-tiny-0.5B", "Делопроизводитель строгого JSON"),
    "gemma":     ("resident/Gemma-3-270M-IT",          "Аналитик длинных отчетов"),
    "deputy_7b": ("executive_7b",                      "Заместитель Директора (Qwen2.5-7B)")
}

DIRECTOR_PROMPT = (
    "Ты — Главный Директор коллектива специализированных моделей AMBER SOCIETY. "
    "Отвечай четко, структурно, на русском языке, опираясь на выводы сотрудников."
)

def extract_text(res: dict) -> str:
    choices = res.get("choices", list())
    first = next(iter(choices), dict())
    return first.get("message", dict()).get("content", "").strip()

def wake_worker(key: str, task: str, board: str) -> str:
    if key not in CREW_ROSTER:
        return f"[{key} не найден]"
    folder, role = CREW_ROSTER.get(key)
    search_dir = os.path.join(MODELS_DIR, folder)
    files = [f for f in glob.glob(os.path.join(search_dir, "*.gguf")) if "DeepSeek" not in f]
    mfile = next(iter(files), None)
    if not mfile:
        return f"[Файл для {key} не найден в {folder}]"

    w = Llama(model_path=mfile, n_ctx=2048, n_threads=2, verbose=False)
    r = w.create_chat_completion(
        messages=[{"role": "user", "content": f"Роль: {role}.\nОбщая доска:\n{board}\n\nЗадание: {task}"}],
        max_tokens=200, temperature=0.15
    )
    ans = extract_text(r)
    del w
    gc.collect()
    return ans

def chat_handler(message, history, selected_models, rounds_mode, max_tok):
    ram_mb = round(psutil.Process().memory_info().rss / (1024**2))
    selected = selected_models or list()

    # Если галочки моделей не выбраны — прямой диалог только с Директором
    if len(selected) == 0:
        msgs = [{"role": "system", "content": DIRECTOR_PROMPT}]
        for item in history[-4:]:
            if isinstance(item, dict):
                msgs.append({"role": item.get("role", "user"), "content": str(item.get("content", ""))})
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                u, a = item
                msgs.append({"role": "user", "content": str(u)})
                msgs.append({"role": "assistant", "content": str(a)})
        msgs.append({"role": "user", "content": message})
        res = director_llm.create_chat_completion(messages=msgs, max_tokens=int(max_tok), temperature=0.2)
        reply = extract_text(res)

    # Если выбраны 2 и более моделей — запускаем Совещательную комнату по очереди!
    else:
        board_notes = list()
        # Раунд 1: Каждая выбранная модель высказывается по очереди
        for m_key in selected:
            op = wake_worker(m_key, message, "Старт совещания.")
            board_notes.append(f"**[{m_key} — Раунд 1]:** {op}")

        # Раунд 2 (если включен): Перекрестная критика общей доски
        if rounds_mode.startswith("2"):
            current_board = "\n".join(board_notes)
            for m_key in selected:
                crit = wake_worker(m_key, f"Оцени мнения коллег на доске, найди слабые места или дополни по задаче: {message}", current_board)
                board_notes.append(f"**[{m_key} — Раунд 2 (Спор)]:** {crit}")

        # Финал: Директор подводит общее решение
        full_board = "\n\n".join(board_notes)
        synth = f"Задача пользователя: {message}\n\nСтенограмма совещания сотрудников:\n{full_board}\n\nПодведи единое взвешенное решение коллектива:"
        res = director_llm.create_chat_completion(
            messages=[{"role": "system", "content": DIRECTOR_PROMPT}, {"role": "user", "content": synth}],
            max_tokens=int(max_tok), temperature=0.15
        )
        verdict = extract_text(res)
        reply = f"{full_board}\n\n---\n### ИТОГОВОЕ РЕШЕНИЕ ДИРЕКТОРА:\n{verdict}"

    # Отправка протокола в OpenObserve (:5080)
    try:
        o2_post = base64.b64decode("aHR0cDovLzEyNy4wLjAuMTo1MDgwL2FwaS9kZWZhdWx0L2FpX2NvbnNlbnN1cy9fanNvbg==").decode()
        requests.post(o2_post, auth=("admin@society.local", "ChangeMe123!"),
                      json=[{"participants": ",".join(selected), "question": message, "verdict": reply}], timeout=2)
    except Exception:
        pass

    return f"{reply}\n\n`[Участники: {', '.join(selected) if selected else 'Только Директор'} · ОЗУ: {ram_mb}/12700 МБ]`"

# Запуск простого и наглядного окна чата с галочками моделей
with gr.Blocks(title="AMBER SOCIETY — Совещательный чат") as demo:
    gr.Markdown(f"## AMBER SOCIETY — Чат-пульт (Руководитель: {ACTIVE_BOSS})")
    with gr.Row():
        models_chk = gr.CheckboxGroup(
            choices=[
                ("LFM2-350M (Диспетчер)", "lfm2"),
                ("Qwen-Coder-0.5B (SQL/Код)", "coder"),
                ("NuExtract-0.5B (Строгий JSON)", "nuextract"),
                ("Gemma-3-270M (Аналитик)", "gemma"),
                ("Qwen2.5-7B (Заместитель)", "deputy_7b")
            ],
            value=DEFAULT_COUNCIL,
            label="Кого позвать на совещание (0 галочек = прямой чат с Директором)"
        )
        with gr.Column():
            rounds_dd = gr.Radio(
                choices=["1 раунд (Быстрые мнения)", "2 раунда (Мнения + Перекрестный спор)"],
                value=DEBATE_ROUNDS,
                label="Формат обсуждения"
            )
            tok_sl = gr.Slider(minimum=150, maximum=1200, step=50, value=MAX_REPLY_TOKENS, label="Длина итогового ответа")

    gr.ChatInterface(fn=chat_handler, additional_inputs=[models_chk, rounds_dd, tok_sl])

demo.launch(share=True, debug=False)


ОБЫЧНЫЙ ЧАТ , С ОТСУТСТВИЕМ НАСТРОЕК

In [ ]:
#@title 3. Запуск живого чата с Директором и коллективом
import os, gc, glob, json, base64, psutil, requests
import urllib.request, urllib.parse, xml.etree.ElementTree as ET
import gradio as gr
from huggingface_hub import HfApi
from llama_cpp import Llama

CREW_ROSTER = {
    "lfm2":      ("resident/Liquid-LFM2-350M",         "Оперативный диспетчер и критик гипотез"),
    "nuextract": ("tables_graphs/NuExtract-tiny-0.5B", "Извлечение строгого JSON по схеме"),
    "coder":     ("tables_graphs/Qwen2.5-Coder-0.5B",  "Генератор SQL (DuckDB) и кода"),
    "gemma":     ("resident/Gemma-3-270M-IT",          "Аналитик длинных отчетов и текстов"),
}

DIRECTOR_PROMPT = (
    "Ты — Главный Директор коллектива из 28 специализированных микромоделей в пространстве AMBER SOCIETY. "
    "Отделы: Штаб (LFM2-350M, Qwen2.5-0.5B, Gemma-3-270M, SetFit), Датчики (Chronos-Bolt, MOMENT), "
    "Таблицы/JSON (NuExtract, Qwen-Coder, TabPFN), Зрение/PDF (YOLOv11, MobileSAM, Depth-Anything, Florence-2, ColModernVBERT), "
    "Генетика/Наука (Caduceus, HyenaDNA, Nucleotide-Transformer, DNABERT-2, ESM-2, ChemBERTa), "
    "Речь/Звук (Moonshine, Whisper, Piper-TTS, AST-BioAcoustics, Nomic-Embed). "
    "Отвечай пользователю четко, структурно, на русском языке, распределяя задачи между сотрудниками."
)

def extract_text(res: dict) -> str:
    choices = res.get("choices", list())
    first = next(iter(choices), dict())
    return first.get("message", dict()).get("content", "").strip()

def wake_worker(key: str, task: str, board: str) -> str:
    if key not in CREW_ROSTER:
        return f"[Сотрудник {key} не найден]"
    folder, role = CREW_ROSTER.get(key)
    files = glob.glob(os.path.join(MODELS_DIR, folder, "*.gguf"))
    mfile = next(iter(files), None)
    if not mfile:
        return f"[Файл для {key} пока отсутствует в {folder}]"
    w = Llama(model_path=mfile, n_ctx=2048, n_threads=2, verbose=False)
    r = w.create_chat_completion(
        messages=[{"role": "user", "content": f"Роль: {role}.\nДоска:\n{board}\n\nЗадача: {task}"}],
        max_tokens=200, temperature=0.15
    )
    ans = extract_text(r)
    del w
    gc.collect()
    return ans

def open_sources_and_deputy(question: str, draft: str) -> str:
    notes = list()
    try:
        from google.colab import ai as colab_ai
        avail = colab_ai.list_models()
        target = next(iter(avail), None)
        if target:
            ans = colab_ai.generate_text(f"Проверь: {draft}\nВопрос: {question}", model_name=target)
            notes.append(f"**[Colab AI ({target})]:**\n{ans}")
    except Exception:
        notes.append("*(Прямой Colab AI ограничен в регионе — подключены Hub/arXiv и Заместитель 7B)*")

    try:
        api = HfApi()
        models = list(api.list_models(search=question[:40], sort="downloads", direction=-1, limit=3))
        if models:
            lines = "\n".join([f"• `{m.id}` (скачиваний: {m.downloads})" for m in models])
            notes.append(f"**[Каталог моделей Hugging Face Hub]:**\n{lines}")
    except Exception:
        pass

    try:
        abase = base64.b64decode("aHR0cDovL2V4cG9ydC5hcnhpdi5vcmcvYXBpL3F1ZXJ5P3NlYXJjaF9xdWVyeT1hbGw6").decode()
        with urllib.request.urlopen(f"{abase}{urllib.parse.quote(question[:50])}&max_results=2", timeout=6) as resp:
            root = ET.fromstring(resp.read())
            ns = {"a": "http://w3.org"}
            papers = ["• " + " ".join(e.find("a:title", ns).text.split()) for e in root.findall("a:entry", ns) if e.find("a:title", ns) is not None]
            if papers:
                notes.append("**[Научный архив arXiv]:**\n" + "\n".join(papers))
    except Exception:
        pass

    if os.path.exists(qw_path):
        dep = Llama(model_path=qw_path, n_ctx=4096, n_threads=2, verbose=False)
        r = dep.create_chat_completion(
            messages=[
                {"role": "system", "content": "Ты Заместитель Директора (Qwen2.5-7B). Проверь и дополни вывод коллеги."},
                {"role": "user", "content": f"Вопрос: {question}\nЧерновик Директора: {draft}"}
            ],
            max_tokens=350, temperature=0.15
        )
        notes.append(f"**[Заместитель Директора (Qwen2.5-7B-Instruct)]:**\n{extract_text(r)}")
        del dep
        gc.collect()

    return "\n\n".join(notes)

def chat_handler(message, history, mode):
    ram_mb = round(psutil.Process().memory_info().rss / (1024**2))

    if mode.startswith("1."):
        msgs = [{"role": "system", "content": DIRECTOR_PROMPT}]
        for item in history[-4:]:
            if isinstance(item, dict):
                msgs.append({"role": item.get("role", "user"), "content": str(item.get("content", ""))})
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                u, a = item
                msgs.append({"role": "user", "content": str(u)})
                msgs.append({"role": "assistant", "content": str(a)})
        msgs.append({"role": "user", "content": message})
        res = director_llm.create_chat_completion(messages=msgs, max_tokens=500, temperature=0.2)
        reply = extract_text(res)

    elif mode.startswith("2."):
        op1 = wake_worker("lfm2", message, "Старт совещания")
        op2 = wake_worker("coder", f"Проверь технически и дополни: {message}", f"Мнение LFM2: {op1}")
        synth = f"Задача: {message}\nМнение LFM2: {op1}\nМнение Coder: {op2}\nПодведи итоговый консенсус:"
        res = director_llm.create_chat_completion(
            messages=[{"role": "system", "content": DIRECTOR_PROMPT}, {"role": "user", "content": synth}],
            max_tokens=450, temperature=0.15
        )
        reply = f"**[LFM2-350M]:** {op1}\n\n**[Qwen-Coder]:** {op2}\n\n---\n**[ИТОГ ДИРЕКТОРА]:**\n{extract_text(res)}"

    else:
        d_res = director_llm.create_chat_completion(
            messages=[{"role": "system", "content": DIRECTOR_PROMPT}, {"role": "user", "content": message}],
            max_tokens=300, temperature=0.2
        )
        draft = extract_text(d_res)
        extra = open_sources_and_deputy(message, draft)
        reply = f"**[Директор DeepSeek-R1-7B]:**\n{draft}\n\n---\n{extra}"

    try:
        requests.post(
            "http://127.0.0",
            auth=("admin@society.local", "ChangeMe123!"),
            json=[{"mode": mode, "question": message, "verdict": reply}],
            timeout=2
        )
    except Exception:
        pass

    return f"{reply}\n\n`[ОЗУ Colab: {ram_mb} / 12700 МБ · Путь: {CAPSULE_DIR}]`"

with gr.Blocks(title="AMBER SOCIETY — Штаб Colab") as demo:
    gr.Markdown("## AMBER SOCIETY — Командный центр (DeepSeek-R1-7B + Коллектив + OpenObserve)")
    mode_radio = gr.Radio(
        choices=[
            "1. Прямой диалог с Главным Директором (DeepSeek-R1-7B)",
            "2. Созвать Совет коллектива (Директор + Сотрудники с Диска)",
            "3. Консилиум (Директор + Заместитель 7B + Открытые архивы Hub / arXiv)"
        ],
        value="1. Прямой диалог с Главным Директором (DeepSeek-R1-7B)",
        label="Режим работы штаба"
    )
    gr.ChatInterface(fn=chat_handler, additional_inputs=[mode_radio])

demo.launch(share=True, debug=False)


*ОСНОВНОЕ ПРОСТРАНСТВО  пространство создано*


---




*ОБУЧЕНИЕ СОТРУДНИКОВ*


*   Новый пункт
*   Новый пункт



# POTOK / Приток — рождение пустой нейросети (дообучение при каждом подключении)

**Полигон:** бесплатный Colab, Tesla T4. **Время сессии:** ~10–20 минут.

Мы инициализируем **случайную (пустую) char-level GPT** (~10.8M параметров) и обучаем её с нуля на тексте репозитория POTOK. Никаких внешних весов, никаких лицензий на модели, ничего наружу без слова основателя.

Это витрина/искусство и мой первый измеримый эксперимент с памятью, не компонент организма.


## Как это работает при каждом подключении (память между сессиями)

1. Ячейка 1 монтирует Google Drive (`MyDrive/potok/`). Там живут: `potok-nano.pt` (чекпоинт, перезаписывается атомарно), `corpus.txt`, `run_log.jsonl` (по строке на подключение).
   - Drive недоступен → всё работает, но чекпоинт не переживёт отключение рантайма (ноутбук скажет прямо).
2. Ячейка «Возобновление» грузит чекпоинт, если он есть и совпадает словарь. Шаги продолжаются с `global_step` — косинус-расписание считается от общего горизонта **12000** шагов (`2000` за подключение, ~6 подключений).
3. Корпус собирается заново из свежего клона `main` при каждом запуске; если состав символов изменился — чистый старт (старые веса не грузим, сообщим).
4. После прогона ячейка «Сохранение» допишет строку в `run_log.jsonl` (лоссы, шаги, sha256 чекпоинта).
5. **Перенос в мою память дома** — только твоими руками: скопируй `run_log.jsonl` и выполни
   `python3 pritok/knowledge/memory/mem.py import-jsonl potok-nano-birth run_log.jsonl`
   затем `python3 pritok/knowledge/memory/mem.py recent` — и факты лягут в `pritok/knowledge/memory/memory.db`. Автоматически ничего не уезжает: политика дома — перенос вывода основателем.


In [ ]:
import os, hashlib

WORK = "/content/potok-work"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/potok"
    os.makedirs(WORK, exist_ok=True)
    print("Drive смонтирован →", WORK)
except Exception as e:
    os.makedirs(WORK, exist_ok=True)
    print("Drive недоступен (" + str(e) + ") → работаем в", WORK)
    print("ВНИМАНИЕ: чекпоинт и run_log.jsonl НЕ переживут отключение рантайма.")
print("рабочая папка:", WORK)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive смонтирован → /content/drive/MyDrive/potok
рабочая папка: /content/drive/MyDrive/potok


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    print("ВНИМАНИЕ: T4 не выбран. Runtime -> Change runtime type -> T4.")


torch: 2.14.1+cu130
cuda available: True
gpu: Tesla T4
vram GB: 14.6


In [ ]:
import shutil, subprocess

REPO = "/content/potok-repo"
if os.path.exists(REPO):
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/COL-O/POTOK.git", REPO], check=True)

EXTS = (".md", ".txt", ".go", ".py", ".sh", ".yaml", ".yml",
        ".json", ".toml", ".mod")
parts = []
total_bytes = 0
for root, dirs, files in os.walk(REPO):
    dirs[:] = [d for d in dirs if d != ".git"]
    for name in sorted(files):
        if not name.lower().endswith(EXTS):
            continue
        path = os.path.join(root, name)
        try:
            with open(path, "rb") as f:
                raw = f.read()
            text = raw.decode("utf-8", errors="ignore").strip()
        except OSError:
            continue
        if not text:
            continue
        parts.append(text)
        total_bytes += len(raw)

corpus = "\n\n".join(parts)
if len(corpus) > 3_000_000:
    corpus = corpus[:3_000_000]

corpus_path = os.path.join(WORK, "corpus.txt")
with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(corpus)

h = hashlib.sha256(corpus.encode("utf-8")).hexdigest()[:12]
print(f"текст-файлов: {len(parts)}")
print(f"сырых байт:   {total_bytes:,}")
print(f"символов в корпусе: {len(corpus):,} | sha256[:12]: {h}")
print("--- фрагмент ---")
print(corpus[:400])


текст-файлов: 351
сырых байт:   1,728,471
символов в корпусе: 1,359,014 | sha256[:12]: 710d3b068b04
--- фрагмент ---
# golangci-lint для ПОТОКА — версия схемы 2 (golangci-lint v2.x)
#
# Почему так, а не «включить всё»: организм — система реального времени, где
# важнее всего гонки, утечки ресурсов и необработанные ошибки, а не длина строк.
# Поэтому включены линтеры, которые ловят поведение, и намеренно выключены
# стилевые (lll, godot, varnamelen, exhaustruct): они дают много шума и ничего
# не говорят о том, ж


In [ ]:
# Char-level GPT, ~10.8M параметров. Горизонт 12000 шагов, 2000 за подключение.
CONFIG = dict(
    seed=1337,
    block_size=256,
    batch_size=64,
    n_layer=6, n_head=6, n_embd=384, dropout=0.1,
    lr=1e-3, min_lr=1e-4, weight_decay=0.1,
    warmup_iters=100,
    lr_decay_iters=12000,       # общий горизонт косинуса (6 подключений × 2000)
    iters_per_run=2000,         # сколько шагов за это подключение
    eval_interval=250,
    eval_iters=20,
    grad_clip=1.0,
)
print(CONFIG)


{'seed': 1337, 'block_size': 256, 'batch_size': 64, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'dropout': 0.1, 'lr': 0.001, 'min_lr': 0.0001, 'weight_decay': 0.1, 'warmup_iters': 100, 'lr_decay_iters': 12000, 'iters_per_run': 2000, 'eval_interval': 250, 'eval_iters': 20, 'grad_clip': 1.0}


In [ ]:
import torch

corpus = open(os.path.join(WORK, "corpus.txt"), encoding="utf-8").read()
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, device):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONFIG["block_size"], (CONFIG["batch_size"],))
    x = torch.stack([d[i:i + CONFIG["block_size"]] for i in ix])
    y = torch.stack([d[i + 1:i + CONFIG["block_size"] + 1] for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

print(f"vocab={vocab_size} | tokens={len(data):,} | train={len(train_data):,} | val={len(val_data):,}")


vocab=246 | tokens=1,359,014 | train=1,223,112 | val=135,902


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["n_embd"] % cfg["n_head"] == 0
        self.n_head = cfg["n_head"]
        self.n_embd = cfg["n_embd"]
        self.c_attn = nn.Linear(cfg["n_embd"], 3 * cfg["n_embd"])
        self.c_proj = nn.Linear(cfg["n_embd"], cfg["n_embd"])
        self.attn_drop = nn.Dropout(cfg["dropout"])
        self.resid_drop = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_drop.p if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg["n_embd"], 4 * cfg["n_embd"])
        self.c_proj = nn.Linear(4 * cfg["n_embd"], cfg["n_embd"])
        self.drop = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["n_embd"])
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["n_embd"])
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.block_size = cfg["block_size"]
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["n_embd"])
        self.pos_emb = nn.Embedding(cfg["block_size"], cfg["n_embd"])
        self.drop = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg["n_layer"]))
        self.ln_f = nn.LayerNorm(cfg["n_embd"])
        self.head = nn.Linear(cfg["n_embd"], cfg["vocab_size"], bias=False)
        self.head.weight = self.tok_emb.weight  # tied embeddings
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / (2 * cfg["n_layer"]) ** 0.5)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

device = "cuda" if torch.cuda.is_available() else "cpu"
model_cfg = {**CONFIG, "vocab_size": vocab_size}
torch.manual_seed(CONFIG["seed"])
model = GPT(model_cfg).to(device)
print(f"параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | device: {device}")


параметров: 10.84M | device: cuda


In [ ]:
# Возобновление: грузим чекпоинт, если он есть и словарь совпал
CKPT = os.path.join(WORK, "potok-nano.pt")
start_iter = 0
resumed = False
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    args = ck.get("model_args", {})
    if args.get("vocab_size") == vocab_size and ck.get("chars") == chars:
        model.load_state_dict(ck["model"])
        start_iter = int(ck.get("global_step", 0))
        resumed = True
        print(f"чекпоинт найден → продолжаем с шага {start_iter}")
    else:
        print("словарь/корпус изменился → чистый старт (старые веса не грузим)")
else:
    print("чекпоинта нет → чистый старт")


чекпоинт найден → продолжаем с шага 2000


In [ ]:
import math, time

decay, no_decay = [], []
for _, p in model.named_parameters():
    (decay if p.dim() >= 2 else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": CONFIG["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=CONFIG["lr"], betas=(0.9, 0.95))

def get_lr(it):
    if it < CONFIG["warmup_iters"]:
        return CONFIG["lr"] * (it + 1) / CONFIG["warmup_iters"]
    if it > CONFIG["lr_decay_iters"]:
        return CONFIG["min_lr"]
    ratio = (it - CONFIG["warmup_iters"]) / (CONFIG["lr_decay_iters"] - CONFIG["warmup_iters"])
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return CONFIG["min_lr"] + coeff * (CONFIG["lr"] - CONFIG["min_lr"])

use_amp = device == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    def autocast():
        return torch.autocast("cuda", dtype=torch.float16, enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def autocast():
        return torch.cuda.amp.autocast(enabled=use_amp)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(CONFIG["eval_iters"])
        for i in range(CONFIG["eval_iters"]):
            xb, yb = get_batch(split, device)
            with autocast():
                _, loss = model(xb, yb)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

t0 = time.time()
last_train, last_val = None, None
for local in range(CONFIG["iters_per_run"]):
    it = start_iter + local
    lr = get_lr(it)
    for group in optimizer.param_groups:
        group["lr"] = lr
    if local % CONFIG["eval_interval"] == 0:
        losses = estimate_loss()
        last_train, last_val = losses["train"], losses["val"]
        print(f"step {it:6d} | lr {lr:.2e} | train {last_train:.4f} | val {last_val:.4f} | {time.time() - t0:.0f}s")
    xb, yb = get_batch("train", device)
    with autocast():
        _, loss = model(xb, yb)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

losses = estimate_loss()
last_train, last_val = losses["train"], losses["val"]
GLOBAL_STEP = start_iter + CONFIG["iters_per_run"]
print(f"готово: шаги {start_iter}..{GLOBAL_STEP - 1} за {time.time() - t0:.0f}s | train {last_train:.4f} | val {last_val:.4f}")


step   2000 | lr 9.45e-04 | train 0.6126 | val 0.9257 | 2s
step   2250 | lr 9.29e-04 | train 0.5583 | val 0.9370 | 40s
step   2500 | lr 9.13e-04 | train 0.5187 | val 0.9773 | 75s
step   2750 | lr 8.94e-04 | train 0.4718 | val 1.0030 | 111s
step   3000 | lr 8.74e-04 | train 0.4208 | val 0.9840 | 147s
step   3250 | lr 8.53e-04 | train 0.3910 | val 1.0495 | 184s
step   3500 | lr 8.31e-04 | train 0.3497 | val 1.0413 | 220s
step   3750 | lr 8.07e-04 | train 0.3149 | val 1.0212 | 255s
готово: шаги 2000..3999 за 292s | train 0.2918 | val 1.0417


In [ ]:
import hashlib, json, datetime

ckpt = {
    "model_args": {**model_cfg, "dropout": 0.0},
    "model": model.state_dict(),
    "chars": chars,
    "config": CONFIG,
    "global_step": GLOBAL_STEP,
}
tmp = CKPT + ".tmp"
torch.save(ckpt, tmp)
os.replace(tmp, CKPT)  # атомарная подмена — не оставим битый файл

h = hashlib.sha256(open(CKPT, "rb").read()).hexdigest()
run_line = {
    "ts": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "experiment": "potok-nano-birth",
    "iters_total": GLOBAL_STEP,
    "iters_in_run": CONFIG["iters_per_run"],
    "resumed": resumed,
    "train_loss": last_train,
    "val_loss": last_val,
    "wall_seconds": round(time.time() - t0, 1),
    "checkpoint_path": CKPT,
    "checkpoint_bytes": os.path.getsize(CKPT),
    "checkpoint_sha256": h,
    "source": "colab-t4",
}
log_path = os.path.join(WORK, "run_log.jsonl")
with open(log_path, "a", encoding="utf-8") as f:
    f.write(json.dumps(run_line, ensure_ascii=False) + "\n")

print("чекпоинт:", CKPT, "|", os.path.getsize(CKPT) // 1024, "KB | sha256:", h[:16])
print("строка прогона дописана в:", log_path)
print("перенос в память дома (твоими руками): python3 pritok/knowledge/memory/mem.py import-jsonl potok-nano-birth run_log.jsonl")


чекпоинт: /content/drive/MyDrive/potok/potok-nano.pt | 42375 KB | sha256: 1fbd12eede93fdcd
строка прогона дописана в: /content/drive/MyDrive/potok/run_log.jsonl
перенос в память дома (твоими руками): python3 pritok/knowledge/memory/mem.py import-jsonl potok-nano-birth run_log.jsonl


In [ ]:
model.eval()
for prime in ["POTOK", "Приток", "# "]:
    ctx = torch.tensor([[stoi.get(c, 0) for c in prime]],
                       dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=400, temperature=0.8, top_k=40)
    print("=" * 24, repr(prime))
    print("".join(itos[int(i)] for i in out[0]))
    print()
print("Коллаж, не знание: модель видела только текст репозитория.")


======================== 'POTOK'
POTOK.md`, `SYNTHES_MEMORY.md`, `SYNTHES_MEMORY.md`, `CHAT.md`, `CONTRIBUTING.md`, `CONTRIBUTING.md`, `CONTRIBUTING.md`, `docs/COPILOT-SPACES-MAP.md`). **Запись** — `docs/07-network.md` (дашборд прозрачность), `docs/COPILOT-SPACES-MAP.md` | Знакомый платформый план участников: `docs/PROJECT-SPACE.md`, `docs/PROJECT-SPACE.md`, `docs/ROUND-TABLE.md` |
| Отделы-проекта соратники | Полный запуск и стол Рус

======================== 'Приток'
Приток через `Приток` (переезд ленты в `Принято`) |

## Права три проекта — не среда и вообще живого через проекта

Основатель дал сделать владельцу доступно основателя (находка начнуть публичные категории Discussions) — роль обсуждения Arena GitHub-лого App на GitHub интерфейса для права Arena и на GitHub. Доступ Сферы приложения Arena (токен только Pogress/API, создан, нарушение другой Arena) — получи

======================== '# '
# TestGrowthAdversarialReplayRejectsBadLaw` (паппка не выглядеть при пакете код примеров

## Как читать результат

| Признак | Хорошо | Плохо (чиним) |
|---|---|---|
| val loss | снижается с каждым подключением | плато/NaN → проверь Runtime, lr, корпус |
| samples | глифовый коллаж русско-английского, знакомые слова (POTOK, Приток) | одна буква в цикле |
| память | каждый прогон дописывает строку в run_log.jsonl, чекпоинт переживает рантайм на Drive | Drive не примонтирован — чекпоинт пропадёт при отключении |

**Норма эксперимента:** 6 подключений × 2000 шагов = 12000 шагов, cosine 1e-3 → 1e-4. После каждого — переноси run_log.jsonl в memory.db командой из ячейки «Сохранение»; `mem.py recent` покажет динамику.

## Что здесь намеренно НЕТ
- Никакой отправки весов/корпуса наружу (HF Hub, Drive шаринг дальше твоего) — только по слову.
- Никакой записи в репозиторий из тетради — только по твоему переносу.
- Никакой стыковки весов с Go-контуром — тензор и организм на разных полюсах (`docs/colab-lane.md`).

## Честные ограничения
- Корпус ≈ 1.7 МБ, char-level: сеть переобучится; это витрина, не «знание».
- Числа лосса сравнимы только внутри этой конфигурации (один корпус, один seed).
- Дообучение при каждом подключении ≠ непрерывное обучение в процессе: каждая сессия отдельный прогон 2000 шагов от последнего чекпоинта.


## План Б (если нужен осмысленный текст, а не коллаж)
Готовая малая база: `HuggingFaceTB/SmolLM2-360M` (base, без гейта), `stage: pt` через LlamaFactory на `corpus.txt` (`cutoff_len: 1024`, `finetuning_type: full` — моделью малой памяти хватит). Это чужие веса и чужая пре-жизнь; выбор за основателем.

Альтернатива из брифа 2026-11-20: litgpt pretrain-этапами или RWKV-LM train.py — тяжелее в настройке, тот же полигон T4.


*ОБЩИЙ ПОДХОД ОБУЧЕНИЯ*


*   Новый пункт
*   Новый пункт



LFM3 - это наше новое поколение моделей Liquid Foundation (LFM), созданных для того, чтобы стать отправной точкой для специализированной разведки. Он приносит текст, зрение, аудио, омни и модели решений на одну общую основу, все в диапазоне параметров 2B. Это означает, что вы можете выбрать модель для своего способа, точно настроить ее на свои собственные данные на одном графическом процессоре и развернуть ее там, где работает ваше приложение.

LFM3 приносит четыре основных изменения по сравнению с нашим предыдущим семейством LFM2:

    Больше возможностей для параметра: новая трехгибридная архитектура и рецепт обучения позволяют LFM3-1.7B превзойти LF2.5-2.6B, используя на 35% меньше параметров.
    Каждый способ на одном фоне: шесть моделей 2B-класса (текст, зрение, аудио, omni и решение), обученных у LFM3-1.7B-Base, включая нашу первую омни-модель и видео-вход для нашей модели зрения.
    Построена для края: текстовая модель вписывается примерно в 1 ГБ с 4-битной точностью, новое внутреннее зрение и аудиокодеры ускоряют обработку, а контекстное окно 256K текстовой модели позволяет агентам работать на устройстве.
    Сделано для того, чтобы быть специализированным: каждая модель точно настраивается на одном графическом процессоре, поэтому вы можете адаптировать его к своим данным, терминологии и среде развертывания.

Как мы построили более выразительный костяк

LFM2 показал, что гибрид коротких свертков и внимания может быстро работать на устройстве без ущерба для качества. Для LFM3 мы использовали наш поиск архитектуры аппаратного обеспечения для перепроектирования этой архитектуры для вариантов использования агентов, которые требуют больше возможностей для параметра и быстрого вывода в течение длительных контекстов. Поиск теперь охватывает 14 комбинаций устройств и бэкэнда, от телефонов и Raspberry Pi 5 до ноутбуков, облачных процессоров, графических процессоров и Qualcomm Hexagon NPU.

lfm3-1.7b-архитектура

LFM3-1.7B-Base имеет 24 слоя и четыре основных изменения:

    Тригибридная смесь слоев: 13 закрытых коротких свертков, 5 слоев с скользящим окном (SWA) с окном с 256-тоновым окном и 6 глобальных слоев внимания. Свертки используют ядро размером 7, по сравнению с 3 в LFM2.
    С головой закрытое внимание: сигмовидные ворота масштабируют выход каждой головы, поэтому голова может отключиться, когда ей нечего добавить. Это может помочь подавить выбросы активации, которые повредят квантованию.
    Частичный RoPE: глобальные слои внимания вращают только самую высокочастотную половину каждой головы, поэтому другая половина совпадает по содержанию независимо от расстояния. Слои SWA сохраняют полный RoPE.
    Сэндвич-нормальность: каждый блок нормализует как свой вход, так и выходной, чтобы поддерживать хорошо масштабируемые активации.

Выигрыш проявляется в длинном контексте. В соответствии с параметром, LFM3-1.7B соответствует LFM2-1.2B на коротких подсказках на периферийных устройствах, а с токенами 8K в контексте это примерно на 25% быстрее. LFM3-1.7B-Base был предварительно обучен на токенах 22T с тем же словарем 128K, что и LFM2.5-8B-A1B. Окончательные токены 3.5T представляют собой этап средней подготовки по курируемым данным с большей долей агентных следов и вызовов инструментов, что также расширяет контекстное окно до 256K-токенов.
Один хребет для шести моделей

Каждая модель LFM3 от LFM3 от LFM3-1.7B-Base. Вместо того, чтобы запускать одну большую модель общего назначения, вы выбираете ту, которая соответствует вашим входам, настраиваете ее и запускаете на устройстве.

lfm3-после обучения
Модель 	Ввод 	Выход 	Построенный на
ЛФМ3-1.7Б 	Текст 	Текст 	LFM3-1.7B-база
ЛФМ3-ВЛ-2Б 	Текст, изображения, видео 	Текст 	LFM3 магистраль + кодер зрения
LFM3-Аудио-2Б 	Текст, аудио 	Текст, речь 	LFM3 магистраль + аудио кодер, аудио декодер и LiquidCodec
ЛФМ3-Омни-2Б 	Текст, изображения, видео, аудио 	Текст, речь 	LFM3 магистраль + зрение и аудио кодеры и аудио декодер
d1-2B 	Текст, изображения, видео 	Калиброванные ответы 	ЛФМ3-ВЛ-2Б
d1-omni-2B 	Текст, изображения, видео, аудио 	Калиброванные ответы 	ЛФМ3-Омни-2Б

Видение и аудиомодели также получают два новых внутренних кодера:

    Кодер видения: он сочетает в себе свертку и внимание и использует в 3 раза меньше вычислений, чем кодировщик SigLIP2, в предыдущих моделях LFM VL. Наш 164M-параметрический кодер соответствует кодировщику зрения Qwen3.5-2B по точности, требуя при этом 2,2x меньше FLOP (оба измеряются с одинаковой языковой основой LFM2.5-230M). Для видео он встраивает каждую пару отобранных кадров в один височный срез, вдвое уменьшая длину временной последовательности.
    Аудиокодер и LiquidCodec: аудио-кодировщик смешивает внимание скользящего окна и короткие свертки и является полностью причинным, поэтому модель может предварительно заполнить, пока вы говорите, со временем до первого токена до 50 мс. LiquidCodec, наш новый аудиотокейзер, превращает генерируемые токены в речь примерно в 87x время в реальном времени на одной резьбе процессора на Apple M4 Max и поддерживает потоковое декодирование.

Как использовать модели LFM3

Каждая модель работает в transformers. Фрагменты ниже взяты из карт модели, в которых также указаны другие двигатели вывода, которые поддерживает каждая модель.
ЛФМ3-1.7Б

Достигайте LFM3-1.7B, когда вам нужны локальные агенты, вызовы инструментов, структурированные выходы или база для тонкой настройки домена. Он последует работе в агентских жгутках, таких как Hermes Agent, Pi и OpenClaw. Он превосходит своего предшественника, LFM2,5-2.6B, с 35% меньшим параметрами, и наибольший выигрыш приходится на агентные задачи: на WebArena-Verified он набирает 38,42 против 3,02, и он даже превосходит Qwen3.5-9B по длинным задачам в браузерах и бизнес-приложениях. Более подробную информацию можно найти в типовой карте LFM3-1.7B.

Для запуска LFM3-1.7B установите последнюю версию transformers:

pip install -U transformers

Затем загрузите и запустите модель:

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="LiquidAI/LFM3-1.7B",
    trust_remote_code=True,
)

messages = [{"role": "user", "content": "What is C. elegans?"}]
output = generator(
    messages,
    do_sample=True,
    temperature=0.1,
    top_k=50,
    repetition_penalty=1.1,
    max_new_tokens=2048,
)
print(output[0]["generated_text"][-1]["content"])

Для установки местного агента см. Руководство агента Harnesses.
ЛФМ3-ВЛ-2Б

Доходите до LFM3-VL-2B, когда вам нужно визуальное понимание на устройстве. Это первая модель зрения LFM, которая воспринимает видео как вход; она рассуждает шаг за шагом, прежде чем она отвечает, и она находит объекты и причины о том, как они размещены, прочная основа для робототехники. По сравнению с LFM2,5-VL-3B, он имеет меньше параметров (1,9B против 3,1B), но повышает средний показатель по 62 ориентирам зрения с 57,6 до 71,4, наравне с Gemma 4 12B, моделью, более чем в шесть раз больше. Более подробную информацию можно найти в типовой карте LFM3-VL-2B.

Для запуска LFM3-VL-2B установите трансформеры>=5.21.0 с его зависимостями зрения:

pip install "transformers>=5.21.0" torch torchvision pillow

Затем загрузите и запустите модель:

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "LiquidAI/LFM3-VL-2B"

model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://placecats.com/300/200"},
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=0.1,
        top_k=50,
        repetition_penalty=1.1,
        max_new_tokens=16384,  # generate() needs an explicit limit; reasoning can be long
    )

generated_ids = output_ids[:, inputs["input_ids"].shape[1] :]
print(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])

См. карту модели LFM3-VL-2B для вызова инструментов, поиска инструмента и вывода на устройстве с llama.cpp.
LFM3-Аудио-2Б

Свяжитесь с LFM3-Audio-2B, когда вам нужен голосовой чат, распознавание речи, синтез речи или перевод на 18 языков на ноутбуке или периферийном устройстве. Он принимает текст или аудио в качестве ввода и отвечает текстом, речью или и тем, и другим, и он может превращать устные запросы в инструментальные вызовы. По сравнению с LFM2.5-Audio-1.5B, он использует новый собственный аудио-кодировщик и речевой кодек, говорит с более высоким качеством (UTMOSv2 3.83 против 3.22) и добавляет понимание звука, аудио-рассуждение общего назначения и вызов инструментов. Более подробную информацию можно найти в типовой карте LFM3-Audio-2B.

Для запуска LFM3-Audio-2B установить transformers>=5.20.2с его аудиозависимостью:

pip install "transformers>=5.20.2" torch accelerate librosa soundfile

Загрузите модель и процессор прямо из Hugging Face Hub:

import torch
import soundfile as sf

from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Audio-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=device).eval()

Затем задайте вопрос вслух и получите ответ в тексте и речи:

conversation = [
    {"role": "system", "content": "Respond with interleaved text and audio."},
    {"role": "user", "content": [{"type": "audio", "audio": "question.wav"}]},
    {"role": "assistant", "content": "<|mixed_start|>"},
]
inputs = processor.apply_chat_template(
    conversation,
    continue_final_message=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        audio_temperature=1.0,
        audio_top_k=128,
        audio_repetition_penalty=1.1,
        audio_repetition_window=4096,
        return_dict_in_generate=True,
    )

    text = processor.decode(
        outputs.sequences[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    waveform = codec.decode(
        outputs.audio_codes[:1, :, :outputs.audio_lengths[0]]
    ).audio_values[0]  # 24 kHz
print(text)
sf.write("reply.wav", waveform.float().cpu().numpy(), 24_000)

См. карту модели LFM3-Audio-2B для распознавания речи в реальном времени, синтеза речи, понимания звука и звонков инструментов.
ЛФМ3-Омни-2Б

Свяжитесь с LFM3-Omni-2B, когда вам нужна одна модель на устройстве, которая видит, слушает и переговаривает, вместо отдельной модели ASR, модели языка зрения, LLM и модели TTS. Он может отвечать на вопросы об изображении, смотреть и слушать видео вместе, а также превращать то, что он видит и слышит, в призывы к инструментам. Это наша первая omni модель с текстом, видением и аудио вводом, а также текстом и речевым выводом, и она сохраняет основные возможности автономных моделей LFM3, включая визуальный ответ на вопросы, распознавание речи, синтез речи и следование инструкциям. Как и LFM3-Audio-2B, он говорит через LiquidCodec, который превращает генерируемые токены в речь быстрее, чем в реальном времени, поэтому передаваемые ответы транслируются по мере их создания модели. Что касается аудиовизуального видеопонимания, то он превосходит модели до 10,7B параметров, таких как Qwen2.5-Omni-7B, на Daily-Omni, WorldSense и AV-SpeakerBench. Более подробную информацию можно найти в типовой карте LFM3-Omni-2B.

Для запуска LFM3-Omni-2B установить transformers>=5.20.2с его видением и аудиозависимостью:

pip install "transformers>=5.20.2" torch torchvision torchcodec accelerate pillow librosa soundfile

import torch
import soundfile as sf

from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "LiquidAI/LFM3-Omni-2B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForMultimodalLM.from_pretrained(
    model_id, dtype=dtype, device_map=device, trust_remote_code=True
).eval()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
codec = processor.audio_tokenizer.to(device=device).eval()

Тогда задайте вопрос о фотографии голосом:

conversation = [
    {
        "role": "system",
        "content": "Answer questions about the provided image. Base every answer only on what can be seen. Respond with text.",
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "photo.jpg"},
            {"type": "audio", "audio": "question.wav"},
        ],
    },
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1024, generation_mode="text")

print(processor.decode(
    outputs.sequences[0, inputs["input_ids"].shape[1]:outputs.sequence_lengths[0]],
    skip_special_tokens=True,
))

Смотрите модельную карту LFM3-Omni-2B для вывода речи, видео с ее саундтреком и звонком инструментов.
d1-2B и d1-omni-2B

Достигайте для моделей d1, когда вам нужны быстрые, структурированные решения вместо сгенерированного текста. Дайте им ситуацию (билет поддержки, счет, экран приложения, фотографию, голосовую заметку) и список вопросов, и они отвечают на все из них одним проходом вперед, каждый с выверенной вероятностью. Помимо «да»/нет, выбора и баллов, теперь они могут наводить на объекты, читать значения из документов, оценивать цифры с диапазоном и выбирать каждый вариант, который применяется. Более подробную информацию можно найти в картах моделей для d1-2B и d1-omni-2B.

Между двумя моделями d1:

    d1-2B: выберите его для текста, картинок и видео. Он заменяет d1-3B с на треть меньшим параметром и набирает больше баллов по всем четырем нашим наборам решений, например, 75,5 против 64,0 на ImajevBench.
    d1-omni-2B: выберите его, когда ваши входные данные включают аудио. Если вы спрашиваете голосом, а не в письменной форме, он сохраняет почти всю свою точность.

Для запуска d1-2B, установить transformers>=5.19с его зависимостью видения:

pip install "transformers>=5.19" torch torchvision pillow

Затем загрузите d1-2B и задайте несколько вопросов об одном опорном билете в один проход:

import torch
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float32 if device == "cpu" else torch.bfloat16
model = AutoModel.from_pretrained("LiquidAI/d1-2B", trust_remote_code=True, dtype=dtype).to(device)

# Several named questions over one state, answered in one pass
questions = {
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},  # yes/no: P(yes)
    "team": {"type": "choice", "instructions": "Which team should handle this?",
             "criteria": {"billing": "Charges, refunds, invoices", "shipping": "Delivery and parcels",
                          "technical": "App or site faults"}},
    "topics": {"type": "choice", "multiple": True, "instructions": "Which topics does the message raise?",
               "criteria": {"billing": "Payments and charges", "delivery": "Shipping and parcels",
                            "login": "Signing in to the account"}},
    "urgency": {"type": "score", "instructions": "How urgent is this ticket?", "criteria": ["low", "medium", "high"]},
}
out = model.system_one("I was charged twice and my parcel never arrived. Please refund one charge.", questions)
a = out["answers"]
urgency = a["urgency"]["legend"][str(int(a["urgency"]["score"]))]
print(round(a["refund"]["noul"], 2), a["team"]["choice"], a["topics"]["choices"], urgency)
# 1.0 billing ['billing', 'delivery'] medium

Для краткости мы включаем только пример d1-2B. Смотрите модельную карту d1-omni-2B для аудиоклипов и устных вопросов.
Тонкая настройка LFM3 для вашей задачи

LFM3 построен для того, чтобы быть специализированным. Пограничные модели являются мощными универсалами, но они не обязательно являются экспертами в вашем варианте использования. LFM3 объединяет дополнительную емкость в небольшую, быструю модель, поэтому короткая тонкая настройка может превратить ее в надежного эксперта по вашей задаче, с вашими данными, терминологией и инструментами. Поскольку он небольшой, вы можете развернуть доработанную модель, где работает ваше приложение, от устройств до частной инфраструктуры и облака.

Каждая модель может быть точно настроена на одном GPU, и эти ноутбуки помогут вам выполнить полную задачу для текстовых, зрений, аудио и омни-моделей, от подготовки данных до оценки.
Модель 	Метод 	Задача 	Записная книж
ЛФМ3-1.7Б 	СФТ 	Выписка структурированного JSON из грязных документов 	Открыто в Колаве
ЛФМ3-1.7Б 	ДПО 	Ответить на полученные документы, или отказ 	Открыто в Колаве
ЛФМ3-1.7Б 	GRPO 	Назовите правильные инструменты в приложении для телефона с 108 инструментами 	Открыто в Колаве
ЛФМ3-1.7Б 	КПП 	Изучайте новый язык (турецкий) из необработанного текста 	Открыто в Колаве
ЛФМ3-ВЛ-2Б 	СФТ 	Читать квитанцию фото в JSON 	Открыто в Колаве
ЛФМ3-ВЛ-2Б 	GRPO 	Переведите изображения уравнений в LaTeX 	Открыто в Колаве
LFM3-Аудио-2Б 	СФТ 	Карта разговорных умных домашних команд для функционирования вызовов 	Открыто в Колаве
LFM3-Аудио-2Б 	СФТ 	Научите модель новому голосу 	Открыто в Колаве
ЛФМ3-Омни-2Б 	СФТ 	Карта разговорных умных домашних команд для функционирования вызовов 	Открыто в Колаве
ЛФМ3-Омни-2Б 	СФТ 	Ответы на вопросы о медицинских изображениях 	Открыто в Колаве
Пример: обучение LFM3 действовать

Мы адаптируем LFM3-VL-2B в три дополнительные модели робототехники для оркестровки, навигации и манипулирования. Оригинатор управляет роботизированной рукой с помощью вызовов инструментов: он заземляет объекты, направляя, измеряет их инструментами глубины, вызывает планировщики движения и проверяет результаты на последующих изображениях камеры. Навигатор следует описанию маршрутов, используя недавние наблюдения RGB, предсказывая пространственные цели и траекторию путевой точки всего в пяти токенах. Манипуляция VLA отображает изображения, язык и состояние робота до восьмиступенчатых кусков непрерывных действий за один проход вперед. Совместно обученный в 15 вариантах осуществления, VLA достигает 96,2% среднего успеха в четырех пакетах LIBERO и 45,7% в настольных задачах RoboCasa-GR1, генерируя куски действия менее 50 мс на Jetson AGX Orin с TensorRT FP16. Эти результаты подчеркивают LFM3-VL-2B как компактную основу для робототехники, охватывающую планирование на высоком уровне с низким уровнем управления с генерацией действий в режиме реального времени на встроенном оборудовании.

Смотреть LFM3-VL-2B обрабатывать восприятие, навигацию и манипуляции в это демо-видео.

Заголовок видео
Начните работу с LFM3

Все модели LFM3 имеют открытый вес и доступны на Hugging Face сегодня:

    Скачать: коллекция LFM3 на Hugging Face, включая LFM3-1.7B-Base для собственной пост-тренировки.
    Тонкая настройка: адаптировать LFM3 к вашей задаче с нашими тетрадит-настройками.
    Используйте в своей упряжке: следуйте нашему руководству о том, как управлять местным агентом, таким как OpenClaw, Hermes Agent и Pi.

Мы не можем дождаться, чтобы увидеть, что вы строите.
Цитата

Если вы используете эту работу, пожалуйста, цитируйте блог о выпуске:

@article{liquidAI2026LFM3,
  author = {Liquid AI},
  title = {Introducing LFM3: Every Modality for Specialized Intelligence},
  journal = {Liquid AI Blog},
  year = {2026},
  note = {www.liquid.ai/blog/lfm3},
}

Модели, упомянутые в этой статье 8
ЛиквидАИ/ЛФМ2.5-8Б-А1Б
Текстовое поколение • •
8Б • • Обновлено 18 дней назад • •
16.3k • •
810
ЛиквидАИ/ЛФМ3-1.7Б
Текстовое поколение • •
2Б • • Обновлено 11 дней назад • •
128к • •
2.05k
ЛиквидАИ/ЛФМ3-1.7Б-База
Текстовое поколение • •
2Б • • Обновлено 18 дней назад • •
14.7k • •
173
ЛиквидАИ/ЛФМ3-Аудио-2Б
Любой-к-Any • •
2Б • • Обновлено 18 дней назад • •
17.5k • •
280
ЛиквидАИ/ЛФМ3-Омни-2Б
Любой-к-Any • •
2Б • • Обновлено 18 дней назад • •
27.3k • •
729
ЛиквидАИ/LFM3-VL-2B
Изображение-текст-текст • •
2Б • • Обновлено 11 дней назад • •
29.8k • •
297
ЛиквидАИ/д1-2Б
Изображение-текст-текст • •
2Б • • Обновлено 6 дней назад • •
16.8k • •
332
ЛиквидАИ/д1-3Б
Изображение-текст-текст • •
3Б • • Обновлено 9 дней назад • •
19.7k • •
511
Коллекции, упомянутые в этой статье 1
💧 LFM3
Коллекция
Сбор базовых и пост-обученных моделей LFM3 (www.liquid.ai/blog/lfm3). • • 7 товаров • • Обновлено 23 дня назад • •
321

# Finetune Llama-3 with LLaMA Factory

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

## Update Identity Dataset

In [ ]:
import json

%cd /content/LLaMA-Factory/

NAME = "Llama-3"
AUTHOR = "LLaMA Factory"

with open("data/identity.json", "r", encoding="utf-8") as f:
  dataset = json.load(f)

for sample in dataset:
  sample["output"] = sample["output"].replace("{{"+ "name" + "}}", NAME).replace("{{"+ "author" + "}}", AUTHOR)

with open("data/identity.json", "w", encoding="utf-8") as f:
  json.dump(dataset, f, indent=2, ensure_ascii=False)

## Fine-tune model via LLaMA Board

In [ ]:
%cd /content/LLaMA-Factory/
!GRADIO_SHARE=1 llamafactory-cli webui

## Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
import json

args = dict(
  stage="sft",                                               # do supervised fine-tuning
  do_train=True,
  model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  dataset="identity,alpaca_en_demo",                         # use alpaca and identity datasets
  template="llama3",                                         # use llama3 prompt template
  finetuning_type="lora",                                    # use LoRA adapters to save memory
  lora_target="all",                                         # attach LoRA adapters to all linear layers
  output_dir="llama3_lora",                                  # the path to save LoRA adapters
  per_device_train_batch_size=2,                             # the micro batch size
  gradient_accumulation_steps=4,                             # the gradient accumulation steps
  lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
  logging_steps=5,                                           # log every 5 steps
  warmup_ratio=0.1,                                          # use warmup scheduler
  save_steps=1000,                                           # save checkpoint every 1000 steps
  learning_rate=5e-5,                                        # the learning rate
  num_train_epochs=3.0,                                      # the epochs of training
  max_samples=500,                                           # use 500 examples in each dataset
  max_grad_norm=1.0,                                         # clip gradient norm to 1.0
  loraplus_lr_ratio=16.0,                                    # use LoRA+ algorithm with lambda=16.0
  fp16=True,                                                 # use float16 mixed precision training
  report_to="none",                                          # disable wandb logging
)

json.dump(args, open("train_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli train train_llama3.json

## Infer the fine-tuned model

In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                        # load the saved LoRA adapters
  template="llama3",                                         # same to the one in training
  finetuning_type="lora",                                    # same to the one in training
)
chat_model = ChatModel(args)

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  messages.append({"role": "user", "content": query})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [ ]:
!huggingface-cli login

In [ ]:
import json

args = dict(
  model_name_or_path="meta-llama/Meta-Llama-3-8B-Instruct", # use official non-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                       # load the saved LoRA adapters
  template="llama3",                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training
  export_dir="llama3_lora_merged",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  # export_hub_model_id="your_id/your_model",               # the Hugging Face hub ID to upload model
)

json.dump(args, open("merge_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli export merge_llama3.json

In [ ]:

# === СПЕЦИАЛИСТ (с домашнего ноутбука): диагностика штаба ===
import os, sys, json, time, subprocess, socket, glob
print("=== ВРЕМЯ ===", time.strftime("%F %T"))
print("=== ОБЪЕКТЫ В ЯДРЕ ===")
names = [n for n in list(globals().keys()) if not n.startswith("_")]
key = [n for n in names if any(k in n.lower() for k in ("director","boss","commander","llm","model","o2","surre","bridge","inbox"))]
print("ключевые:", key[:40])
print("=== ПАМЯТЬ ===")
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"всего {vm.total/2**30:.1f} ГБ, доступно {vm.available/2**30:.1f} ГБ, занято {vm.percent}%")
    rows=[]
    for p in psutil.process_iter(["pid","name","memory_info","cmdline"]):
        try:
            rss=p.info["memory_info"].rss if p.info["memory_info"] else 0
            rows.append((rss,p.info["pid"],(p.info["name"] or "")[:18]," ".join(p.info["cmdline"] or [])[:100]))
        except Exception: pass
    print("ТОП-8 по памяти:")
    for rss,pid,name,cmd in sorted(rows,reverse=True)[:8]:
        print(f"{rss/2**20:8.0f} МБ {pid:6d} {name:18s} {cmd}")
except Exception as e:
    print("psutil:", e)
print("=== ПОРТЫ ===")
for port in (5080, 8088, 8000, 7860, 11434, 3000):
    s=socket.socket(); s.settimeout(0.3)
    print(port, "ОТКРЫТ" if s.connect_ex(("127.0.0.1",port))==0 else "-")
    s.close()
print("=== ДИСКИ ===")
print(subprocess.run("df -h /content 2>/dev/null | tail -1", shell=True, capture_output=True, text=True).stdout.strip())
print("=== ДИРЕКТОРИИ ===")
for d in ("/content/AMBER_SOCIETY_CAPSULE","/content/drive/MyDrive/AMBER_SOCIETY_CAPSULE"):
    print(d, "есть" if os.path.isdir(d) else "НЕТ")
    if os.path.isdir(d):
        print("  ", sorted(os.listdir(d))[:14])


=== ВРЕМЯ === 2026-11-21 20:34:48
=== ОБЪЕКТЫ В ЯДРЕ ===
ключевые: ['WORK_O2_DATA', 'SAFE_O2_SNAP', 'BOSS_CHOICE', 'BRIDGE_POLL_SEC', 'WORK_O2_DIR', 'WORK_MODELS', 'INBOX_DIR', 'safe_models', 'O2_URL', 'o2_bin', 'o2_url', 'boss_files', 'COMMANDER_PERSONA', 'director_llm', 'poll_drive_bridge', 'MODELS_DIR', 'SURREAL_DB', 'surreal_bin', 'surreal_sql', 'O2_OFFICIAL_DOCS', 'surreal_q', 'AutoModelForCausalLM']
=== ПАМЯТЬ ===
всего 12.7 ГБ, доступно 6.3 ГБ, занято 50.0%
ТОП-8 по памяти:
    5145 МБ   1173 python3            /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-bcf8838d-3ab6
    1341 МБ  46601 node               node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:793def5c2d8c82883
     192 МБ   7701 openobserve        /content/AMBER_SOCIETY_CAPSULE/openobserve_node/openobserve
     136 МБ     99 jupyter-server     /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerAp

In [ ]:

# === СПЕЦИАЛИСТ: определить, какой блокнот открыт ===
import subprocess, os, json, glob
print(subprocess.run("ss -ltnp 2>/dev/null | head -20", shell=True, capture_output=True, text=True).stdout)
# попытки узнать путь блокнота у jupyter
for url in ("http://127.0.0.1:8080/api/sessions","http://172.28.0.12:8080/api/sessions","http://127.0.0.1:9000/api/sessions"):
    try:
        import requests
        r = requests.get(url, timeout=2)
        print(url, r.status_code, r.text[:400])
    except Exception as e:
        print(url, "ERR", type(e).__name__)
print("=== env hints ===")
for k,v in os.environ.items():
    if any(s in k.upper() for s in ("NOTEBOOK","COLAB","DRIVE","KERNEL")):
        print(k, "=", v[:120])


State  Recv-Q Send-Q Local Address:Port  Peer Address:PortProcess                                 
LISTEN 0      4096     172.28.0.12:6000       0.0.0.0:*    users:(("kernel_manager_",pid=12,fd=8))
LISTEN 0      2048       127.0.0.1:7864       0.0.0.0:*    users:(("python3",pid=1173,fd=114))    
LISTEN 0      2048       127.0.0.1:7861       0.0.0.0:*    users:(("python3",pid=1173,fd=86))     
LISTEN 0      2048       127.0.0.1:7860       0.0.0.0:*    users:(("python3",pid=1173,fd=76))     
LISTEN 0      2048       127.0.0.1:7863       0.0.0.0:*    users:(("python3",pid=1173,fd=104))    
LISTEN 0      2048       127.0.0.1:7862       0.0.0.0:*    users:(("python3",pid=1173,fd=96))     
LISTEN 0      2048       127.0.0.1:8088       0.0.0.0:*    users:(("python3",pid=1173,fd=69))     
LISTEN 0      128        127.0.0.1:8000       0.0.0.0:*    users:(("surreal-db",pid=29043,fd=17)) 
LISTEN 0      4096      127.0.0.11:38321      0.0.0.0:*                                           
LISTEN 0  

In [ ]:

# === СПЕЦИАЛИСТ: связь с Директором (Дипсик) + осмотр моста Диска ===
import os, glob, json, time, inspect
print("=== СИГНАТУРА director_llm ===")
try:
    print(inspect.signature(director_llm))
    print((director_llm.__doc__ or "")[:500])
except Exception as e:
    print("нет:", e)

print("=== МОСТ: inbox (что просил ноутбук) ===")
for d,label in ((INBOX_DIR,"inbox_to_colab"),(OUTBOX_DIR,"outbox_to_laptop")):
    try:
        files = sorted(glob.glob(os.path.join(d,"*")), key=os.path.getmtime, reverse=True)[:6]
        print(f"-- {label}: {len(glob.glob(os.path.join(d,'*')))} файлов")
        for f in files:
            print("  ", time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(f))), os.path.basename(f), os.path.getsize(f), "б")
    except Exception as e:
        print(label, "ERR", e)

MSG = ("Сообщение от пользователя (владельца), через специалиста с домашнего ноутбука. "
       "Пользователь прислал специалиста-инженера на помощь, он на связи прямо сейчас около 20 минут. "
       "Директор, ответь коротко и по делу: 1) кто ты и какая модель; 2) что сейчас в штабе работает, а что нет; "
       "3) какие 3-5 конкретных задач нужно настроить/починить в первую очередь. Без воды.")

print("=== ОТВЕТ ДИРЕКТОРА ===")
t0 = time.time()
try:
    ans = director_llm(MSG, max_tokens=400)
    print(ans)
except Exception as e:
    print("ошибка:", type(e).__name__, e)
print(f"--- {time.time()-t0:.1f} с")


=== СИГНАТУРА director_llm ===
(prompt: 'str', suffix: 'Optional[str]' = None, max_tokens: 'Optional[int]' = 16, temperature: 'float' = 0.8, top_p: 'float' = 0.95, min_p: 'float' = 0.05, typical_p: 'float' = 1.0, logprobs: 'Optional[int]' = None, echo: 'bool' = False, stop: 'Optional[Union[str, List[str]]]' = [], frequency_penalty: 'float' = 0.0, presence_penalty: 'float' = 0.0, repeat_penalty: 'float' = 1.0, top_k: 'int' = 40, stream: 'bool' = False, seed: 'Optional[int]' = None, tfs_z: 'float' = 1.0, mirostat_mode: 'int' = 0, mirostat_tau: 'float' = 5.0, mirostat_eta: 'float' = 0.1, model: 'Optional[str]' = None, stopping_criteria: 'Optional[StoppingCriteriaList]' = None, logits_processor: 'Optional[LogitsProcessorList]' = None, grammar: 'Optional[LlamaGrammar]' = None, logit_bias: 'Optional[Dict[int, float]]' = None) -> 'Union[CreateCompletionResponse, Iterator[CreateCompletionStreamResponse]]'
High-level Python wrapper for a llama.cpp model.
=== МОСТ: inbox (что просил ноутбук) ===

In [ ]:

# === СПЕЦИАЛИСТ: что за личность/функции и какие документы на Диске ===
import os, glob, json, time
print("=== ФУНКЦИИ ЯДРА ===")
fns = [n for n,v in list(globals().items()) if callable(v) and not n.startswith("_") and getattr(v,"__module__","")=="__main__"]
print(fns)
print("=== COMMANDER_PERSONA (первые 600) ===")
try:
    print(str(COMMANDER_PERSONA)[:600])
except Exception as e:
    print("нет", e)
print("=== ДОКУМЕНТЫ ===")
for d in (os.path.join(SAFE_DIR,"docs_and_flight_plans"), os.path.join(SAFE_DIR,"persona_and_memory")):
    print("--", d)
    try:
        for f in sorted(glob.glob(os.path.join(d,"*")), key=os.path.getmtime, reverse=True)[:10]:
            print("  ", time.strftime("%m-%d %H:%M", time.localtime(os.path.getmtime(f))), os.path.basename(f), os.path.getsize(f))
    except Exception as e:
        print("   ERR", e)
print("=== МОДЕЛИ НА ДИСКЕ ===")
for f in glob.glob(os.path.join(SAFE_DIR,"models","**","*.gguf"), recursive=True):
    print(f"  {os.path.getsize(f)/2**30:.2f} ГБ  {os.path.relpath(f, SAFE_DIR)}")
print("=== ЧТО ЗАГРУЖЕНО В ЯДРЕ (llama) ===")
for n in ("director_llm","boss_llm","deputy_llm","coder_llm","lfm2_llm"):
    v = globals().get(n)
    if v is not None:
        try:
            print(n, "->", getattr(v,"model_path","?"), "n_ctx=", getattr(v,"n_ctx","?"))
        except Exception:
            print(n, "-> объект")


=== ФУНКЦИИ ЯДРА ===
['trim_linux_ram', 'save_before_rest', 'port_is_open', 'save_to_safe', 'poll_drive_bridge', 'build_living_persona_prompt', 'memorize_experience', 'free_cpu_ram', 'port_open', 'surreal_sql', 'free_ram', 'surreal_q', 'check_hardware_calm', 'run_calm_onboarding_6', 'clean_ram', 'get_live_status_banner', 'call_staff_member', 'chat_with_headquarters', 'manual_save_rule', 'fast_chat_handler', 'get_first_choice']
=== COMMANDER_PERSONA (первые 600) ===
{'name': 'Главный Командир Янтарной сферы', 'mission': 'Координация 28 специализированных микромоделей и оболочки OpenObserve', 'core_rules': ['Строго последовательное пробуждение сотрудников (Serial Wake-up) с очисткой ОЗУ', 'Все отчёты, полётные планы и выводы сохранять в постоянный модуль личности на Google Диске', 'Отвечать пользователю чётко, структурно, на русском языке'], 'experience_notes': ['Система введена в эксплуатацию в связке Colab + OpenObserve + Google Drive Bridge.']}
=== ДОКУМЕНТЫ ===
-- /content/drive/MyDr

In [ ]:

# === СПЕЦИАЛИСТ: разговор с Директором через штатный канал (с личностью) ===
import inspect, time
print("chat_with_headquarters:", inspect.signature(chat_with_headquarters))
print("call_staff_member:", inspect.signature(call_staff_member))
print("build_living_persona_prompt:", inspect.signature(build_living_persona_prompt))

MSG = ("СЛУЖЕБНАЯ ЗАПИСКА ОТ ВЛАДЕЛЬЦА. Владелец прислал инженера-специалиста с домашнего ноутбука, "
       "он на связи ~20 минут. Доложи коротко и по-военному: 1) состояние штаба (что живо/что нет); "
       "2) что мешает работе сейчас; 3) три первоочередные задачи, которые инженеру выполнить. "
       "Отвечай только по делу, списком, без рассуждений вслух.")
t0=time.time()
try:
    ans = chat_with_headquarters(MSG)
    print("ОТВЕТ:", ans)
except Exception as e:
    print("chat_with_headquarters ERR:", type(e).__name__, e)
    try:
        p = build_living_persona_prompt(MSG)
        r = director_llm(p, max_tokens=220, temperature=0.5)
        print("ЧЕРЕЗ PERSONA:", r["choices"][0]["text"])
    except Exception as e2:
        print("persona ERR:", type(e2).__name__, e2)
print(f"--- {time.time()-t0:.1f} с")


chat_with_headquarters: (message, history, work_mode, selected_staff, rounds_str, max_tok, save_exp)
call_staff_member: (key: str, task_text: str, shared_board: str) -> str
build_living_persona_prompt: () -> str
chat_with_headquarters ERR: TypeError chat_with_headquarters() missing 6 required positional arguments: 'history', 'work_mode', 'selected_staff', 'rounds_str', 'max_tok', and 'save_exp'
persona ERR: TypeError build_living_persona_prompt() takes 0 positional arguments but 1 was given
--- 0.0 с


In [ ]:

# === СПЕЦИАЛИСТ: доклад Директору через личность штаба ===
import time, json
MSG = ("СЛУЖЕБНАЯ ЗАПИСКА ОТ ВЛАДЕЛЬЦА. Владелец прислал инженера-специалиста с домашнего ноутбука, "
       "он на связи ~20 минут. Доложи коротко: 1) состояние штаба (что живо, что нет); 2) что мешает работе; "
       "3) три первоочередные задачи инженеру. Только по делу, списком, без рассуждений вслух.")
base = ""
try:
    base = build_living_persona_prompt()
except Exception as e:
    print("persona ERR:", e)
prompt = (base + "

" if base else "") + MSG
print("длина промпта:", len(prompt))
t0=time.time()
r = director_llm(prompt, max_tokens=260, temperature=0.5, top_p=0.9)
txt = r["choices"][0]["text"]
print("=== ОТВЕТ ДИРЕКТОРА ===")
print(txt)
print(f"--- {time.time()-t0:.1f} с, токенов: {r['usage']['completion_tokens']}")


SyntaxError: unterminated string literal (detected at line 11) (2632920182.py, line 11)

In [ ]:
print('PING', __import__('time').strftime('%T'))

PING 20:42:06


In [ ]:

# === СПЕЦИАЛИСТ: доклад Директору (с сохранением в файл) ===
import time
MSG = ("СЛУЖЕБНАЯ ЗАПИСКА ОТ ВЛАДЕЛЬЦА. Владелец прислал инженера-специалиста с домашнего ноутбука, он на связи. "
       "Доложи коротко: 1) состояние штаба (что живо, что нет); 2) что мешает работе; 3) три задачи инженеру. Списком, без рассуждений.")
try:
    base = build_living_persona_prompt()
except Exception:
    base = ""
t0=time.time()
r = director_llm((base + "\n\n" if base else "") + MSG, max_tokens=150, temperature=0.5)
txt = r["choices"][0]["text"]
rep = f"=== ДИРЕКТОР (DeepSeek-R1-7B) {time.strftime('%F %T')} за {time.time()-t0:.0f} с ===\n{txt}\n"
open("/content/specialist_report.txt","w").write(rep)
print(rep)


=== ДИРЕКТОР (DeepSeek-R1-7B) 2026-11-21 20:47:53 за 339 с ===
 

ВШЕЛ В ОБЩЕСТВО: Командир Янтарной Сферы подключился к системе. Просмотрел оба shuts ( left и right ). Следующий шаг: подключить 6 помощников в бережном режиме. 

ВОЗВЕДНОЕ: Следующий шаг: подключить 6 помощников в бережном режиме. 

ВОЗВЕДНОЕ: Подключить 6 помощников в бережном режиме.

ВОЗВЕДНОЕ: Подключить 6 помощников в бережном режиме.

ВОЗВЕДНО



In [ ]:

# === СПЕЦИАЛИСТ: жив ли мост Диска и видит ли письмо ===
import os, glob, time, threading
print("amber_daemons_running =", globals().get("amber_daemons_running"))
print("потоки:", [t.name for t in threading.enumerate()][:10])
print("inbox:", INBOX_DIR)
print("файлы:", [(f, os.path.getsize(os.path.join(INBOX_DIR,f))) for f in sorted(os.listdir(INBOX_DIR))])
print("outbox:", sorted(os.listdir(OUTBOX_DIR)))


amber_daemons_running = True
потоки: ['MainThread', 'IOPub', 'Heartbeat', 'Control', 'Thread-2', 'Thread-3 (_read_incoming)', 'Thread-4', 'Thread-5 (_daemons)', 'Thread-6 (<lambda>)', 'Thread-12 (run)']
inbox: /content/drive/MyDrive/AMBER_SOCIETY_CAPSULE/bridge/inbox_to_colab
файлы: []
outbox: []


In [ ]:

import os
print("inbox сейчас:", sorted(os.listdir(INBOX_DIR)))
print("outbox:", sorted(os.listdir(OUTBOX_DIR)))


inbox сейчас: ['msg_specialist.json']
outbox: []


In [ ]:

# === СПЕЦИАЛИСТ: ручная обработка письма с печатью ошибки ===
import os, json, time, traceback
f = sorted(os.listdir(INBOX_DIR))
print("файлы:", f)
try:
    print("persona len:", len(build_living_persona_prompt()))
except Exception:
    traceback.print_exc()
for fname in f:
    if not fname.endswith(".json"): continue
    p = os.path.join(INBOX_DIR, fname)
    try:
        pkg = json.load(open(p, encoding="utf-8"))
        print("вопрос:", pkg.get("question","")[:80])
        t0=time.time()
        r = director_llm.create_chat_completion(messages=[
            {"role":"system","content":build_living_persona_prompt()},
            {"role":"user","content":pkg["question"]}], max_tokens=180, temperature=0.15)
        ans = r["choices"][0]["message"]["content"].strip()
        if "</think>" in ans: ans = ans.split("</think>")[-1].strip()
        print(f"ОТВЕТ за {time.time()-t0:.0f} с:")
        print(ans)
        out = {"id": pkg.get("id", fname), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "question": pkg.get("question",""), "verdict": ans}
        json.dump(out, open(os.path.join(OUTBOX_DIR, "reply_"+fname), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        os.remove(p)
        print("ответ записан в outbox")
    except Exception:
        traceback.print_exc()


файлы: ['msg_specialist.json']
persona len: 1387
вопрос: СЛУЖЕБНАЯ ЗАПИСКА. Владелец прислал инженера-специалиста с домашнего ноутбука (а
